# Recolección y Preparación de Datos
**Proyecto:** Segmentación automática de columna vertebral en radiografías  
**Dataset activo:** `Scoliosis_Dataset` — versión limpia T1–T12 y L1–L5  
**Maestría en Inteligencia Artificial (MaIA) — Universidad de los Andes**

---

## ¿Qué hace este notebook?

Este notebook prepara y evalúa el dataset limpio `Scoliosis_Dataset` para trabajar con **MedSAM**. Esta versión ya trae las máscaras multiclase reorganizadas con IDs consecutivos para T1–T12 y L1–L5, por lo que no se debe usar el diccionario antiguo de 22/35 entidades.

El flujo principal es:

1. Leer `indice_dataset.csv` y validar que existan imágenes/máscaras.
2. Dividir pacientes válidos en `train`, `val` y `test`.
3. Exportar imágenes, máscaras y `prompts.json` en formato MedSAM.
4. Entrenar/evaluar variantes de MedSAM usando `val`.
5. Reservar `test` exclusivamente para la evaluación final.

> Nota metodológica: los prompts actuales (`bbox_gt`) se derivan de las máscaras anotadas. Por tanto, las métricas evalúan segmentación condicionada por cajas ideales, no un pipeline automático end-to-end.

## Estructura real del dataset

```
Scoliosis_Dataset/
├── Normal/                               # radiografías sanas        → N_1.jpg, N_2.jpg ...
├── Scoliosis/                            # radiografías escoliosis   → S_100.jpg, S_101.jpg ...
├── LabelBinaryJPG/                       # máscaras binarias
├── LabelMultiClass_ID_PNG/               # máscaras multiclase con IDs 0..17
├── LabelMultiClass_Gray_JPG/             # visualización en grises
├── LabelMultiClass_Color_JPG/            # visualización en color
├── RadiographMetrics/                    # métricas Cobb recalculadas
├── diccionario_etiquetas_T1_T12_L1_L5.json
└── indice_dataset.csv
```

**Codificación de las máscaras multiclase limpias:**  
`0`=fondo · `1`=T1 · `2`=T2 · ... · `12`=T12 · `13`=L1 · ... · `17`=L5

No se encontró un archivo COCO/instances en esta versión. La fuente oficial de clases para segmentación es `diccionario_etiquetas_T1_T12_L1_L5.json`.

## Secciones del notebook

1. Configuración del entorno y rutas  
2. Inventario del dataset  
3. Análisis de distribución y ángulo de Cobb  
4. Inspección visual de imágenes y máscaras  
5. Control de calidad  
6. Normalización y preprocesamiento para MedSAM  
7. División train / val / test  
8. Exportación MedSAM  
9. Entrenamiento, evaluación en validación y test final bloqueado

<!-- codex-explicacion -->
Este notebook marca el salto de baseline a problema real: segmentar vertebras individuales y evaluar por clase. Aqui se define la logica de particion estratificada, se prueban escenarios MedSAM y se establece la diferencia entre validacion y test final.

Es importante leerlo como laboratorio metodologico: varias rutas se prueban y se descartan, pero de aqui sale la necesidad de prompts automaticos mejores.


---
## 1. Configuración del entorno y rutas

### 1.1 Instalación de dependencias

El primer paso es asegurarse de que todas las librerías necesarias estén disponibles. Se instalan con `pip` en modo silencioso (`-q`):

- **numpy / pandas** — manipulación numérica y tabular
- **matplotlib / seaborn** — visualización estadística
- **Pillow (PIL)** — lectura de imágenes PNG de 16 bits. Las máscaras multiclase son `uint16` y OpenCV las truncaría a `uint8` silenciosamente, corrompiendo los IDs de vértebras
- **scikit-learn** — `train_test_split` con estratificación
- **opencv-python-headless** — CLAHE, redimensionado, conversión de color y manejo de imágenes
- **tqdm** — barras de progreso en los bucles de exportación


### 1.2 Importación de librerías y semilla aleatoria

Se importan todos los módulos necesarios. El `SEED = 42` garantiza que cualquier operación aleatoria —muestreo para visualizaciones, división del dataset— produzca siempre el mismo resultado, haciendo el experimento reproducible.


In [ ]:
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print('✅ Entorno listo')


### 1.3 Definición de rutas

Aquí se define DATASET_ROOT apuntando a la carpeta limpia Scoliosis_Dataset.

A partir de esa ruta se derivan las subcarpetas de imágenes, máscaras y métricas. Esta versión trae además indice_dataset.csv, que se usa como inventario oficial y permite validar qué archivos existen realmente.

También se carga diccionario_etiquetas_T1_T12_L1_L5.json, el mapa oficial de esta versión. Las máscaras multiclase ya usan IDs consecutivos: 0=fondo, 1..12=T1..T12 y 13..17=L1..L5. Por eso no se aplica el remapeo antiguo desde IDs 6..22.


In [ ]:
# ─────────────────────────────────────────────────────────────
# DATASET ACTIVO
# ─────────────────────────────────────────────────────────────
DATASET_ROOT = Path('C:/Users/luisf/Downloads/ProyectoFinal/Scoliosis_Dataset')

# Subcarpetas fijas del dataset limpio
DIR_NORMAL      = DATASET_ROOT / 'Normal'
DIR_SCOLIOSIS   = DATASET_ROOT / 'Scoliosis'
DIR_MASK_ID     = DATASET_ROOT / 'LabelMultiClass_ID_PNG'
DIR_MASK_BIN    = DATASET_ROOT / 'LabelBinaryJPG'
DIR_METRICS     = DATASET_ROOT / 'RadiographMetrics'
COBB_SUMMARY_PATH = DIR_METRICS / 'metricas_cobb_resumen_recalculado.csv'
DATASET_INDEX_PATH = DATASET_ROOT / 'indice_dataset.csv'
LABELS_DICT_PATH = DATASET_ROOT / 'diccionario_etiquetas_T1_T12_L1_L5.json'

# Salida separada para no mezclarla con exportaciones del dataset anterior
OUTPUT_ROOT = DATASET_ROOT.parent / 'dataset_procesado_scoliosis_medsam'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Cargar diccionario oficial de la versión limpia T1–T12/L1–L5
with open(LABELS_DICT_PATH, encoding='utf-8') as f:
    LABELS_DICT = json.load(f)

MAPEO_ID = {int(k): v for k, v in LABELS_DICT['mascara_multiclase_id_png'].items()}
CLASES = {k: v for k, v in MAPEO_ID.items() if 1 <= k <= 17}
N_CLASES = len(CLASES)
NOMBRES_CLASES = [CLASES[i] for i in range(1, N_CLASES + 1)]

print(f'Dataset:  {DATASET_ROOT}')
print(f'Existe:   {DATASET_ROOT.exists()}')
print(f'Indice:   {DATASET_INDEX_PATH.exists()}')
print(f'COCO:     no encontrado en esta version')
print(f'Clases:   {N_CLASES} → {NOMBRES_CLASES}')


## 2. Inventario del dataset

Antes de entrenar cualquier modelo se construye un inventario unico de casos utilizables. Esta parte existe para no depender de supuestos implicitos sobre los nombres de archivos ni sobre que todos los registros del indice tengan imagen y mascara disponibles.

Se usa `indice_dataset.csv` como fuente principal de metadatos clinicos y rutas relativas. A partir de ese indice se validan tres cosas: que la imagen exista, que la mascara exista y que ambas puedan asociarse al mismo `image_id`. Los casos incompletos quedan marcados desde el inicio, de modo que no entren accidentalmente en `train`, `val` o `test`.

En este dataset limpio no se encontro un archivo COCO de anotaciones; por eso el flujo trabaja directamente con las mascaras multiclase ya incluidas en el dataset.

<!-- codex-explicacion -->
El inventario asegura que cada paciente tenga imagen y mascara asociada. Sin este conteo, una mejora posterior podria deberse solo a que se entreno con un subconjunto distinto.


In [ ]:
def absolutizar_ruta(rel_path):
    if pd.isna(rel_path) or str(rel_path).strip() == '':
        return None
    return DATASET_ROOT / str(rel_path)


df_index = pd.read_csv(str(DATASET_INDEX_PATH), dtype=str, keep_default_na=False, encoding='utf-8', engine='python')

registros = []
for _, fila in df_index.iterrows():
    img_path = absolutizar_ruta(fila['ruta_radiografia'])
    mask_id_path = absolutizar_ruta(fila['ruta_mascara_multiclase_id_png'])
    mask_bin_path = absolutizar_ruta(fila['ruta_mascara_binaria'])

    imagen = str(fila['imagen'])
    patient_id = Path(imagen).stem
    grupo = str(fila['grupo'])
    tipo = 'sano' if grupo.lower() == 'normal' else 'escoliosis'

    registros.append({
        'patient_id'    : patient_id,
        'tipo'          : tipo,
        'grupo_original': grupo,
        'ruta_img'      : str(img_path) if img_path is not None else None,
        'ruta_mask_id'  : str(mask_id_path) if mask_id_path is not None else None,
        'ruta_mask_bin' : str(mask_bin_path) if mask_bin_path is not None else None,
        'ruta_metrics'  : None,
        'tiene_img'     : bool(img_path is not None and img_path.exists()),
        'tiene_mask_id' : bool(mask_id_path is not None and mask_id_path.exists()),
        'tiene_mask_bin': bool(mask_bin_path is not None and mask_bin_path.exists()),
        'tiene_metrics' : False,
    })

df = pd.DataFrame(registros)

print(f'Total filas en indice: {len(df)}')
print(f"Casos con imagen y mascara multiclase: {int((df['tiene_img'] & df['tiene_mask_id']).sum())}")
print()
print(df.groupby('tipo')[['tiene_img', 'tiene_mask_id', 'tiene_mask_bin']].sum().astype(int))

faltantes = df[~(df['tiene_img'] & df['tiene_mask_id'])]
if len(faltantes) > 0:
    print('\nFilas excluibles por archivos faltantes:')
    display(faltantes[['patient_id', 'tipo', 'ruta_img', 'ruta_mask_id', 'tiene_img', 'tiene_mask_id']])


## 3. Analisis de distribucion y angulo de Cobb

Esta seccion resume como esta compuesto el dataset antes de dividirlo. No es solo una descripcion: sirve para detectar desbalance entre clases, revisar tamanos de imagen y entender si la severidad de la escoliosis esta distribuida de forma razonable.

El conjunto utilizable queda en 249 casos validos: 71 normales y 178 con escoliosis. El indice original contiene un caso adicional con escoliosis, pero se excluye porque no tiene todos los archivos necesarios. Separar esta informacion evita inflar artificialmente el tamano real del dataset.

El angulo de Cobb se usa aqui como variable clinica de apoyo. Mas adelante permite crear estratos de particion, de forma que `train`, `val` y `test` no queden sesgados hacia casos muy faciles o muy severos por azar.

<!-- codex-explicacion -->
Esta seccion justifica la particion estratificada: no basta separar al azar si se quiere conservar normales, escoliosis y severidad en train/val/test.


In [ ]:
def get_size(ruta):
    try:
        img = Image.open(ruta)
        return img.width, img.height
    except:
        return None, None

df[['ancho', 'alto']] = df['ruta_img'].apply(lambda r: pd.Series(get_size(r)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

conteo = df['tipo'].value_counts()
axes[0].bar(conteo.index, conteo.values, color=['#4CAF50', '#F44336'])
axes[0].set_title('Distribución: Sanos vs Escoliosis')
axes[0].set_ylabel('Número de imágenes')
for i, v in enumerate(conteo.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

col = df['tipo'].map({'sano': '#4CAF50', 'escoliosis': '#F44336'})
axes[1].scatter(df['ancho'], df['alto'], c=col, alpha=0.5, s=30)
axes[1].set_title('Variabilidad de dimensiones de radiografías')
axes[1].set_xlabel('Ancho (px)'); axes[1].set_ylabel('Alto (px)')

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'distribucion_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('Estadísticas de dimensiones:')
print(df[['ancho','alto']].describe().round(0).astype(int))


### 3.2 Análisis del ángulo de Cobb

El ángulo de Cobb es la medida clínica estándar para cuantificar la severidad de la escoliosis. Se mide como el ángulo entre las líneas trazadas a lo largo de las vértebras más inclinadas de la curva espinal. Los criterios clínicos son:

- < 10° → Normal (sin escoliosis significativa)
- 10–24° → Leve
- 25–39° → Moderada
- ≥ 40° → Severa

En esta versión limpia, las métricas Cobb están consolidadas en RadiographMetrics/metricas_cobb_resumen_recalculado.csv. La tabla contiene el ángulo, puntos de inflexión, ápice y rutas a curvas/overlays recalculados.


In [ ]:
if COBB_SUMMARY_PATH.exists():
    df_cobb = pd.read_csv(COBB_SUMMARY_PATH)
    df_cobb['patient_id'] = 'S_' + df_cobb['patient_id'].astype(int).astype(str)
    df_cobb = df_cobb.rename(columns={'angulo_cobb_deg': 'cobb_deg'})
else:
    df_cobb = pd.DataFrame(columns=['patient_id', 'cobb_deg'])

def clasificar(deg):
    if pd.isna(deg): return 'Sin dato'
    if deg < 10:   return 'Normal (<10°)'
    elif deg < 25: return 'Leve (10-24°)'
    elif deg < 40: return 'Moderada (25-39°)'
    else:          return 'Severa (>=40°)'

df_cobb['severidad'] = df_cobb['cobb_deg'].apply(clasificar)

if len(df_cobb) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    med = df_cobb['cobb_deg'].median()
    axes[0].hist(df_cobb['cobb_deg'].dropna(), bins=20, color='#F44336', edgecolor='white', alpha=0.8)
    axes[0].axvline(med, ls='--', color='k', label=f'Mediana: {med:.1f} grados')
    axes[0].set_title('Distribución del ángulo de Cobb')
    axes[0].set_xlabel('Ángulo (grados)'); axes[0].legend()

    sc = df_cobb['severidad'].value_counts()
    axes[1].bar(sc.index, sc.values, color=['#FF9800','#FF5722','#B71C1C','#777777'][:len(sc)])
    axes[1].set_title('Clasificación por severidad clínica')
    axes[1].set_ylabel('Cantidad de pacientes')
    axes[1].tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / 'cobb_distribucion.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(df_cobb['cobb_deg'].describe().round(1))
else:
    print('No se encontro resumen Cobb recalculado.')


### 3.3 Cobertura de vértebras en las máscaras

Esta versión del dataset está organizada solo para T1–T12 y L1–L5. Las máscaras multiclase usan IDs 1..17 para esas vértebras y 0 para fondo.

La celda siguiente verifica cuántas vértebras están presentes por imagen, si hay torácicas/lumbares anotadas, si la secuencia T1–L5 está completa y si aparece algún ID fuera del diccionario limpio. Esto ayuda a detectar imágenes incompletas o inconsistencias antes de entrenar/evaluar.


In [ ]:
def analizar_mask(ruta_mask_id: str) -> dict:
    try:
        mask = np.array(Image.open(ruta_mask_id))
        vals = set(mask.flatten().tolist()) - {0}
        vals_validos = vals & set(range(1, 18))
        vals_fuera_diccionario = vals - set(range(1, 18))
        return {
            'n_clases'              : len(vals_validos),
            'vertebras'             : [MAPEO_ID.get(v, f'cls_{v}') for v in sorted(vals_validos)],
            'tiene_toracicas'       : bool(vals_validos & set(range(1, 13))),   # T1-T12
            'tiene_lumbares'        : bool(vals_validos & set(range(13, 18))),  # L1-L5
            'T1_L5_completa'        : all(v in vals_validos for v in range(1, 18)),
            'ids_fuera_diccionario' : sorted(vals_fuera_diccionario),
        }
    except Exception as e:
        return {'n_clases': 0, 'error': str(e)}

df_m = df[df['tiene_mask_id']].copy()
analisis = pd.DataFrame(df_m['ruta_mask_id'].apply(analizar_mask).tolist())
df_m = pd.concat([df_m.reset_index(drop=True), analisis], axis=1)

print('Cobertura de regiones vertebrales en la version limpia:')
for col, label in [
    ('tiene_toracicas',  'Con toracicas (T1-T12)    '),
    ('tiene_lumbares',   'Con lumbares (L1-L5)      '),
    ('T1_L5_completa',   'T1-L5 completo            ')]:
    n = int(df_m[col].sum())
    pct = 100 * n / len(df_m)
    print(f'  {label}: {n}/{len(df_m)} ({pct:.0f}%)')

ids_fuera = df_m['ids_fuera_diccionario'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False).sum()
print(f'\nMascaras con IDs fuera de 1..17: {int(ids_fuera)}')
print('\nPromedio de vertebras anotadas por imagen:')
print(df_m.groupby('tipo')['n_clases'].describe().round(1))


---
## 4. Inspección visual de imágenes y máscaras

La estadística es útil, pero nada reemplaza ver los datos directamente. Esta sección muestra muestras aleatorias de radiografías con sus máscaras superpuestas. Esto permite detectar problemas que los números no revelan: anotaciones mal alineadas, radiografías con artefactos, vértebras cuyas máscaras no cubren bien la silueta real, etc.

### 4.1 Muestras por tipo de paciente

Se visualizan 4 muestras de cada subgrupo (sanos y escoliosis). Para cada imagen se muestra:

- **Fila superior:** la radiografía original en escala de grises
- **Fila inferior:** la misma radiografía con la máscara multiclase superpuesta en colores. Cada color diferente representa una vértebra distinta. La transparencia parcial (alpha=0.55 para la imagen base, 0.6 para la máscara) permite ver simultáneamente la anatomía real y las anotaciones.


In [ ]:
def visualizar(df_sub, n=4, titulo=''):
    sub = df_sub[df_sub['tiene_mask_id']].sample(min(n, len(df_sub)), random_state=SEED)
    if sub.empty:
        print('No hay imagenes con mascara en este subconjunto.')
        return
    fig, axes = plt.subplots(2, len(sub), figsize=(4.5 * len(sub), 9))
    if len(sub) == 1:
        axes = axes.reshape(2, 1)
    for col, (_, fila) in enumerate(sub.iterrows()):
        img  = np.array(Image.open(fila['ruta_img']).convert('L'))
        mask = np.array(Image.open(fila['ruta_mask_id']))  # uint16
        n_v  = len(set(mask.flatten()) - {0})
        axes[0, col].imshow(img, cmap='gray')
        axes[0, col].set_title(f"{fila['patient_id']} ({fila['tipo']})", fontsize=9)
        axes[0, col].axis('off')
        axes[1, col].imshow(img, cmap='gray', alpha=0.55)
        axes[1, col].imshow(mask, cmap='nipy_spectral', alpha=0.6, vmin=0, vmax=22)
        axes[1, col].set_title(f'Mascara ({n_v} vert. anotadas)', fontsize=9)
        axes[1, col].axis('off')
    plt.suptitle(titulo, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

visualizar(df[df['tipo'] == 'sano'],       n=4, titulo='Muestra — Pacientes Sanos')
visualizar(df[df['tipo'] == 'escoliosis'], n=4, titulo='Muestra — Escoliosis')


### 4.2 Comparacion entre mascara binaria y mascara multiclase

Esta revision visual comprueba que la mascara binaria y la mascara multiclase representan la misma region anatomica desde dos niveles de detalle distintos.

La mascara binaria responde a la pregunta global: "donde esta la columna?". La mascara multiclase responde a una pregunta mas fina: "a que vertebra pertenece cada pixel?". En este dataset limpio las clases utiles son `1..17`, correspondientes a `T1..T12` y `L1..L5`; el fondo es `0`.

Esta prueba no mide desempeno del modelo. Es un control de coherencia de las anotaciones antes de usarlas para entrenamiento o evaluacion.
        


In [ ]:
ej   = df[df['tiene_mask_id'] & df['tiene_mask_bin']].iloc[0]
img  = np.array(Image.open(ej['ruta_img']).convert('L'))
mbin = np.array(Image.open(ej['ruta_mask_bin']))
mid  = np.array(Image.open(ej['ruta_mask_id']))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title(f"Radiografia original\n{ej['patient_id']}")
axes[1].imshow(mbin, cmap='gray')
axes[1].set_title('Mascara binaria\n(spine=255, fondo=0)')
im = axes[2].imshow(mid, cmap='nipy_spectral', vmin=0, vmax=22)
n_v = len(set(mid.flatten()) - {0})
axes[2].set_title(f'Mascara multiclase ID (uint16)\n{n_v} vertebras anotadas')
plt.colorbar(im, ax=axes[2], label='ID de vertebra')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'comparacion_mascaras.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Control de calidad

Esta seccion deja un unico filtro de calidad antes de dividir los datos. La idea no es descartar imagenes "dificiles", sino separar dos tipos de hallazgos:

- **Errores excluyentes**: impiden entrenar o evaluar de forma supervisada, por ejemplo imagen faltante, mascara faltante, archivo ilegible, mascara vacia o etiquetas fuera del rango `0..17`.
- **Advertencias no excluyentes**: describen posibles limitaciones del dato, pero no justifican eliminarlo por si solas. En este dataset, la resolucion baja del lado menor cae en esta categoria porque muchas radiografias son naturalmente angostas; excluirlas sesgaria mucho el conjunto, especialmente los casos sanos.

Por eso `df_ok` conserva todos los casos sin errores excluyentes y `df_bad` solo contiene casos que no pueden usarse de forma confiable. Las advertencias quedan documentadas en columnas separadas para poder reportarlas o revisarlas, pero no reducen el dataset de entrenamiento/evaluacion.

<!-- codex-explicacion -->
El control de calidad evita que casos con etiquetas corruptas o inconsistentes entren al entrenamiento como si fueran ejemplos validos.


In [ ]:
def revisar_calidad(fila):
    errores = []
    advertencias = []
    img = None
    mask = None

    if not fila['tiene_img']:
        errores.append('imagen_faltante')
    else:
        try:
            img = np.array(Image.open(fila['ruta_img']).convert('L'))
            h, w = img.shape
            if min(h, w) < 256:
                advertencias.append(f'resolucion_baja:{w}x{h}')
            if img.std() < 5:
                errores.append('imagen_casi_uniforme')
        except Exception as e:
            errores.append(f'imagen_ilegible:{e}')

    if not fila['tiene_mask_id']:
        errores.append('mascara_multiclase_faltante')
    else:
        try:
            mask = np.array(Image.open(fila['ruta_mask_id']))
            if img is not None and mask.shape != img.shape:
                advertencias.append(f'shape_distinto:img={img.shape[1]}x{img.shape[0]},mask={mask.shape[1]}x{mask.shape[0]}')
            if mask.max() == 0:
                errores.append('mascara_vacia')

            ids = set(np.unique(mask).astype(int))
            ids_fuera_rango = sorted(i for i in ids if i < 0 or i > N_CLASES)
            if ids_fuera_rango:
                errores.append(f'ids_fuera_rango:{ids_fuera_rango}')
        except Exception as e:
            errores.append(f'mascara_ilegible:{e}')

    return errores, advertencias


resultados_qc = [revisar_calidad(f) for _, f in tqdm(df.iterrows(), total=len(df), desc='Control de calidad')]
df['errores_qc'] = [r[0] for r in resultados_qc]
df['advertencias_qc'] = [r[1] for r in resultados_qc]
df['n_errores_qc'] = df['errores_qc'].apply(len)
df['n_advertencias_qc'] = df['advertencias_qc'].apply(len)

df_ok = df[df['n_errores_qc'] == 0].copy()
df_bad = df[df['n_errores_qc'] > 0].copy()

print(f'Total en indice          : {len(df)}')
print(f'Casos utilizables df_ok  : {len(df_ok)}')
print(f'Casos excluidos df_bad   : {len(df_bad)}')
print(f'Casos con advertencias   : {int((df["n_advertencias_qc"] > 0).sum())}')

if not df_bad.empty:
    print('\nCasos excluidos por errores de calidad:')
    display(df_bad[['patient_id', 'tipo', 'errores_qc', 'advertencias_qc']])

if (df['n_advertencias_qc'] > 0).any():
    print('\nResumen de advertencias no excluyentes:')
    advertencias_flat = (
        df.explode('advertencias_qc')
          .dropna(subset=['advertencias_qc'])
          .assign(tipo_advertencia=lambda x: x['advertencias_qc'].str.split(':').str[0])
          .groupby(['tipo', 'tipo_advertencia'])
          .size()
          .reset_index(name='n')
    )
    display(advertencias_flat)
        


---
## 6. Normalización y preprocesamiento para MedSAM

MedSAM espera imágenes RGB de 1024×1024. Las radiografías originales son monocanal y tienen tamaños variables, por lo que se aplica un preprocesamiento único y reproducible:

1. Lectura en escala de grises.
2. CLAHE para mejorar contraste local.
3. Letterbox resize a 1024×1024, conservando la proporción anatómica.
4. Duplicación del canal gris a RGB durante la exportación MedSAM.

Las máscaras se redimensionan con interpolación `nearest` para conservar exactamente los IDs anatómicos. Usar interpolación bilineal en máscaras sería incorrecto porque crearía valores de clase intermedios que no existen.


In [ ]:
CONFIGS = {
    'medsam': (1024, 1024),
}

def preproc_img(ruta: str, size: tuple) -> np.ndarray:
    """
    Pipeline de preprocesamiento de radiografía:
      1. Lectura en escala de grises
      2. CLAHE (mejora contraste local)
      3. Letterbox resize (mantiene aspecto + padding negro)
      4. Normalización a float32 en [0, 1]
    """
    img = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f'No se pudo leer: {ruta}')

    # CLAHE: mejora el contraste local sin saturar zonas uniformes
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    # Letterbox: escalar con el factor que no supere ninguna dimensión
    th, tw = size
    h, w   = img.shape
    s      = min(tw / w, th / h)
    nw, nh = int(w * s), int(h * s)
    img_r  = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LANCZOS4)

    # Centrar la imagen escalada en un canvas de padding negro
    canvas = np.zeros((th, tw), np.uint8)
    yo, xo = (th - nh) // 2, (tw - nw) // 2
    canvas[yo:yo+nh, xo:xo+nw] = img_r

    return canvas.astype(np.float32) / 255.0


def preproc_mask(ruta: str, size: tuple) -> np.ndarray:
    """
    Redimensiona la mascara uint16 con interpolacion NEAREST.
    NEAREST es obligatorio: preserva los IDs exactos de vertebras.
    PIL maneja uint16 correctamente; cv2 lo truncaria a uint8.
    """
    mask  = np.array(Image.open(ruta))   # uint16
    th, tw = size
    h, w   = mask.shape
    s      = min(tw / w, th / h)
    nw, nh = int(w * s), int(h * s)
    m_r    = np.array(Image.fromarray(mask).resize((nw, nh), Image.NEAREST))

    canvas = np.zeros((th, tw), np.uint16)
    yo, xo = (th - nh) // 2, (tw - nw) // 2
    canvas[yo:yo+nh, xo:xo+nw] = m_r
    return canvas


# Verificacion visual con la primera imagen del dataset limpio
ej = df_ok.iloc[0]
ip = preproc_img(ej['ruta_img'], (512, 512))
mp = preproc_mask(ej['ruta_mask_id'], (512, 512))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(ip, cmap='gray')
axes[0].set_title(f"Imagen preprocesada 512x512\n{ej['patient_id']} — CLAHE + letterbox")
im = axes[1].imshow(mp, cmap='nipy_spectral', vmin=0, vmax=22)
axes[1].set_title(f'Mascara preprocesada 512x512 (uint16)\nClases presentes: {sorted(set(mp.flatten())-{0})}')
plt.colorbar(im, ax=axes[1])
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
print(f'Imagen  -> shape={ip.shape}, rango=[{ip.min():.2f}, {ip.max():.2f}]')
print(f'Mascara -> dtype={mp.dtype}, shape={mp.shape}')


## 7. División train / val / test estratificada clínicamente

La división del dataset se hace antes de cualquier entrenamiento y se mantiene fija para todas las variantes del modelo. Esto evita que una variante vea ejemplos distintos a otra y permite comparar resultados de forma justa.

Mantener solo la proporción `sano` / `escoliosis` es necesario, pero no suficiente en este dataset. Dentro de los casos con escoliosis hay una diferencia clínica grande entre casos no severos y severos según el ángulo de Cobb. Si la división fuera completamente aleatoria, validación o test podrían quedar con una concentración accidental de casos severos, haciendo que las métricas dependan del azar de la partición.

Por eso se usa una estratificación con tres grupos:

- `sano`
- `escoliosis_no_severa`
- `escoliosis_severa`

Esta clasificación se usa principalmente para **repartir mejor la severidad clínica entre `train`, `val` y `test`**. No significa que los tres grupos tengan el mismo peso estadístico para sacar conclusiones independientes.

En los datos válidos actuales hay aproximadamente 71 casos sanos, 28 con escoliosis no severa y 150 con escoliosis severa. Con una división 70/15/15, el grupo `escoliosis_no_severa` queda con pocos casos en validación y test. Por eso sus métricas por estrato deben leerse como exploratorias: ayudan a detectar sesgos o fallos evidentes, pero no permiten afirmar con mucha fuerza el desempeño específico en ese subgrupo.

No se separa en `leve`, `moderada` y `severa` porque el dataset tiene muy pocos casos leves. Agrupar los casos no severos evita splits inválidos o demasiado inestables y mantiene una evaluación metodológicamente más clara.
    


In [ ]:
df_limpio = df_ok[df_ok['tiene_img'] & df_ok['tiene_mask_id']].copy().reset_index(drop=True)
print(f'Dataset limpio total: {len(df_limpio)} imagenes')
print(df_limpio['tipo'].value_counts())
print()

# Integrar severidad Cobb para estratificar clínicamente.
# Normales no tienen Cobb en la tabla; se conservan como estrato "sano".
if 'df_cobb' in globals() and len(df_cobb) > 0:
    df_limpio = df_limpio.merge(
        df_cobb[['patient_id', 'cobb_deg']].drop_duplicates('patient_id'),
        on='patient_id',
        how='left'
    )
else:
    df_limpio['cobb_deg'] = np.nan

def severidad_para_split(row):
    if row['tipo'] == 'sano':
        return 'sano'

    cobb = row.get('cobb_deg', np.nan)
    if pd.isna(cobb):
        return 'escoliosis_no_severa'

    # Corte clínico principal: severa si Cobb >= 40 grados.
    # Leve/moderada se agrupan porque hay muy pocos casos leves para estratificar aparte.
    if cobb >= 40:
        return 'escoliosis_severa'
    return 'escoliosis_no_severa'

df_limpio['estrato_split'] = df_limpio.apply(severidad_para_split, axis=1)

print('Distribucion de estratos para split:')
print(df_limpio['estrato_split'].value_counts())
print()

# Validacion defensiva: cada estrato debe tener suficientes casos para sobrevivir
# a la doble particion train/temp y val/test.
conteo_estratos = df_limpio['estrato_split'].value_counts()
estratos_pequenos = conteo_estratos[conteo_estratos < 7]
if len(estratos_pequenos) > 0:
    raise ValueError(
        'Hay estratos con muy pocos casos para una division 70/15/15. '
        f'Revisar agrupacion: {estratos_pequenos.to_dict()}'
    )

# Paso 1: separar 70% train del 30% restante manteniendo estratos clinicos.
df_tr, df_vt = train_test_split(
    df_limpio,
    test_size=0.30,
    stratify=df_limpio['estrato_split'],
    random_state=SEED
)

# Paso 2: dividir el 30% restante en mitades iguales (15% val + 15% test),
# manteniendo nuevamente la proporcion de estratos.
df_val, df_te = train_test_split(
    df_vt,
    test_size=0.50,
    stratify=df_vt['estrato_split'],
    random_state=SEED
)

df_tr  = df_tr.copy();  df_tr['split']  = 'train'
df_val = df_val.copy(); df_val['split'] = 'val'
df_te  = df_te.copy();  df_te['split']  = 'test'
df_splits = pd.concat([df_tr, df_val, df_te], ignore_index=True)

# Verificar que la estratificacion funciono correctamente.
tabla_tipo = (
    df_splits
    .groupby(['split', 'tipo']).size()
    .unstack(fill_value=0)
    .reindex(['train', 'val', 'test'])
)

tabla_estrato = (
    df_splits
    .groupby(['split', 'estrato_split']).size()
    .unstack(fill_value=0)
    .reindex(['train', 'val', 'test'])
)

print('Distribucion por split y tipo:')
display(tabla_tipo)
print()
print('Distribucion por split y estrato clinico:')
display(tabla_estrato)
print()
print(f'Totales -> train={len(df_tr)} | val={len(df_val)} | test={len(df_te)}')

# Guardar CSV para reproducibilidad futura.
csv_path = OUTPUT_ROOT / 'splits_estratificados.csv'
cols_split = [
    'patient_id', 'tipo', 'estrato_split', 'cobb_deg', 'split',
    'ruta_img', 'ruta_mask_id'
]
df_splits[cols_split].to_csv(csv_path, index=False)
print(f'splits_estratificados.csv guardado en: {csv_path}')


---
## 8. Exportación MedSAM

Esta sección exporta exclusivamente el formato que usa MedSAM:

- `images/`: radiografías preprocesadas en RGB, 1024×1024.
- `masks/`: máscaras multiclase preprocesadas, conservando IDs anatómicos.
- `prompts.json`: bounding boxes por vértebra, calculadas desde la máscara anotada.

Importante: estas cajas son `bbox_gt`, es decir, prompts ideales derivados del ground truth. Sirven para evaluar la capacidad de segmentación condicionada de MedSAM, pero no representan todavía detección automática de vértebras.


In [ ]:
def export_medsam(df_sub, split, size, out):
    """Exporta imagenes RGB, mascaras y bounding boxes (prompts) para MedSAM."""
    (out / split / 'images').mkdir(parents=True, exist_ok=True)
    (out / split / 'masks').mkdir(parents=True, exist_ok=True)
    prompts = []

    for _, f in tqdm(df_sub.iterrows(), total=len(df_sub), desc=f'MedSAM-{split}'):
        img  = preproc_img(f['ruta_img'], size)
        mask = preproc_mask(f['ruta_mask_id'], size)
        pid  = f['patient_id']

        # SAM/MedSAM espera RGB: duplicar el canal gris en 3 canales.
        rgb  = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
        cv2.imwrite(str(out / split / 'images' / f'{pid}.png'), rgb)
        Image.fromarray(mask).save(str(out / split / 'masks' / f'{pid}.png'))

        # Bounding box por vertebra como prompt ideal.
        # Estas cajas vienen de la mascara ground truth, no de un detector automatico.
        bboxes = {}
        for id_v in sorted(set(mask.flatten().tolist()) - {0}):
            ys, xs = np.where(mask == id_v)
            bboxes[int(id_v)] = {
                'vertebra' : MAPEO_ID.get(int(id_v), f'cls_{id_v}'),
                'bbox_xyxy': [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())],
                'prompt_origen': 'bbox_gt'
            }
        prompts.append({'patient_id': pid, 'prompts': bboxes})

    with open(out / split / 'prompts.json', 'w') as fh:
        json.dump(prompts, fh, indent=2, ensure_ascii=False)

print('Funcion de exportacion MedSAM definida correctamente')


### Ejecutar la exportación MedSAM

La siguiente celda exporta `train`, `val` y `test` en formato MedSAM. Este proceso puede tardar algunos minutos porque todas las imágenes y máscaras se redimensionan a 1024×1024.


In [ ]:
# Ejecutar exportacion MedSAM para todos los splits
EXPORTS = [('train', df_tr), ('val', df_val), ('test', df_te)]

out_medsam = OUTPUT_ROOT / 'medsam'
size_medsam = CONFIGS['medsam']

print(f'\n{"="*50}')
print(f'Exportando MEDSAM — {size_medsam[0]}x{size_medsam[1]} px')
print('=' * 50)

for sname, dsub in EXPORTS:
    export_medsam(dsub, sname, size_medsam, out_medsam)

print('\nExportacion MedSAM completa.')


In [ ]:
meta = {
    'fecha'         : str(pd.Timestamp.now()),
    'dataset'       : 'Scoliosis_Dataset',
    'diccionario'   : str(LABELS_DICT_PATH),
    'indice'        : str(DATASET_INDEX_PATH),
    'estructura_dataset': {
        'imagenes_sanos'     : 'Normal/N_{id}.jpg',
        'imagenes_escoliosis': 'Scoliosis/S_{id}.jpg',
        'mascaras_id_png'    : 'LabelMultiClass_ID_PNG/LabelMulti_{N|S}_{id}.png (IDs 0..17)',
        'mascaras_binarias'  : 'LabelBinaryJPG/Label_{N|S}_{id}.jpg',
        'metricas_cobb'      : 'RadiographMetrics/metricas_cobb_resumen_recalculado.csv',
    },
    'totales': {
        'sanos'     : int((df_limpio['tipo'] == 'sano').sum()),
        'escoliosis': int((df_limpio['tipo'] == 'escoliosis').sum()),
        'limpio'    : len(df_limpio),
        'train'     : len(df_tr),
        'val'       : len(df_val),
        'test'      : len(df_te),
    },
    'preprocesamiento': {
        'imagen' : 'escala de grises + CLAHE (clipLimit=2, tile=8x8) + letterbox 1024x1024 + RGB',
        'mascara': 'letterbox 1024x1024 + interpolacion NEAREST (IDs 0..17)',
        'prompt' : 'bbox_gt derivado de mascara ground truth',
    },
    'clases'  : {'n': N_CLASES, 'nombres': NOMBRES_CLASES},
    'modelo'  : {'nombre': 'MedSAM', 'size': list(CONFIGS['medsam'])},
    'estratificacion': 'sano / escoliosis_no_severa / escoliosis_severa',
    'seed'    : SEED,
}

with open(OUTPUT_ROOT / 'metadata_medsam.json', 'w') as fh:
    json.dump(meta, fh, indent=2, ensure_ascii=False, default=str)

t = meta['totales']
print('=' * 65)
print('  RESUMEN FINAL — PREPARACION MEDSAM')
print('=' * 65)
print(f"  Sanos                : {t['sanos']}  (Normal/N_{{id}}.jpg)")
print(f"  Escoliosis           : {t['escoliosis']} (Scoliosis/S_{{id}}.jpg)")
print(f"  Dataset limpio       : {t['limpio']} imagenes")
print(f"  Train                : {t['train']} imagenes  (70%)")
print(f"  Val                  : {t['val']}  imagenes  (15%)")
print(f"  Test                 : {t['test']}  imagenes  (15%)")
print(f"  Clases               : {N_CLASES} (T1-T12, L1-L5)")
print(f"  Exportado            : MedSAM 1024x1024")
print(f"  Estratificacion      : sano / escoliosis_no_severa / escoliosis_severa")
print(f"  Prompt               : bbox_gt desde mascara anotada")
print('=' * 65)


# 9. MedSAM: protocolo experimental

A partir de esta seccion el notebook pasa de preparar datos a probar MedSAM. Para que los resultados sean interpretables, la seccion se organiza como un protocolo experimental:

1. Se cargan los datos exportados y las funciones comunes.
2. Se hacen sanity checks no reportables para confirmar que imagenes, mascaras y prompts funcionan.
3. Se entrenan escenarios de prueba.
4. Cada escenario se evalua sobre el mismo split de validacion.
5. Solo al final, cuando el modelo queda elegido, se usa test.

Hay dos formas metodologicas de leer los experimentos:

- **Escenarios independientes**: cada cambio arranca desde el mismo checkpoint inicial. Sirven para responder "que aporta este cambio por si solo?".
- **Escenarios acumulativos**: un cambio se agrega encima de otro cuando el resultado previo parece prometedor. Sirven para responder "que pasa si construimos progresivamente una mejor receta?".

En esta version, varios bloques posteriores estan escritos como una cadena acumulativa: primero se ajusta el decoder, luego se prueba reequilibrio, despues una variante suave y finalmente encoder parcial. Esto puede tener sentido como exploracion, pero debe nombrarse asi para no confundirlo con variantes independientes.

La metrica principal para tomar decisiones sera Dice macro por paciente en `val`, complementada con IoU, resultados por vertebra y resumen por estrato clinico. Las visualizaciones y metricas de una sola muestra se usan solo para monitoreo.

<!-- codex-explicacion -->
Aqui empieza el protocolo de experimentos con MedSAM. La idea fue probar variantes de entrenamiento manteniendo el mismo esquema de datos y metricas.


In [ ]:
# Mapa de escenarios que se usara como guia de lectura.
# No entrena modelos; solo documenta que cambia y como se debe comparar.
PLAN_ESCENARIOS_MEDSAM = pd.DataFrame([
    {
        "escenario": "sanity_checks",
        "tipo": "no_reportable",
        "parte_de": "MedSAM preentrenado",
        "cambio": "verificar prompts, formatos e inferencia puntual",
        "decision": "no se usa para elegir modelo",
    },
    {
        "escenario": "decoder_basico",
        "tipo": "cambio_1",
        "parte_de": "MedSAM preentrenado",
        "cambio": "entrenar mask_decoder, mantener encoder congelado",
        "decision": "comparar en val completo",
    },
    {
        "escenario": "decoder_plus_weighted_sampler",
        "tipo": "cambio_1_mas_cambio_2",
        "parte_de": "decoder_basico",
        "cambio": "agregar muestreo ponderado para reducir desbalance",
        "decision": "comparar contra decoder_basico en val",
    },
    {
        "escenario": "decoder_plus_weighted_plus_soft",
        "tipo": "acumulativo",
        "parte_de": "decoder_plus_weighted_sampler",
        "cambio": "usar ponderacion suave en la loss",
        "decision": "comparar contra escenarios anteriores en val",
    },
    {
        "escenario": "decoder_plus_weighted_plus_soft_plus_encoder",
        "tipo": "acumulativo_mayor_capacidad",
        "parte_de": "decoder_plus_weighted_plus_soft",
        "cambio": "abrir ultimo bloque del encoder con learning rate bajo",
        "decision": "usar solo si mejora val sin inestabilidad evidente",
    },
    {
        "escenario": "augmentation_variant",
        "tipo": "futuro",
        "parte_de": "modelo final candidato",
        "cambio": "entrenar con variaciones/cortes controlados",
        "decision": "debe revalidarse antes de tocar test",
    },
])

display(PLAN_ESCENARIOS_MEDSAM)
                        


In [ ]:
# ==========================================
# 1) IMPORTS Y CONFIGURACION GENERAL
# ==========================================

%matplotlib inline
import os
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


## 9.1 Configuracion comun de clases

Esta subseccion fija las clases que se mantendran constantes en todos los escenarios. En este dataset las mascaras ya vienen con una codificacion limpia: `0` es fondo, `1..12` son `T1..T12` y `13..17` son `L1..L5`. Por eso no se hace una remapificacion desde clases cervicales u otras etiquetas externas.

Los diccionarios de clase se definen para que todas las funciones usen la misma relacion entre nombre anatomico e ID numerico. Esto evita que un experimento cambie accidentalmente la interpretacion de una vertebra.

<!-- codex-explicacion -->
Se define el mapa anatomico T1-L5. Mantener esta tabla fija es clave para que las metricas estrictas sean comparables.


In [ ]:
# ==========================================
# 2) CLASES DEL DATASET Y CLASES OBJETIVO
# ==========================================

# La version limpia ya usa IDs consecutivos:
# 0 = fondo, 1..12 = T1..T12, 13..17 = L1..L5.
CLASES_OBJETIVO = NOMBRES_CLASES.copy()
N_CLASES = len(CLASES_OBJETIVO)

VERTEBRA_TO_ID = {nombre: idx for idx, nombre in MAPEO_ID.items() if idx != 0}
ID_TO_VERTEBRA = {idx: nombre for nombre, idx in VERTEBRA_TO_ID.items()}

# En esta version el ID real de la mascara y el ID local de evaluacion son iguales.
CLASS_TO_ID = VERTEBRA_TO_ID.copy()
ID_TO_CLASS = {idx: nombre for nombre, idx in CLASS_TO_ID.items()}

print("=" * 80)
print("Configuracion actual para MEDSAM")
print("=" * 80)
print("Dataset activo: Scoliosis_Dataset limpio")
print("Clases objetivo (17):", CLASES_OBJETIVO)
print("IDs de mascara para T1-L5:")
for c in CLASES_OBJETIVO:
    print(f"  {c:>3} -> {VERTEBRA_TO_ID[c]}")
print("=" * 80)


In [ ]:
# ==========================================
# 3) FUNCIONES AUXILIARES PARA BUSCAR ARCHIVOS
# ==========================================

def buscar_archivo(nombre_archivo, raiz_inicial=Path("."), max_resultados=20):
    encontrados = []
    for p in raiz_inicial.rglob(nombre_archivo):
        encontrados.append(p)
        if len(encontrados) >= max_resultados:
            break
    return encontrados

EXTS_IMG = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

def buscar_archivo_por_stem(carpeta, stem):
    for ext in EXTS_IMG:
        p = carpeta / f"{stem}{ext}"
        if p.exists():
            return p
    return None


## 9.2 Carga de artefactos exportados

Aqui se cargan los archivos ya exportados para MedSAM: imagenes preprocesadas, mascaras, prompts y metadatos. La evaluacion y el entrenamiento deben usar estos artefactos, no rutas crudas distintas.

Esto garantiza que todos los escenarios trabajen con la misma version de datos, el mismo tamano de entrada, la misma codificacion de mascaras y los mismos prompts `bbox_gt`.

<!-- codex-explicacion -->
Esta carga verifica que la exportacion MedSAM quedo disponible. Si falla aqui, el problema no es el modelo sino la preparacion del dataset.

<!-- codex-normalizacion -->
Desde este punto, el proyecto deja de depender de las rutas crudas de imagen/m?scara para los experimentos principales. La carga de artefactos exportados asegura que MedSAM y las redes de cajas trabajen sobre la misma base procesada.


In [ ]:
# ==========================================
# 4) DETECTAR LA RAIZ DE MEDSAM
# ==========================================

MEDSAM_DATA_ROOT = None

variables_candidatas = [
    "MEDSAM_DATA_ROOT",
    "medsam_data_root",
    "MEDSAM_ROOT",
    "medsam_root",
    "OUT_MEDSAM",
    "out_medsam",
    "EXPORT_MEDSAM_DIR",
    "export_medsam_dir"
]

for var_name in variables_candidatas:
    if var_name in globals():
        try:
            MEDSAM_DATA_ROOT = Path(globals()[var_name])
            if MEDSAM_DATA_ROOT.exists():
                print(f"Se usara {var_name} = {MEDSAM_DATA_ROOT}")
                break
        except Exception:
            pass

if MEDSAM_DATA_ROOT is None:
    prompts_encontrados = buscar_archivo("prompts.json", Path("."))
    prompts_medsam = [p for p in prompts_encontrados if "medsam" in str(p).lower()]

    if len(prompts_medsam) > 0:
        MEDSAM_DATA_ROOT = prompts_medsam[0].parent.parent
        print(f"MEDSAM_DATA_ROOT detectado automaticamente: {MEDSAM_DATA_ROOT}")
    elif len(prompts_encontrados) > 0:
        MEDSAM_DATA_ROOT = prompts_encontrados[0].parent.parent
        print(f"MEDSAM_DATA_ROOT detectado automaticamente: {MEDSAM_DATA_ROOT}")
    else:
        raise FileNotFoundError(
            "No se encontro prompts.json. "
            "Corre primero la exportacion a MEDSAM o indica el path exacto."
        )

MEDSAM_DATA_ROOT = Path(MEDSAM_DATA_ROOT)
print("MEDSAM_DATA_ROOT final:", MEDSAM_DATA_ROOT)


In [ ]:
# ==========================================
# 5) RESOLVER SPLITS Y PATHS
# ==========================================

SPLITS = ["train", "val", "test"]

def resolver_paths_split(split):
    split_dir = MEDSAM_DATA_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(f"No existe el split: {split_dir}")

    prompts_path = split_dir / "prompts.json"
    if not prompts_path.exists():
        raise FileNotFoundError(f"No existe prompts.json en: {split_dir}")

    candidatos_img = ["images", "imgs", "image", "jpg", "png"]
    candidatos_mask = ["masks", "labels", "mask", "annotations"]

    image_dir = None
    mask_dir = None

    for c in candidatos_img:
        p = split_dir / c
        if p.exists() and p.is_dir():
            image_dir = p
            break

    for c in candidatos_mask:
        p = split_dir / c
        if p.exists() and p.is_dir():
            mask_dir = p
            break

    if image_dir is None:
        raise FileNotFoundError(f"No encontre carpeta de imagenes dentro de {split_dir}")

    if mask_dir is None:
        raise FileNotFoundError(f"No encontre carpeta de mascaras dentro de {split_dir}")

    return split_dir, image_dir, mask_dir, prompts_path

SPLIT_INFO = {}
for split in SPLITS:
    split_dir, image_dir, mask_dir, prompts_path = resolver_paths_split(split)
    SPLIT_INFO[split] = {
        "split_dir": split_dir,
        "image_dir": image_dir,
        "mask_dir": mask_dir,
        "prompts_path": prompts_path
    }

print("\nResumen de paths detectados:")
for split in SPLITS:
    print(f"\n[{split}]")
    print(" split_dir   :", SPLIT_INFO[split]["split_dir"])
    print(" image_dir   :", SPLIT_INFO[split]["image_dir"])
    print(" mask_dir    :", SPLIT_INFO[split]["mask_dir"])
    print(" prompts_path:", SPLIT_INFO[split]["prompts_path"])


In [ ]:
# ==========================================
# 6) CARGAR PROMPTS Y REVISAR SU ESTRUCTURA
# ==========================================

PROMPTS = {}
for split in SPLITS:
    with open(SPLIT_INFO[split]["prompts_path"], "r", encoding="utf-8") as f:
        PROMPTS[split] = json.load(f)

print("\nCantidad de entradas en prompts.json:")
for split in SPLITS:
    print(f"  {split}: {len(PROMPTS[split])}")

for split in SPLITS:
    print("\n" + "=" * 80)
    print(f"SPLIT: {split}")
    print("type(PROMPTS[split]):", type(PROMPTS[split]))

    if isinstance(PROMPTS[split], list):
        print("len:", len(PROMPTS[split]))
        if len(PROMPTS[split]) > 0:
            print("type primer elemento:", type(PROMPTS[split][0]))
            if isinstance(PROMPTS[split][0], dict):
                print("keys primer elemento:", list(PROMPTS[split][0].keys()))
                print("primer elemento completo:")
                print(PROMPTS[split][0])


## 9.3 Funciones comunes de mascaras, prompts y metricas

Estas funciones son infraestructura comun. No representan un experimento por si mismas.

Su papel es cargar imagenes y mascaras, filtrar vertebras objetivo, construir mascaras binarias por vertebra, reconstruir una mascara semantica T1-L5 y calcular Dice/IoU. Todas las variantes deben usar estas mismas funciones para que la comparacion sea justa.

<!-- codex-explicacion -->
Estas funciones estandarizan como se calcula Dice/IoU. La comparacion entre escenarios depende de que todos usen exactamente las mismas metricas.


In [ ]:
# ==========================================
# 7) FUNCIONES DE CARGA, FILTRADO Y MASCARAS
# ==========================================

def cargar_imagen(path_imagen):
    img = Image.open(path_imagen).convert("RGB")
    return np.array(img)

def cargar_mascara(path_mascara):
    mask = Image.open(path_mascara)
    return np.array(mask).astype(np.int32)

def normalizar_nombre_vertebra(nombre):
    if nombre is None:
        return None
    nombre = str(nombre).strip().upper()
    return nombre if nombre in CLASES_OBJETIVO else None

def filtrar_prompts_t1_l5(prompts_split):
    # Nota metodologica: los prompts se leen del archivo exportado; en esta version vienen de las mascaras GT.
    """
    Estructura real:
    [
        {
            "patient_id": "...",
            "prompts": {
                "1": {"vertebra": "T1", "bbox_xyxy": [...]},
                ...
            }
        },
        ...
    ]
    """
    prompts_filtrados = {}

    if not isinstance(prompts_split, list):
        raise TypeError(f"Se esperaba list en prompts_split y llegó {type(prompts_split)}")

    for item in prompts_split:
        if not isinstance(item, dict):
            continue

        patient_id = item.get("patient_id")
        prompts_item = item.get("prompts")

        if patient_id is None or prompts_item is None:
            continue

        sub = {}

        if isinstance(prompts_item, dict):
            for _, info in prompts_item.items():
                if not isinstance(info, dict):
                    continue

                nombre = normalizar_nombre_vertebra(info.get("vertebra"))
                if nombre is not None:
                    sub[nombre] = info

        elif isinstance(prompts_item, list):
            for info in prompts_item:
                if not isinstance(info, dict):
                    continue

                nombre = normalizar_nombre_vertebra(info.get("vertebra"))
                if nombre is not None:
                    sub[nombre] = info

        if len(sub) > 0:
            prompts_filtrados[patient_id] = sub

    return prompts_filtrados

def construir_mask_binaria_vertebra(mask_multiclase, vertebra_objetivo):
    """
    Convierte la mascara multiclase global en una mascara binaria para una sola vertebra.
    """
    if vertebra_objetivo not in VERTEBRA_TO_ID:
        raise ValueError(f"Vertebra no conocida: {vertebra_objetivo}")

    vertebra_id_real = VERTEBRA_TO_ID[vertebra_objetivo]
    mask_bin = (mask_multiclase == vertebra_id_real).astype(np.uint8)
    return mask_bin

def construir_mask_multiclase_t1_l5(mask_multiclase):
    """
    Devuelve la mascara T1-L5 en IDs locales 1..17:
    0 = fondo
    1..17 = T1..L5
    """
    # En Scoliosis_Dataset las mascaras ya vienen en 0..17.
    # Se fuerza uint8 y se eliminan posibles IDs fuera del diccionario por seguridad.
    nueva = mask_multiclase.astype(np.uint8).copy()
    nueva[~np.isin(nueva, list(range(0, N_CLASES + 1)))] = 0
    return nueva

def extraer_bbox_desde_mask(mask_binaria):
    ys, xs = np.where(mask_binaria > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    return [int(x0), int(y0), int(x1), int(y1)]

def resolver_paths_muestra(split, patient_id):
    image_dir = SPLIT_INFO[split]["image_dir"]
    mask_dir = SPLIT_INFO[split]["mask_dir"]

    path_imagen = buscar_archivo_por_stem(image_dir, patient_id)
    path_mascara = buscar_archivo_por_stem(mask_dir, patient_id)

    if path_imagen is None:
        raise FileNotFoundError(f"No encontre imagen para {patient_id} en {image_dir}")

    if path_mascara is None:
        raise FileNotFoundError(f"No encontre mascara para {patient_id} en {mask_dir}")

    return path_imagen, path_mascara


In [ ]:
# ==========================================
# 8) FILTRAR T1-L5 Y TOMAR UNA MUESTRA DE PRUEBA
# ==========================================

PROMPTS_FILTRADOS = {
    split: filtrar_prompts_t1_l5(PROMPTS[split])
    for split in SPLITS
}

print("\nCantidad de imagenes con prompts T1-L5:")
for split in SPLITS:
    print(f"  {split}: {len(PROMPTS_FILTRADOS[split])}")

for split in SPLITS:
    print("\n" + "=" * 80)
    print(f"SPLIT: {split}")
    print("imagenes filtradas:", len(PROMPTS_FILTRADOS[split]))

    if len(PROMPTS_FILTRADOS[split]) > 0:
        sample_key = next(iter(PROMPTS_FILTRADOS[split]))
        print("sample_key:", sample_key)
        print("vertebras disponibles:", list(PROMPTS_FILTRADOS[split][sample_key].keys()))
        primera_vertebra = next(iter(PROMPTS_FILTRADOS[split][sample_key]))
        print("ejemplo prompt:")
        print(PROMPTS_FILTRADOS[split][sample_key][primera_vertebra])

split = "train"

if len(PROMPTS_FILTRADOS[split]) == 0:
    raise RuntimeError("No se encontraron prompts T1-L5 en train.")

sample_key = next(iter(PROMPTS_FILTRADOS[split].keys()))
path_imagen, path_mascara = resolver_paths_muestra(split, sample_key)

imagen = cargar_imagen(path_imagen)
mascara = cargar_mascara(path_mascara)

# mascara remapeada solo para visualizacion T1-L5
mascara_t1_l5 = construir_mask_multiclase_t1_l5(mascara)

prompts_sample = PROMPTS_FILTRADOS[split][sample_key]

print("\nMuestra seleccionada:")
print(" split       :", split)
print(" sample_key  :", sample_key)
print(" path_imagen :", path_imagen)
print(" path_mascara:", path_mascara)
print(" vertebras en prompts:", list(prompts_sample.keys()))
print(" shape imagen:", imagen.shape)
print(" shape mask original :", mascara.shape)
print(" unique mask original:", np.unique(mascara)[:30])
print(" unique mask T1-L5   :", np.unique(mascara_t1_l5)[:30])

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(imagen)
ax[0].set_title("Imagen")
ax[0].axis("off")

ax[1].imshow(mascara_t1_l5, cmap="nipy_spectral")
ax[1].set_title("Mascara remapeada T1-L5")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 9) VISUALIZAR UNA VERTEBRA Y SU BBOX
# ==========================================

vertebra_objetivo = "L3" if "L3" in prompts_sample else list(prompts_sample.keys())[0]
prompt_info = prompts_sample[vertebra_objetivo]
bbox_prompt = prompt_info["bbox_xyxy"]

print("Vertebra objetivo :", vertebra_objetivo)
print("ID real en dataset:", VERTEBRA_TO_ID[vertebra_objetivo])
print("Prompt info       :", prompt_info)
print("BBox              :", bbox_prompt)

x0, y0, x1, y1 = bbox_prompt

fig, ax = plt.subplots(1, 2, figsize=(14, 7))

ax[0].imshow(imagen)
rect = plt.Rectangle(
    (x0, y0), x1 - x0, y1 - y0,
    fill=False, edgecolor="red", linewidth=2
)
ax[0].add_patch(rect)
ax[0].set_title(f"Imagen + bbox de {vertebra_objetivo}")
ax[0].axis("off")

crop = imagen[y0:y1, x0:x1]
ax[1].imshow(crop)
ax[1].set_title(f"Crop local de {vertebra_objetivo}")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 10) CONSTRUIR LA MASCARA BINARIA REAL DE LA VERTEBRA
# ==========================================

mask_bin_vertebra = construir_mask_binaria_vertebra(mascara, vertebra_objetivo)

print("Vertebra objetivo :", vertebra_objetivo)
print("ID real en dataset:", VERTEBRA_TO_ID[vertebra_objetivo])
print("Valores unicos mask_bin_vertebra:", np.unique(mask_bin_vertebra))

fig, ax = plt.subplots(1, 2, figsize=(14, 7))

ax[0].imshow(imagen)
ax[0].set_title(f"Imagen completa - {vertebra_objetivo}")
ax[0].axis("off")

ax[1].imshow(mask_bin_vertebra, cmap="gray")
ax[1].set_title(f"Mascara binaria global - {vertebra_objetivo}")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 11) VALIDAR BBOX VS MASCARA REAL DE ESA VERTEBRA
# ==========================================

x0, y0, x1, y1 = bbox_prompt

crop_img = imagen[y0:y1, x0:x1]
crop_mask_bin = mask_bin_vertebra[y0:y1, x0:x1]

bbox_desde_mask = extraer_bbox_desde_mask(mask_bin_vertebra)

print("BBox prompt       :", bbox_prompt)
print("BBox desde mascara:", bbox_desde_mask)
print("Valores unicos en crop_mask_bin:", np.unique(crop_mask_bin))
print("Pixeles positivos en crop_mask_bin:", int(crop_mask_bin.sum()))

fig, ax = plt.subplots(1, 3, figsize=(18, 6))

ax[0].imshow(crop_img)
ax[0].set_title(f"Crop imagen - {vertebra_objetivo}")
ax[0].axis("off")

ax[1].imshow(crop_mask_bin, cmap="gray")
ax[1].set_title(f"Crop mascara binaria - {vertebra_objetivo}")
ax[1].axis("off")

ax[2].imshow(crop_img)
overlay = np.zeros_like(crop_img)
overlay[..., 1] = crop_mask_bin * 255
ax[2].imshow(overlay, alpha=0.35)
ax[2].set_title(f"Overlay crop - {vertebra_objetivo}")
ax[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# CARGAR MODELO MEDSAM
# ==========================================

import torch
from segment_anything import sam_model_registry
from segment_anything import SamPredictor

# Ajusta este path si es necesario (según tu HTML ya lo tenías)
MedSAM_CKPT_PATH = "C:/Users/luisf/MedSAM/work_dir/MedSAM/medsam_vit_b.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Cargar modelo
medsam_model = sam_model_registry["vit_b"](checkpoint=MedSAM_CKPT_PATH)
medsam_model = medsam_model.to(device)
medsam_model.eval()

# Crear predictor
predictor = SamPredictor(medsam_model)

print("MedSAM cargado correctamente")


## 10. Sanity checks no reportables

Antes de entrenar o comparar modelos se hacen pruebas pequenas para confirmar que el pipeline esta conectado correctamente. Estas pruebas ayudan a detectar errores de formato, coordenadas, prompts o mascaras.

No se usan para afirmar desempeno del modelo ni para elegir una variante. Son controles tecnicos.

<!-- codex-explicacion -->
Estas pruebas no se reportan como resultados finales; sirven para detectar errores visuales o logicos antes de gastar tiempo en entrenamientos largos.


### 10.2 Primera inferencia puntual

Esta celda verifica que MedSAM pueda recibir la imagen y la caja, y que devuelva una mascara binaria. Es una prueba de funcionamiento del modelo y del predictor, no una evaluacion reportable.
            


In [ ]:
# ==========================================
# 13) PRIMERA INFERENCIA REAL CON MEDSAM
# ==========================================

box_np = np.array(bbox_prompt, dtype=np.float32)[None, :]

predictor.set_image(imagen)

masks, scores, logits = predictor.predict(
    box=box_np,
    multimask_output=False
)

mask_pred = masks[0].astype(np.uint8)

print("score MedSAM:", float(scores[0]))
print("shape mask_pred:", mask_pred.shape)
print("valores unicos mask_pred:", np.unique(mask_pred))
print("pixeles positivos pred:", int(mask_pred.sum()))


In [ ]:
# ==========================================
# 14) COMPARACION PREDICCION VS GROUND TRUTH
# ==========================================

def dice_score(pred, gt, eps=1e-8):
    pred = pred.astype(np.uint8)
    gt = gt.astype(np.uint8)
    inter = np.logical_and(pred == 1, gt == 1).sum()
    return (2 * inter + eps) / (pred.sum() + gt.sum() + eps)

def iou_score(pred, gt, eps=1e-8):
    pred = pred.astype(np.uint8)
    gt = gt.astype(np.uint8)
    inter = np.logical_and(pred == 1, gt == 1).sum()
    union = np.logical_or(pred == 1, gt == 1).sum()
    return (inter + eps) / (union + eps)

dice_v = dice_score(mask_pred, mask_bin_vertebra)
iou_v = iou_score(mask_pred, mask_bin_vertebra)

print(f"Dice {vertebra_objetivo}: {dice_v:.4f}")
print(f"IoU  {vertebra_objetivo}: {iou_v:.4f}")


### 10.3 Pruebas puntuales adicionales

Las siguientes celdas repiten la inferencia en otro paciente y en varias vertebras. Sirven para inspeccionar cualitativamente si el comportamiento es razonable antes de lanzar evaluaciones largas.

Estas metricas pueden orientar depuracion, pero no deben mezclarse con las tablas finales porque dependen del caso elegido manualmente.
        


In [ ]:
# ==========================================
# 15) VISUALIZACION FINAL: GT VS PRED
# ==========================================

fig, ax = plt.subplots(1, 3, figsize=(18, 8))

# imagen + bbox
ax[0].imshow(imagen)
rect = plt.Rectangle(
    (x0, y0), x1 - x0, y1 - y0,
    fill=False, edgecolor="red", linewidth=2
)
ax[0].add_patch(rect)
ax[0].set_title(f"Imagen + bbox ({vertebra_objetivo})")
ax[0].axis("off")

# GT
ax[1].imshow(imagen)
overlay_gt = np.zeros_like(imagen)
overlay_gt[..., 1] = mask_bin_vertebra * 255
ax[1].imshow(overlay_gt, alpha=0.30)
ax[1].set_title(f"Ground truth - {vertebra_objetivo}")
ax[1].axis("off")

# Pred
ax[2].imshow(imagen)
overlay_pred = np.zeros_like(imagen)
overlay_pred[..., 0] = mask_pred * 255
ax[2].imshow(overlay_pred, alpha=0.30)
ax[2].set_title(f"Prediccion MedSAM - {vertebra_objetivo}")
ax[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 12A) ELEGIR OTRO PACIENTE DE PRUEBA
# ==========================================l

split = "train"

# Puedes cambiar este valor manualmente si ya conoces uno bueno
PACIENTE_PRUEBA = None

# Si quieres excluir pacientes problemáticos visualmente, agrégalos aquí
PACIENTES_EXCLUIR = {"S_142"}

# Buscar pacientes con T1-L5 completo
pacientes_completos = []
for pid, prompts_pid in PROMPTS_FILTRADOS[split].items():
    if all(v in prompts_pid for v in CLASES_OBJETIVO):
        if pid not in PACIENTES_EXCLUIR:
            pacientes_completos.append(pid)

print("Cantidad de pacientes completos T1-L5:", len(pacientes_completos))
print("Primeros 15 candidatos:", pacientes_completos[:15])

if PACIENTE_PRUEBA is None:
    if len(pacientes_completos) == 0:
        raise RuntimeError("No encontré pacientes completos T1-L5 en este split.")
    PACIENTE_PRUEBA = pacientes_completos[0]

print("PACIENTE_PRUEBA seleccionado:", PACIENTE_PRUEBA)


In [ ]:
# ==========================================
# 12B) CARGAR PACIENTE DE PRUEBA
# ==========================================

sample_key = PACIENTE_PRUEBA

path_imagen, path_mascara = resolver_paths_muestra(split, sample_key)

imagen = cargar_imagen(path_imagen)
mascara = cargar_mascara(path_mascara)
mascara_t1_l5 = construir_mask_multiclase_t1_l5(mascara)
prompts_sample = PROMPTS_FILTRADOS[split][sample_key]

print("\nPaciente cargado:")
print(" split       :", split)
print(" sample_key  :", sample_key)
print(" path_imagen :", path_imagen)
print(" path_mascara:", path_mascara)
print(" vertebras disponibles:", list(prompts_sample.keys()))
print(" shape imagen:", imagen.shape)
print(" shape mascara:", mascara.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(imagen)
ax[0].set_title(f"Imagen - {sample_key}")
ax[0].axis("off")

ax[1].imshow(mascara_t1_l5, cmap="nipy_spectral")
ax[1].set_title(f"Mascara T1-L5 - {sample_key}")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 12C) SELECCIONAR VARIAS VERTEBRAS DE PRUEBA
# ==========================================

VERTEBRAS_PRUEBA = ["T8", "T12", "L2", "L3", "L5"]

# Dejar solo las que realmente existan en este paciente
VERTEBRAS_PRUEBA = [v for v in VERTEBRAS_PRUEBA if v in prompts_sample]

print("VERTEBRAS_PRUEBA:", VERTEBRAS_PRUEBA)

if len(VERTEBRAS_PRUEBA) == 0:
    raise RuntimeError("No hay vértebras válidas para probar en este paciente.")


In [ ]:
# ==========================================
# 13A) ASEGURAR PREDICTOR DE MEDSAM
# ==========================================

from segment_anything import SamPredictor

if "predictor" not in globals():
    if "medsam" in globals():
        predictor = SamPredictor(medsam)
    elif "medsam_model" in globals():
        predictor = SamPredictor(medsam_model)
    else:
        raise NameError(
            "No existe ni 'medsam' ni 'medsam_model' en memoria. "
            "Vuelve a correr la celda donde cargas MedSAM."
        )

print("Predictor listo.")


In [ ]:
# ==========================================
# 13B) INFERENCIA MEDSAM EN VARIAS VERTEBRAS
# ==========================================

predictor.set_image(imagen)

resultados = []

for vertebra_objetivo in VERTEBRAS_PRUEBA:
    prompt_info = prompts_sample[vertebra_objetivo]
    bbox_prompt = prompt_info["bbox_xyxy"]
    x0, y0, x1, y1 = bbox_prompt

    box_np = np.array(bbox_prompt, dtype=np.float32)[None, :]

    masks, scores, logits = predictor.predict(
        box=box_np,
        multimask_output=False
    )

    mask_pred = masks[0].astype(np.uint8)
    mask_gt = construir_mask_binaria_vertebra(mascara, vertebra_objetivo)

    inter = np.logical_and(mask_pred == 1, mask_gt == 1).sum()
    union = np.logical_or(mask_pred == 1, mask_gt == 1).sum()

    dice_v = (2 * inter + 1e-8) / (mask_pred.sum() + mask_gt.sum() + 1e-8)
    iou_v = (inter + 1e-8) / (union + 1e-8)

    resultados.append({
        "patient_id": sample_key,
        "vertebra": vertebra_objetivo,
        "id_real": VERTEBRA_TO_ID[vertebra_objetivo],
        "bbox_prompt": bbox_prompt,
        "score_medsam": float(scores[0]),
        "pix_gt": int(mask_gt.sum()),
        "pix_pred": int(mask_pred.sum()),
        "dice": float(dice_v),
        "iou": float(iou_v),
        "mask_gt": mask_gt,
        "mask_pred": mask_pred
    })

print("Inferencias completadas:", len(resultados))
for r in resultados:
    print(
        f"{r['vertebra']:>3} | "
        f"score={r['score_medsam']:.4f} | "
        f"Dice={r['dice']:.4f} | "
        f"IoU={r['iou']:.4f} | "
        f"GT={r['pix_gt']} | Pred={r['pix_pred']}"
    )


In [ ]:
# ==========================================
# 14A) TABLA RESUMEN
# ==========================================

import pandas as pd

df_resultados = pd.DataFrame(resultados)[
    ["patient_id", "vertebra", "id_real", "score_medsam", "pix_gt", "pix_pred", "dice", "iou"]
].sort_values(by="id_real")

display(df_resultados)

print("\nPromedio Dice:", df_resultados["dice"].mean())
print("Promedio IoU :", df_resultados["iou"].mean())
print("Promedio score MedSAM:", df_resultados["score_medsam"].mean())


In [ ]:
# ==========================================
# 15A) VISUALIZACION MULTIPLE GT VS PRED
# ==========================================

n = len(resultados)
fig, ax = plt.subplots(n, 3, figsize=(18, 6 * n))

if n == 1:
    ax = np.expand_dims(ax, axis=0)

for i, r in enumerate(sorted(resultados, key=lambda z: z["id_real"])):
    vertebra = r["vertebra"]
    bbox_prompt = r["bbox_prompt"]
    x0, y0, x1, y1 = bbox_prompt

    # imagen + bbox
    ax[i, 0].imshow(imagen)
    rect = plt.Rectangle(
        (x0, y0), x1 - x0, y1 - y0,
        fill=False, edgecolor="red", linewidth=2
    )
    ax[i, 0].add_patch(rect)
    ax[i, 0].set_title(f"{vertebra} | bbox")
    ax[i, 0].axis("off")

    # GT
    ax[i, 1].imshow(imagen)
    overlay_gt = np.zeros_like(imagen)
    overlay_gt[..., 1] = r["mask_gt"] * 255
    ax[i, 1].imshow(overlay_gt, alpha=0.30)
    ax[i, 1].set_title(f"{vertebra} | GT")
    ax[i, 1].axis("off")

    # Pred
    ax[i, 2].imshow(imagen)
    overlay_pred = np.zeros_like(imagen)
    overlay_pred[..., 0] = r["mask_pred"] * 255
    ax[i, 2].imshow(overlay_pred, alpha=0.30)
    ax[i, 2].set_title(
        f"{vertebra} | Pred\nDice={r['dice']:.3f} | IoU={r['iou']:.3f}"
    )
    ax[i, 2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 21) COMPARAR BBOX ORIGINAL VS BBOX AMPLIADA POCO
# ==========================================

def expand_bbox_xyxy_pequena(bbox, img_shape, frac_x=0.04, frac_y=0.06):
    x0, y0, x1, y1 = bbox
    h, w = img_shape[:2]

    bw = x1 - x0
    bh = y1 - y0

    pad_x = int(round(bw * frac_x))
    pad_y = int(round(bh * frac_y))

    x0n = max(0, x0 - pad_x)
    y0n = max(0, y0 - pad_y)
    x1n = min(w - 1, x1 + pad_x)
    y1n = min(h - 1, y1 + pad_y)

    return [x0n, y0n, x1n, y1n]

def evaluar_pred(mask_pred, mask_gt, eps=1e-8):
    inter = np.logical_and(mask_pred == 1, mask_gt == 1).sum()
    union = np.logical_or(mask_pred == 1, mask_gt == 1).sum()
    dice = (2 * inter + eps) / (mask_pred.sum() + mask_gt.sum() + eps)
    iou = (inter + eps) / (union + eps)
    return float(dice), float(iou)

predictor.set_image(imagen)

resultados_bbox_pequena = []

for vertebra_objetivo in VERTEBRAS_PRUEBA:
    prompt_info = prompts_sample[vertebra_objetivo]
    bbox_original = prompt_info["bbox_xyxy"]
    bbox_expandida_pequena = expand_bbox_xyxy_pequena(
        bbox_original,
        imagen.shape,
        frac_x=0.04,
        frac_y=0.06
    )

    mask_gt = construir_mask_binaria_vertebra(mascara, vertebra_objetivo)

    # bbox original
    box_np_orig = np.array(bbox_original, dtype=np.float32)[None, :]
    masks_o, scores_o, _ = predictor.predict(
        box=box_np_orig,
        multimask_output=False
    )
    mask_pred_o = masks_o[0].astype(np.uint8)
    dice_o, iou_o = evaluar_pred(mask_pred_o, mask_gt)

    # bbox ampliada poco
    box_np_exp = np.array(bbox_expandida_pequena, dtype=np.float32)[None, :]
    masks_e, scores_e, _ = predictor.predict(
        box=box_np_exp,
        multimask_output=False
    )
    mask_pred_e = masks_e[0].astype(np.uint8)
    dice_e, iou_e = evaluar_pred(mask_pred_e, mask_gt)

    resultados_bbox_pequena.append({
        "vertebra": vertebra_objetivo,
        "id_real": VERTEBRA_TO_ID[vertebra_objetivo],
        "bbox_original": bbox_original,
        "bbox_expandida_pequena": bbox_expandida_pequena,
        "score_orig": float(scores_o[0]),
        "pix_pred_orig": int(mask_pred_o.sum()),
        "dice_orig": dice_o,
        "iou_orig": iou_o,
        "score_exp_peq": float(scores_e[0]),
        "pix_pred_exp_peq": int(mask_pred_e.sum()),
        "dice_exp_peq": dice_e,
        "iou_exp_peq": iou_e,
    })

import pandas as pd
df_bbox_pequena = pd.DataFrame(resultados_bbox_pequena).sort_values("id_real")
display(df_bbox_pequena)

print("\nPromedio Dice original     :", df_bbox_pequena["dice_orig"].mean())
print("Promedio Dice exp. pequeña :", df_bbox_pequena["dice_exp_peq"].mean())
print("Promedio IoU original      :", df_bbox_pequena["iou_orig"].mean())
print("Promedio IoU exp. pequeña  :", df_bbox_pequena["iou_exp_peq"].mean())


### 10.4 Evaluacion exploratoria por vertebra

Esta evaluacion mira vertebras individuales usando cajas de referencia como prompts. La pregunta tecnica es: si el prompt apunta a la vertebra correcta, que tan bien responde MedSAM?

Tambien se prueba la expansion de bbox para medir sensibilidad al tamano del prompt. Si cambios pequenos del bbox modifican mucho Dice/IoU, el pipeline depende demasiado de una caja perfecta.

Esta seccion sigue siendo exploratoria. La comparacion entre escenarios se hace mas adelante con el split completo de validacion.
        


In [ ]:
# ==========================================
# 24) FUNCIONES DE EVALUACION GLOBAL
# ==========================================

def expand_bbox_xyxy_pequena(bbox, img_shape, frac_x=0.04, frac_y=0.06):
    x0, y0, x1, y1 = bbox
    h, w = img_shape[:2]

    bw = x1 - x0
    bh = y1 - y0

    pad_x = int(round(bw * frac_x))
    pad_y = int(round(bh * frac_y))

    x0n = max(0, x0 - pad_x)
    y0n = max(0, y0 - pad_y)
    x1n = min(w - 1, x1 + pad_x)
    y1n = min(h - 1, y1 + pad_y)

    return [x0n, y0n, x1n, y1n]

def evaluar_pred(mask_pred, mask_gt, eps=1e-8):
    inter = np.logical_and(mask_pred == 1, mask_gt == 1).sum()
    union = np.logical_or(mask_pred == 1, mask_gt == 1).sum()
    dice = (2 * inter + eps) / (mask_pred.sum() + mask_gt.sum() + eps)
    iou = (inter + eps) / (union + eps)
    return float(dice), float(iou)


### 10.5 Reconstruccion semantica T1-L5

MedSAM predice una mascara binaria por vertebra. Para evaluar el problema multiclase, esas predicciones se combinan en una mascara semantica unica con IDs `1..17`.

Este paso define como se manejan solapamientos entre vertebras. La version con score-map es mas informativa porque, cuando dos predicciones compiten por un pixel, conserva la prediccion con mayor confianza.
        


In [ ]:
# ==========================================
# 31) RECONSTRUIR MASCARA SEMANTICA T1-L5
# ==========================================

def reconstruir_mascara_semantica_medsam(
    imagen,
    mascara_gt_multiclase,
    prompts_sample,
    predictor,
    frac_x=0.04,
    frac_y=0.06
):
    """
    Devuelve:
    - mask_sem_pred: mascara semantica predicha (0=fondo, 1..17=T1..L5)
    - mask_sem_gt:   mascara semantica GT remapeada a 0..17
    - detalles:      lista con resultados por vertebra
    """

    predictor.set_image(imagen)

    h, w = imagen.shape[:2]
    mask_sem_pred = np.zeros((h, w), dtype=np.uint8)
    mask_sem_gt = construir_mask_multiclase_t1_l5(mascara_gt_multiclase)

    detalles = []

    for vertebra_objetivo in CLASES_OBJETIVO:
        if vertebra_objetivo not in prompts_sample:
            continue

        # Cada bbox usada aqui proviene del prompt exportado. Si el prompt fue creado desde GT,
        # la metrica evalua segmentacion condicionada por una caja ideal, no deteccion automatica.
        prompt_info = prompts_sample[vertebra_objetivo]
        bbox_original = prompt_info["bbox_xyxy"]
        bbox_expandida = expand_bbox_xyxy_pequena(
            bbox_original,
            imagen.shape,
            frac_x=frac_x,
            frac_y=frac_y
        )

        box_np = np.array(bbox_expandida, dtype=np.float32)[None, :]

        masks, scores, logits = predictor.predict(
            box=box_np,
            multimask_output=False
        )

        mask_pred_bin = masks[0].astype(np.uint8)
        id_local = CLASS_TO_ID[vertebra_objetivo]

        # Asignar id semantico donde la prediccion sea 1
        # Si hay solapamientos, la ultima vertebra escrita sobrescribe.
        # Por ahora está bien como baseline.
        mask_sem_pred[mask_pred_bin == 1] = id_local

        # GT binaria de esa vertebra para metricas individuales
        mask_gt_bin = construir_mask_binaria_vertebra(mascara_gt_multiclase, vertebra_objetivo)

        inter = np.logical_and(mask_pred_bin == 1, mask_gt_bin == 1).sum()
        union = np.logical_or(mask_pred_bin == 1, mask_gt_bin == 1).sum()

        dice_v = (2 * inter + 1e-8) / (mask_pred_bin.sum() + mask_gt_bin.sum() + 1e-8)
        iou_v = (inter + 1e-8) / (union + 1e-8)

        detalles.append({
            "vertebra": vertebra_objetivo,
            "id_local": id_local,
            "id_real": VERTEBRA_TO_ID[vertebra_objetivo],
            "bbox_original": bbox_original,
            "bbox_expandida": bbox_expandida,
            "score_medsam": float(scores[0]),
            "pix_gt": int(mask_gt_bin.sum()),
            "pix_pred": int(mask_pred_bin.sum()),
            "dice": float(dice_v),
            "iou": float(iou_v)
        })

    return mask_sem_pred, mask_sem_gt, detalles


In [ ]:
# ==========================================
# 32) RECONSTRUCCION SEMANTICA EN EL PACIENTE ACTUAL
# ==========================================

mask_sem_pred, mask_sem_gt, detalles_sem = reconstruir_mascara_semantica_medsam(
    imagen=imagen,
    mascara_gt_multiclase=mascara,
    prompts_sample=prompts_sample,
    predictor=predictor,
    frac_x=0.04,
    frac_y=0.06
)

print("Mascara semantica pred shape:", mask_sem_pred.shape)
print("Valores unicos pred:", np.unique(mask_sem_pred))
print("Valores unicos gt  :", np.unique(mask_sem_gt))
print("Cantidad de vertebras procesadas:", len(detalles_sem))


In [ ]:
# ==========================================
# 33) TABLA DE RESULTADOS POR VERTEBRA
# ==========================================

import pandas as pd

df_detalles_sem = pd.DataFrame(detalles_sem).sort_values("id_real").reset_index(drop=True)
display(df_detalles_sem)

print("\nPromedio Dice paciente actual:", df_detalles_sem["dice"].mean())
print("Promedio IoU  paciente actual:", df_detalles_sem["iou"].mean())
print("Promedio score MedSAM       :", df_detalles_sem["score_medsam"].mean())


In [ ]:
# ==========================================
# 34) VISUALIZACION DE MASCARA SEMANTICA COMPLETA
# ==========================================

fig, ax = plt.subplots(1, 3, figsize=(18, 8))

ax[0].imshow(imagen)
ax[0].set_title(f"Imagen - {sample_key}")
ax[0].axis("off")

ax[1].imshow(mask_sem_gt, cmap="nipy_spectral", vmin=0, vmax=N_CLASES)
ax[1].set_title("GT semantica T1-L5")
ax[1].axis("off")

ax[2].imshow(mask_sem_pred, cmap="nipy_spectral", vmin=0, vmax=N_CLASES)
ax[2].set_title("Pred semantica T1-L5")
ax[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 35) OVERLAY GT Y PRED SOBRE LA IMAGEN
# ==========================================

fig, ax = plt.subplots(1, 2, figsize=(16, 10))

# GT overlay
ax[0].imshow(imagen)
ax[0].imshow(mask_sem_gt, cmap="nipy_spectral", alpha=0.35, vmin=0, vmax=N_CLASES)
ax[0].set_title("Overlay GT semantica")
ax[0].axis("off")

# Pred overlay
ax[1].imshow(imagen)
ax[1].imshow(mask_sem_pred, cmap="nipy_spectral", alpha=0.35, vmin=0, vmax=N_CLASES)
ax[1].set_title("Overlay Pred semantica")
ax[1].axis("off")

plt.tight_layout()
plt.show()


### 10.6 Metricas macro por paciente

Las metricas globales por imagen se calculan como promedio macro por vertebra. Esto significa que cada vertebra evaluable pesa igual, independientemente de su area en pixeles.

Este criterio evita que vertebras grandes dominen el promedio. La misma definicion debe mantenerse en todos los escenarios para que las comparaciones sean consistentes.
        


In [ ]:
# ==========================================
# 36) METRICA GLOBAL DE LA MASCARA SEMANTICA
# ==========================================

def dice_multiclase_promedio(mask_pred, mask_gt, clases_ids):
    # Promedio macro: cada vertebra pesa igual, independiente de su area en pixeles.
    dices = []
    for cid in clases_ids:
        pred_bin = (mask_pred == cid).astype(np.uint8)
        gt_bin = (mask_gt == cid).astype(np.uint8)

        if gt_bin.sum() == 0 and pred_bin.sum() == 0:
            continue

        inter = np.logical_and(pred_bin == 1, gt_bin == 1).sum()
        dice = (2 * inter + 1e-8) / (pred_bin.sum() + gt_bin.sum() + 1e-8)
        dices.append(float(dice))
    return np.mean(dices) if len(dices) > 0 else np.nan

def iou_multiclase_promedio(mask_pred, mask_gt, clases_ids):
    ious = []
    for cid in clases_ids:
        pred_bin = (mask_pred == cid).astype(np.uint8)
        gt_bin = (mask_gt == cid).astype(np.uint8)

        if gt_bin.sum() == 0 and pred_bin.sum() == 0:
            continue

        inter = np.logical_and(pred_bin == 1, gt_bin == 1).sum()
        union = np.logical_or(pred_bin == 1, gt_bin == 1).sum()
        iou = (inter + 1e-8) / (union + 1e-8)
        ious.append(float(iou))
    return np.mean(ious) if len(ious) > 0 else np.nan

dice_sem_global = dice_multiclase_promedio(
    mask_sem_pred,
    mask_sem_gt,
    clases_ids=list(range(1, N_CLASES + 1))
)

iou_sem_global = iou_multiclase_promedio(
    mask_sem_pred,
    mask_sem_gt,
    clases_ids=list(range(1, N_CLASES + 1))
)

print("Dice multiclase promedio (imagen):", dice_sem_global)
print("IoU  multiclase promedio (imagen):", iou_sem_global)


In [ ]:
# ==========================================
# 37) VERSION CON MAPA DE SCORE PARA SOLAPAMIENTOS
# ==========================================

def reconstruir_mascara_semantica_medsam_con_score(
    imagen,
    mascara_gt_multiclase,
    prompts_sample,
    predictor,
    frac_x=0.04,
    frac_y=0.06,
    return_quality=False
):
    predictor.set_image(imagen)

    h, w = imagen.shape[:2]
    mask_sem_pred = np.zeros((h, w), dtype=np.uint8)
    score_map = np.zeros((h, w), dtype=np.float32)
    pred_count = np.zeros((h, w), dtype=np.uint8)
    mask_sem_gt = construir_mask_multiclase_t1_l5(mascara_gt_multiclase)

    detalles = []

    for vertebra_objetivo in CLASES_OBJETIVO:
        if vertebra_objetivo not in prompts_sample:
            continue

        # Cada bbox usada aqui proviene del prompt exportado. Si el prompt fue creado desde GT,
        # la metrica evalua segmentacion condicionada por una caja ideal, no deteccion automatica.
        prompt_info = prompts_sample[vertebra_objetivo]
        bbox_original = prompt_info["bbox_xyxy"]
        bbox_expandida = expand_bbox_xyxy_pequena(
            bbox_original,
            imagen.shape,
            frac_x=frac_x,
            frac_y=frac_y
        )

        box_np = np.array(bbox_expandida, dtype=np.float32)[None, :]

        masks, scores, logits = predictor.predict(
            box=box_np,
            multimask_output=False
        )

        mask_pred_bin = masks[0].astype(np.uint8)
        score_pred = float(scores[0])
        pred_count += mask_pred_bin

        id_local = CLASS_TO_ID[vertebra_objetivo]

        # En zonas solapadas se conserva la vertebra con mayor score interno de MedSAM.
        # Este score no es Dice/IoU; solo se usa como regla de desempate entre predicciones.
        update_idx = (mask_pred_bin == 1) & (score_pred > score_map)
        mask_sem_pred[update_idx] = id_local
        score_map[update_idx] = score_pred

        mask_gt_bin = construir_mask_binaria_vertebra(mascara_gt_multiclase, vertebra_objetivo)

        inter = np.logical_and(mask_pred_bin == 1, mask_gt_bin == 1).sum()
        union = np.logical_or(mask_pred_bin == 1, mask_gt_bin == 1).sum()

        dice_v = (2 * inter + 1e-8) / (mask_pred_bin.sum() + mask_gt_bin.sum() + 1e-8)
        iou_v = (inter + 1e-8) / (union + 1e-8)

        detalles.append({
            "vertebra": vertebra_objetivo,
            "id_local": id_local,
            "id_real": VERTEBRA_TO_ID[vertebra_objetivo],
            "bbox_original": bbox_original,
            "bbox_expandida": bbox_expandida,
            "score_medsam": score_pred,
            "pix_gt": int(mask_gt_bin.sum()),
            "pix_pred": int(mask_pred_bin.sum()),
            "dice": float(dice_v),
            "iou": float(iou_v),
            "pred_vacia": bool(mask_pred_bin.sum() == 0),
            "gt_vacia": bool(mask_gt_bin.sum() == 0)
        })

    quality = {
        "overlap_pixels": int((pred_count > 1).sum()),
        "pred_pixels": int((pred_count > 0).sum()),
        "overlap_fraction_pred": float((pred_count > 1).sum() / max((pred_count > 0).sum(), 1)),
        "n_predicciones_vacias": int(sum(d["pred_vacia"] for d in detalles)),
        "n_gt_vacias": int(sum(d["gt_vacia"] for d in detalles))
    }

    if return_quality:
        return mask_sem_pred, mask_sem_gt, detalles, score_map, quality

    return mask_sem_pred, mask_sem_gt, detalles, score_map


In [ ]:
# ==========================================
# 38) RECONSTRUCCION CON SCORE-MAP
# ==========================================

mask_sem_pred_score, mask_sem_gt_score, detalles_sem_score, score_map = reconstruir_mascara_semantica_medsam_con_score(
    imagen=imagen,
    mascara_gt_multiclase=mascara,
    prompts_sample=prompts_sample,
    predictor=predictor,
    frac_x=0.04,
    frac_y=0.06
)

dice_sem_global_score = dice_multiclase_promedio(
    mask_sem_pred_score,
    mask_sem_gt_score,
    clases_ids=list(range(1, N_CLASES + 1))
)

iou_sem_global_score = iou_multiclase_promedio(
    mask_sem_pred_score,
    mask_sem_gt_score,
    clases_ids=list(range(1, N_CLASES + 1))
)

print("Dice multiclase promedio (score-map):", dice_sem_global_score)
print("IoU  multiclase promedio (score-map):", iou_sem_global_score)

fig, ax = plt.subplots(1, 2, figsize=(14, 8))
ax[0].imshow(mask_sem_gt_score, cmap="nipy_spectral", vmin=0, vmax=N_CLASES)
ax[0].set_title("GT semantica")
ax[0].axis("off")

ax[1].imshow(mask_sem_pred_score, cmap="nipy_spectral", vmin=0, vmax=N_CLASES)
ax[1].set_title("Pred semantica (score-map)")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 10.X) CREAR DATASETS Y DATALOADERS BASE
# ==========================================

from torch.utils.data import Dataset, DataLoader

class MedSAMVertebraDataset(Dataset):
    def __init__(self, split, prompts_filtrados, frac_x=0.04, frac_y=0.06):
        self.split = split
        self.prompts_filtrados = prompts_filtrados
        self.frac_x = frac_x
        self.frac_y = frac_y
        self.samples = []

        for patient_id, prompts_sample in self.prompts_filtrados[self.split].items():
            for vertebra_objetivo in CLASES_OBJETIVO:
                if vertebra_objetivo not in prompts_sample:
                    continue

                self.samples.append({
                    "patient_id": patient_id,
                    "vertebra": vertebra_objetivo,
                    "bbox": prompts_sample[vertebra_objetivo]["bbox_xyxy"]
                })

        print(f"[{self.split}] muestras por vertebra:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        patient_id = item["patient_id"]
        vertebra_objetivo = item["vertebra"]
        bbox_original = item["bbox"]

        path_imagen, path_mascara = resolver_paths_muestra(self.split, patient_id)

        imagen = cargar_imagen(path_imagen)
        mascara = cargar_mascara(path_mascara)

        bbox_expandida = expand_bbox_xyxy_pequena(
            bbox_original,
            imagen.shape,
            frac_x=self.frac_x,
            frac_y=self.frac_y
        )

        mask_gt_bin = construir_mask_binaria_vertebra(
            mascara,
            vertebra_objetivo
        ).astype(np.float32)

        return {
            "patient_id": patient_id,
            "vertebra": vertebra_objetivo,
            "image": imagen.astype(np.uint8),
            "box": torch.tensor(bbox_expandida, dtype=torch.float32),
            "gt_mask": torch.tensor(mask_gt_bin[None, :, :], dtype=torch.float32)
        }


def collate_fn_medsam(batch):
    return batch


batch_size = 2
workers = 0

train_dataset = MedSAMVertebraDataset(
    split="train",
    prompts_filtrados=PROMPTS_FILTRADOS,
    frac_x=0.04,
    frac_y=0.06
)

val_dataset = MedSAMVertebraDataset(
    split="val",
    prompts_filtrados=PROMPTS_FILTRADOS,
    frac_x=0.04,
    frac_y=0.06
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=workers,
    collate_fn=collate_fn_medsam
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=workers,
    collate_fn=collate_fn_medsam
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

## 11. Escenario 1: decoder_basico

Tipo: cambio 1, escenario base entrenado.

Hipótesis: ajustar únicamente el mask_decoder puede permitir que MedSAM adapte la generación de máscaras al dominio específico de radiografías AP de columna, sin modificar la representación visual general ya aprendida por el image_encoder.

Qué cambia: se entrena exclusivamente el decodificador de máscaras.

Qué se mantiene fijo: image_encoder congelado, prompt_encoder congelado, splits train/val/test, preprocesamiento RGB a 1024x1024, prompts bbox_gt con expansión pequeña y esquema de métricas Dice/IoU.

Nota metodológica: este escenario no evalúa detección automática de vértebras. Las cajas de entrada provienen de bbox_gt derivadas de la anotación real, por lo que el objetivo aquí es medir la capacidad de MedSAM para refinar la segmentación cuando recibe una localización correcta de la estructura anatómica.

<!-- codex-explicacion -->
Primer escenario MedSAM entrenable. Sirve para medir cuanto se gana ajustando el decoder sin tocar demasiadas partes del modelo.


### 11.1 Entrenamiento del escenario decoder_basico

En esta sección se realiza el ajuste fino del escenario base. Para ello, se congela toda la arquitectura de MedSAM y se habilita únicamente el entrenamiento del mask_decoder, con el fin de modificar solo la etapa encargada de transformar los embeddings visuales y de prompt en una máscara binaria final. Posteriormente se definen la función de pérdida, el optimizador y el scheduler, y se ejecuta el entrenamiento sobre todas las instancias vertebrales disponibles en train, validando en cada época sobre el conjunto completo de validation. El mejor checkpoint se selecciona según la menor pérdida de validación obtenida.

In [ ]:
# ==========================================
# 11.1A) CONFIGURACION DECODER BASICO
# ==========================================

for p in medsam_model.parameters():
    p.requires_grad = False

for p in medsam_model.mask_decoder.parameters():
    p.requires_grad = True

n_total = sum(p.numel() for p in medsam_model.parameters())
n_trainable = sum(p.numel() for p in medsam_model.parameters() if p.requires_grad)

print("Parametros totales     :", n_total)
print("Parametros entrenables :", n_trainable)
print("Porcentaje entrenable  :", 100 * n_trainable / n_total)

train_loader_decoder_basic = train_loader
val_loader_decoder_basic = val_loader

print("Train batches:", len(train_loader_decoder_basic))
print("Val batches  :", len(val_loader_decoder_basic))

In [ ]:
# ==========================================
# 11.1B) LOSS, OPTIMIZER Y SCHEDULER
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from segment_anything import SamPredictor

bce_loss = nn.BCEWithLogitsLoss()

def dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.reshape(probs.shape[0], -1)
    targets = targets.reshape(targets.shape[0], -1)

    inter = (probs * targets).sum(dim=1)
    denom = probs.sum(dim=1) + targets.sum(dim=1)

    dice = (2 * inter + eps) / (denom + eps)
    return 1 - dice.mean()

optimizer_decoder_basic = optim.AdamW(
    medsam_model.mask_decoder.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler_decoder_basic = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_decoder_basic,
    mode="min",
    factor=0.5,
    patience=2
)

print("Optimizer decoder_basic listo.")

In [ ]:
# ==========================================
# 11.1C) FUNCION DE EPOCA DECODER BASICO
# ==========================================

def run_epoch_medsam_decoder_basic(loader, train=True, epoch=None):
    if train:
        medsam_model.train()
        desc = f"Train Decoder Epoch {epoch}"
    else:
        medsam_model.eval()
        desc = f"Val Decoder Epoch {epoch}"

    total_loss = 0.0
    total_bce = 0.0
    total_dice = 0.0
    n_samples = 0

    pbar = tqdm(loader, desc=desc, leave=True)

    for batch in pbar:
        for sample in batch:
            image_np = sample["image"]
            box = sample["box"].to(device)
            gt_mask = sample["gt_mask"].to(device)

            if train:
                optimizer_decoder_basic.zero_grad()

            with torch.set_grad_enabled(train):
                pred_logits, _ = forward_medsam_with_box(image_np, box)

                gt_mask_batch = gt_mask.unsqueeze(0) if gt_mask.ndim == 3 else gt_mask

                loss_bce = bce_loss(pred_logits, gt_mask_batch)
                loss_dice = dice_loss_from_logits(pred_logits, gt_mask_batch)
                loss = loss_bce + loss_dice

                if train:
                    loss.backward()
                    optimizer_decoder_basic.step()

            total_loss += loss.item()
            total_bce += loss_bce.item()
            total_dice += loss_dice.item()
            n_samples += 1

        pbar.set_postfix({
            "loss": f"{total_loss / max(n_samples, 1):.4f}",
            "bce": f"{total_bce / max(n_samples, 1):.4f}",
            "dice_loss": f"{total_dice / max(n_samples, 1):.4f}",
        })

    return {
        "loss": total_loss / max(n_samples, 1),
        "bce": total_bce / max(n_samples, 1),
        "dice_loss": total_dice / max(n_samples, 1)
    }

In [ ]:
# ============================================================
# 11.1C-bis) Forward MedSAM con caja
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np


def preparar_imagen_medsam_tensor(image_np, device):
    """
    Convierte una imagen numpy RGB 1024x1024 a tensor BCHW para MedSAM.

    Espera:
    - image_np: numpy array H x W x 3
    - valores en 0-255 o 0-1
    """

    if isinstance(image_np, torch.Tensor):
        image_t = image_np.float()

        if image_t.ndim == 3:
            # Si viene HWC
            if image_t.shape[-1] == 3:
                image_t = image_t.permute(2, 0, 1)
            image_t = image_t.unsqueeze(0)

        elif image_t.ndim == 4:
            pass

        else:
            raise ValueError(f"Forma de imagen no soportada: {image_t.shape}")

        image_t = image_t.to(device)

    else:
        image_t = torch.as_tensor(image_np, dtype=torch.float32, device=device)

        if image_t.ndim == 3:
            image_t = image_t.permute(2, 0, 1).unsqueeze(0)
        elif image_t.ndim != 4:
            raise ValueError(f"Forma de imagen no soportada: {image_t.shape}")

    # Si está en 0-1, pasar a 0-255 para usar preprocess de SAM.
    if image_t.max() <= 1.0:
        image_t = image_t * 255.0

    return image_t


def preparar_box_medsam_tensor(box, device):
    """
    Convierte la caja a tensor con forma [B, 4].
    """

    if isinstance(box, torch.Tensor):
        box_t = box.float().to(device)
    else:
        box_t = torch.as_tensor(box, dtype=torch.float32, device=device)

    if box_t.ndim == 1:
        box_t = box_t.unsqueeze(0)

    if box_t.shape[-1] != 4:
        raise ValueError(f"La caja debe tener forma [4] o [B,4], pero llegó {box_t.shape}")

    return box_t


def forward_medsam_with_box(image_np, box):
    """
    Forward de MedSAM usando una caja como prompt.

    Retorna:
    - pred_logits: máscara logit interpolada a 1024x1024, forma [1, 1, 1024, 1024]
    - iou_pred: predicción de calidad del mask_decoder
    """

    image_t = preparar_imagen_medsam_tensor(image_np, device)
    box_t = preparar_box_medsam_tensor(box, device)

    # Encoder visual congelado o entrenable según cómo esté configurado el modelo.
    image_embedding = medsam_model.image_encoder(image_t)

    # Prompt encoder congelado normalmente.
    with torch.no_grad():
        sparse_embeddings, dense_embeddings = medsam_model.prompt_encoder(
            points=None,
            boxes=box_t,
            masks=None
        )

    low_res_logits, iou_pred = medsam_model.mask_decoder(
        image_embeddings=image_embedding,
        image_pe=medsam_model.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=False
    )

    pred_logits = F.interpolate(
        low_res_logits,
        size=(1024, 1024),
        mode="bilinear",
        align_corners=False
    )

    return pred_logits, iou_pred

In [ ]:
# ============================================================
# Prueba rápida de forward
# ============================================================

sample_test = train_loader_decoder_basic.dataset[0]

pred_logits_test, iou_pred_test = forward_medsam_with_box(
    sample_test["image"],
    sample_test["box"].to(device)
)

print("pred_logits_test:", pred_logits_test.shape)
print("iou_pred_test:", iou_pred_test.shape)
print("gt_mask:", sample_test["gt_mask"].shape)

In [ ]:
# ==========================================
# 11.1D) ENTRENAMIENTO DECODER BASICO
# ==========================================

num_epochs_decoder_basic = 8
patience_decoder_basic = 3

best_val_loss_decoder_basic = float("inf")
best_state_dict_decoder_basic = None
patience_counter_decoder_basic = 0

history_decoder_basic = []

for epoch in range(1, num_epochs_decoder_basic + 1):

    train_metrics = run_epoch_medsam_decoder_basic(
        train_loader_decoder_basic,
        train=True,
        epoch=epoch
    )

    val_metrics = run_epoch_medsam_decoder_basic(
        val_loader_decoder_basic,
        train=False,
        epoch=epoch
    )

    scheduler_decoder_basic.step(val_metrics["loss"])

    history_decoder_basic.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_bce": train_metrics["bce"],
        "train_dice_loss": train_metrics["dice_loss"],
        "val_loss": val_metrics["loss"],
        "val_bce": val_metrics["bce"],
        "val_dice_loss": val_metrics["dice_loss"],
        "lr": optimizer_decoder_basic.param_groups[0]["lr"]
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"lr={optimizer_decoder_basic.param_groups[0]['lr']:.2e}"
    )

    if val_metrics["loss"] < best_val_loss_decoder_basic:
        best_val_loss_decoder_basic = val_metrics["loss"]
        best_state_dict_decoder_basic = {
            k: v.cpu().clone()
            for k, v in medsam_model.state_dict().items()
        }
        patience_counter_decoder_basic = 0
        print("  -> nuevo mejor decoder_basico")
    else:
        patience_counter_decoder_basic += 1
        print(f"  -> sin mejora ({patience_counter_decoder_basic}/{patience_decoder_basic})")

    if patience_counter_decoder_basic >= patience_decoder_basic:
        print("Early stopping activado")
        break

df_history_decoder_basic = pd.DataFrame(history_decoder_basic)
display(df_history_decoder_basic)

### 11.2 Evaluación completa sobre todo el conjunto de validación

Una vez restaurado el mejor modelo del escenario decoder_basico, se realiza una evaluación exhaustiva sobre todos los pacientes del split de validación. A diferencia de pruebas exploratorias con subconjuntos pequeños, aquí se recorre paciente por paciente para reconstruir la máscara semántica multiclase T1–L5 y calcular métricas globales y detalladas. Esta evaluación constituye el resultado principal del experimento, ya que permite cuantificar el rendimiento promedio por paciente, comparar el comportamiento entre normales y escoliosis, e identificar qué vértebras presentan mayor dificultad de segmentación.

In [ ]:
# ==========================================
# 11.2A) EVALUACION COMPLETA EN TODO VAL
# ==========================================

split_eval_full = "val"
pacientes_val_full = sorted(list(PROMPTS_FILTRADOS[split_eval_full].keys()))

print("Total pacientes en val:", len(pacientes_val_full))

resultados_val_full_decoder = []
detalles_val_full_decoder = []

for patient_id in tqdm(pacientes_val_full, desc="Evaluando todo VAL"):

    path_img, path_mask = resolver_paths_muestra(split_eval_full, patient_id)

    imagen_eval = cargar_imagen(path_img)
    mascara_eval = cargar_mascara(path_mask)
    prompts_eval = PROMPTS_FILTRADOS[split_eval_full][patient_id]

    mask_sem_pred, mask_sem_gt, detalles, score_map = reconstruir_mascara_semantica_medsam_con_score(
        imagen=imagen_eval,
        mascara_gt_multiclase=mascara_eval,
        prompts_sample=prompts_eval,
        predictor=predictor,
        frac_x=0.04,
        frac_y=0.06
    )

    clases_presentes = sorted([
        d["id_real"] for d in detalles
        if "id_real" in d
    ])

    if len(clases_presentes) == 0:
        continue

    dice_global = dice_multiclase_promedio(
        mask_sem_pred,
        mask_sem_gt,
        clases_ids=clases_presentes
    )

    iou_global = iou_multiclase_promedio(
        mask_sem_pred,
        mask_sem_gt,
        clases_ids=clases_presentes
    )

    grupo = (
        "normal" if str(patient_id).startswith("N_")
        else "scoliosis" if str(patient_id).startswith("S_")
        else "otro"
    )

    resultados_val_full_decoder.append({
        "patient_id": patient_id,
        "grupo": grupo,
        "n_vertebras": len(clases_presentes),
        "dice_global": float(dice_global),
        "iou_global": float(iou_global)
    })

    for d in detalles:
        detalles_val_full_decoder.append({
            "patient_id": patient_id,
            "grupo": grupo,
            "vertebra": d.get("vertebra"),
            "id_real": d.get("id_real"),
            "score_medsam": d.get("score_medsam", np.nan),
            "dice": d.get("dice", np.nan),
            "iou": d.get("iou", np.nan)
        })

df_val_full_decoder = pd.DataFrame(resultados_val_full_decoder)
df_detalle_val_full_decoder = pd.DataFrame(detalles_val_full_decoder)

display(df_val_full_decoder)
display(df_detalle_val_full_decoder)

print("Pacientes evaluados:", len(df_val_full_decoder))
print("Instancias vertebrales evaluadas:", len(df_detalle_val_full_decoder))


In [ ]:
# ==========================================
# 11.2B) RESUMEN GLOBAL TODO VAL
# ==========================================

print("===== DECODER_BASICO - TODO VAL =====")

print("Pacientes evaluados:", len(df_val_full_decoder))
print("Vertebras evaluadas:", len(df_detalle_val_full_decoder))

print("\nPromedio por paciente:")
print("Dice global:", df_val_full_decoder["dice_global"].mean())
print("IoU global :", df_val_full_decoder["iou_global"].mean())

print("\nPromedio por instancia vertebral:")
print("Dice:", df_detalle_val_full_decoder["dice"].mean())
print("IoU :", df_detalle_val_full_decoder["iou"].mean())

In [ ]:
# ==========================================
# 11.2C) RESUMEN POR GRUPO
# ==========================================

df_resumen_grupo_val_full = (
    df_val_full_decoder
    .groupby("grupo", as_index=False)
    .agg(
        pacientes=("patient_id", "count"),
        vertebras_promedio=("n_vertebras", "mean"),
        dice_mean=("dice_global", "mean"),
        dice_std=("dice_global", "std"),
        iou_mean=("iou_global", "mean"),
        iou_std=("iou_global", "std")
    )
)

display(df_resumen_grupo_val_full)

if set(["normal", "scoliosis"]).issubset(set(df_resumen_grupo_val_full["grupo"])):
    dice_normal = df_resumen_grupo_val_full.loc[
        df_resumen_grupo_val_full["grupo"] == "normal", "dice_mean"
    ].iloc[0]

    dice_scol = df_resumen_grupo_val_full.loc[
        df_resumen_grupo_val_full["grupo"] == "scoliosis", "dice_mean"
    ].iloc[0]

    iou_normal = df_resumen_grupo_val_full.loc[
        df_resumen_grupo_val_full["grupo"] == "normal", "iou_mean"
    ].iloc[0]

    iou_scol = df_resumen_grupo_val_full.loc[
        df_resumen_grupo_val_full["grupo"] == "scoliosis", "iou_mean"
    ].iloc[0]

    print("Delta Dice normal - scoliosis:", dice_normal - dice_scol)
    print("Delta IoU  normal - scoliosis:", iou_normal - iou_scol)


In [ ]:
# ==========================================
# 11.2D) RESUMEN POR VERTEBRA
# ==========================================

df_resumen_vertebra_val_full = (
    df_detalle_val_full_decoder
    .groupby(["vertebra", "id_real"], as_index=False)
    .agg(
        n=("dice", "count"),
        dice_mean=("dice", "mean"),
        dice_std=("dice", "std"),
        iou_mean=("iou", "mean"),
        iou_std=("iou", "std"),
        score_mean=("score_medsam", "mean")
    )
    .sort_values("id_real")
)

display(df_resumen_vertebra_val_full)


In [ ]:
# ==========================================
# 11.2E) RESUMEN POR GRUPO Y VERTEBRA
# ==========================================

df_resumen_vertebra_grupo_val_full = (
    df_detalle_val_full_decoder
    .groupby(["grupo", "vertebra", "id_real"], as_index=False)
    .agg(
        n=("dice", "count"),
        dice_mean=("dice", "mean"),
        dice_std=("dice", "std"),
        iou_mean=("iou", "mean"),
        iou_std=("iou", "std"),
        score_mean=("score_medsam", "mean")
    )
    .sort_values(["grupo", "id_real"])
)

display(df_resumen_vertebra_grupo_val_full)

In [ ]:
# ==========================================
# 11.2F) PEORES CASOS
# ==========================================

print("Peores pacientes por Dice:")
display(
    df_val_full_decoder
    .sort_values("dice_global")
    .head(10)
)

print("Peores vertebras individuales por Dice:")
display(
    df_detalle_val_full_decoder
    .sort_values("dice")
    .head(20)
)



### 11.3 Visualización de un caso representativo

Finalmente, se presenta una única visualización cualitativa correspondiente a un caso representativo del conjunto de validación. Para evitar sesgos visuales asociados a mostrar únicamente el mejor o el peor resultado, se selecciona una radiografía cuyo Dice global se encuentre cercano a la mediana del conjunto. Esta visualización permite contrastar de forma intuitiva la máscara semántica real frente a la máscara predicha por el modelo y complementar la interpretación numérica obtenida en las tablas anteriores.

In [ ]:
# ==========================================
# 11.3) VISUALIZACION FINAL DE CASO REPRESENTATIVO
# ==========================================

df_tmp_visual = df_val_full_decoder.copy()
dice_mediana = df_tmp_visual["dice_global"].median()

df_tmp_visual["dist_mediana"] = (
    df_tmp_visual["dice_global"] - dice_mediana
).abs()

patient_id_visual = (
    df_tmp_visual
    .sort_values("dist_mediana")
    .iloc[0]["patient_id"]
)

path_img, path_mask = resolver_paths_muestra("val", patient_id_visual)

imagen_visual = cargar_imagen(path_img)
mascara_visual = cargar_mascara(path_mask)
prompts_visual = PROMPTS_FILTRADOS["val"][patient_id_visual]

mask_sem_pred_visual, mask_sem_gt_visual, detalles_visual, score_map_visual = reconstruir_mascara_semantica_medsam_con_score(
    imagen=imagen_visual,
    mascara_gt_multiclase=mascara_visual,
    prompts_sample=prompts_visual,
    predictor=predictor,
    frac_x=0.04,
    frac_y=0.06
)

clases_visual = [d["id_real"] for d in detalles_visual]

dice_visual = dice_multiclase_promedio(
    mask_sem_pred_visual,
    mask_sem_gt_visual,
    clases_ids=clases_visual
)

iou_visual = iou_multiclase_promedio(
    mask_sem_pred_visual,
    mask_sem_gt_visual,
    clases_ids=clases_visual
)

fig, ax = plt.subplots(1, 3, figsize=(18, 8))

ax[0].imshow(imagen_visual)
ax[0].set_title(f"Imagen - {patient_id_visual}")
ax[0].axis("off")

ax[1].imshow(mask_sem_gt_visual, cmap="nipy_spectral")
ax[1].set_title("GT semantica")
ax[1].axis("off")

ax[2].imshow(mask_sem_pred_visual, cmap="nipy_spectral")
ax[2].set_title(
    f"Pred semantica\nDice={dice_visual:.4f} | IoU={iou_visual:.4f}"
)
ax[2].axis("off")

plt.tight_layout()
plt.show()

print("Paciente visualizado:", patient_id_visual)
print("Dice mediana del conjunto:", dice_mediana)
print("Dice del caso visualizado:", dice_visual)

## 12. Escenario 2: prompts automáticos por región de columna
Tipo: cambio 2, modificación de prompts.

Hipótesis: si se reemplazan las cajas ideales derivadas de las máscaras por cajas aproximadas generadas desde la imagen, el pipeline se acerca más a una segmentación automática real. En este escenario se mantiene el mismo modelo entrenado en el experimento 11, pero se evalúa qué tanto depende el rendimiento de tener una localización perfecta de cada vértebra.

Qué cambia: las cajas bbox_gt se reemplazan por cajas automáticas estimadas a partir de la imagen. Primero se detecta una línea o corredor central de la columna usando contraste e intensidad; luego se genera una caja aproximada para cada vértebra T1–L5.

Qué se mantiene fijo: modelo MedSAM ajustado en el experimento 11, mask_decoder, preprocesamiento 1024×1024, clases T1–L5, métricas Dice/IoU y reconstrucción semántica con score-map.

Nota metodológica: las máscaras reales no se usan para generar las cajas automáticas. Solo se usan después para medir cobertura de caja, Dice e IoU. Esto permite separar dos errores: error de localización anatómica y error de segmentación de MedSAM.


In [ ]:
# ============================================================
# 12.1) Configuración inicial y normalización de prompts
# ============================================================

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

PROMPTS_BASE = PROMPTS_FILTRADOS if "PROMPTS_FILTRADOS" in globals() else PROMPTS

if "VERTEBRA_TO_ID" not in globals():
    VERTEBRA_TO_ID = {v: i + 1 for i, v in enumerate(CLASES_OBJETIVO)}

ID_TO_VERTEBRA = {v: k for k, v in VERTEBRA_TO_ID.items()}

if "N_CLASES" not in globals():
    N_CLASES = len(CLASES_OBJETIVO)


def normalizar_prompts_split(prompts_split):
    """
    Convierte los prompts a formato:
    {patient_id: prompts_sample}
    """

    if isinstance(prompts_split, dict):
        return prompts_split

    if isinstance(prompts_split, list):
        return {
            item["patient_id"]: item["prompts"]
            for item in prompts_split
        }

    raise TypeError("Formato de prompts no reconocido.")


PROMPTS_DICC = {
    split: normalizar_prompts_split(PROMPTS_BASE[split])
    for split in PROMPTS_BASE.keys()
}

print("Splits disponibles:", PROMPTS_DICC.keys())
print("Ejemplo val:", list(PROMPTS_DICC["val"].keys())[:3])

In [ ]:
# ============================================================
# 12.2) Plantilla anatómica desde train
# ============================================================

def obtener_vertebra_desde_prompt(clave_prompt, info_prompt):
    """
    Obtiene el nombre de la vértebra desde el prompt.
    """

    if "vertebra" in info_prompt:
        return info_prompt["vertebra"]

    try:
        clase_id = int(clave_prompt)
        return ID_TO_VERTEBRA[clase_id]
    except Exception:
        return clave_prompt


def construir_template_bbox_train(prompts_dicc, split_template="train"):
    """
    Construye una plantilla promedio de posición y tamaño por vértebra.

    Se usa solo train. No usa val/test para construir la plantilla.
    """

    filas = []

    for patient_id, prompts_sample in prompts_dicc[split_template].items():

        for clave_prompt, info_prompt in prompts_sample.items():

            vertebra = obtener_vertebra_desde_prompt(clave_prompt, info_prompt)

            if vertebra not in CLASES_OBJETIVO:
                continue

            x0, y0, x1, y1 = info_prompt["bbox_xyxy"]

            filas.append({
                "patient_id": patient_id,
                "vertebra": vertebra,
                "id_real": VERTEBRA_TO_ID[vertebra],
                "cx_rel": ((x0 + x1) / 2) / 1024,
                "cy_rel": ((y0 + y1) / 2) / 1024,
                "w_rel": (x1 - x0) / 1024,
                "h_rel": (y1 - y0) / 1024
            })

    df_template = pd.DataFrame(filas)

    template_bbox = (
        df_template
        .groupby(["vertebra", "id_real"], as_index=False)
        .agg(
            cx_rel=("cx_rel", "median"),
            cy_rel=("cy_rel", "median"),
            w_rel=("w_rel", "median"),
            h_rel=("h_rel", "median")
        )
        .sort_values("id_real")
        .reset_index(drop=True)
    )

    faltantes = [v for v in CLASES_OBJETIVO if v not in template_bbox["vertebra"].tolist()]

    if len(faltantes) > 0:
        print("Advertencia: faltan vértebras en la plantilla:", faltantes)

    return template_bbox, df_template


template_bbox_auto, df_template_bbox_train = construir_template_bbox_train(
    PROMPTS_DICC,
    split_template="train"
)

display(template_bbox_auto)

In [ ]:
# ============================================================
# 12.3) Utilidades para imagen, suavizado y ROI
# ============================================================

def imagen_a_gris_uint8(img_rgb):
    """
    Convierte imagen RGB o gris a uint8.
    """

    if img_rgb.ndim == 3:
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_rgb.copy()

    if gray.max() <= 1.0:
        gray = gray * 255

    return gray.astype(np.uint8)


def normalizar_01(x, eps=1e-8):
    x = x.astype(np.float32)
    return (x - x.min()) / (x.max() - x.min() + eps)


def suavizar_1d(x, kernel_size=21):
    kernel_size = int(kernel_size)

    if kernel_size % 2 == 0:
        kernel_size += 1

    kernel = np.ones(kernel_size, dtype=np.float32) / kernel_size
    return np.convolve(x, kernel, mode="same")


def detectar_roi_radiografia(img_rgb, margen_x=35, margen_y=5):
    """
    Detecta el área no negra de la radiografía.
    Sirve para no buscar bordes en el fondo negro.
    """

    gray = imagen_a_gris_uint8(img_rgb)
    H, W = gray.shape

    valores = gray[gray > 0]

    if len(valores) == 0:
        return [0, 0, W - 1, H - 1]

    thr = max(5, np.percentile(valores, 3))
    mask = (gray > thr).astype(np.uint8)

    kernel = np.ones((21, 21), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

    if num_labels <= 1:
        return [0, 0, W - 1, H - 1]

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])

    x, y, w, h, area = stats[largest]

    if w < W * 0.10 or h < H * 0.20:
        return [0, 0, W - 1, H - 1]

    x0 = max(0, x - margen_x)
    y0 = max(0, y - margen_y)
    x1 = min(W - 1, x + w + margen_x)
    y1 = min(H - 1, y + h + margen_y)

    return [x0, y0, x1, y1]


def seleccionar_picos_1d(profile, xs_abs, n_picos=8, min_sep=18):
    """
    Selecciona picos separados en un perfil 1D.
    """

    if len(profile) == 0:
        return np.array([]), np.array([])

    order = np.argsort(profile)[::-1]

    picos_x = []
    picos_s = []

    for idx in order:
        x = float(xs_abs[idx])
        s = float(profile[idx])

        if all(abs(x - px) >= min_sep for px in picos_x):
            picos_x.append(x)
            picos_s.append(s)

        if len(picos_x) >= n_picos:
            break

    return np.array(picos_x, dtype=np.float32), np.array(picos_s, dtype=np.float32)

In [ ]:
# ============================================================
# 12.4) Estimar eje curvo desde bordes laterales
# ============================================================

def estimar_eje_curvo_por_bordes(
    img_rgb,
    template_bbox=None,
    n_franjas=27,
    search_half_frac=0.23,
    n_picos=8,
    min_sep_frac=0.018,
    poly_degree=3,
    debug=False
):
    """
    Estima un eje curvo de columna a partir de bordes laterales.

    Lógica:
    1. Detecta ROI de radiografía.
    2. Divide la imagen en franjas horizontales.
    3. En cada franja busca candidatos de borde izquierdo y derecho.
    4. Selecciona el par de bordes más razonable.
    5. Calcula centro = (borde_izquierdo + borde_derecho) / 2.
    6. Suaviza los centros con un polinomio.
    """

    gray = imagen_a_gris_uint8(img_rgb)
    H, W = gray.shape

    x_roi0, y_roi0, x_roi1, y_roi1 = detectar_roi_radiografia(img_rgb)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)

    grad_x = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 1, 0, ksize=3))
    grad_y = np.abs(cv2.Sobel(gray_eq, cv2.CV_32F, 0, 1, ksize=3))

    score_img = (
        0.70 * normalizar_01(grad_x) +
        0.20 * normalizar_01(gray_eq) +
        0.10 * normalizar_01(grad_y)
    )

    if template_bbox is not None:
        x_prior_inicial = float(np.median(template_bbox["cx_rel"].to_numpy(dtype=float) * W))
        ancho_esperado = float(np.median(template_bbox["w_rel"].to_numpy(dtype=float) * W))
    else:
        x_prior_inicial = (x_roi0 + x_roi1) / 2
        ancho_esperado = W * 0.10

    ancho_esperado = np.clip(ancho_esperado * 1.15, W * 0.06, W * 0.18)

    min_width = max(W * 0.045, ancho_esperado * 0.55)
    max_width = min(W * 0.28, ancho_esperado * 2.60)

    search_half = int(W * search_half_frac)
    min_sep = int(W * min_sep_frac)

    y_edges = np.linspace(0, H, n_franjas + 1).astype(int)

    puntos_centro = []
    puntos_izq = []
    puntos_der = []
    debug_rows = []

    x_prior = x_prior_inicial

    for i in range(n_franjas):

        y0 = int(y_edges[i])
        y1 = int(y_edges[i + 1])
        y_mid = int((y0 + y1) / 2)

        x0 = max(x_roi0, int(x_prior - search_half))
        x1 = min(x_roi1, int(x_prior + search_half))

        if x1 <= x0 + 10:
            x0 = max(0, int(x_prior - search_half))
            x1 = min(W - 1, int(x_prior + search_half))

        xs_abs = np.arange(x0, x1 + 1)

        crop_score = score_img[y0:y1, x0:x1 + 1]

        if crop_score.size == 0:
            continue

        profile = crop_score.mean(axis=0)
        profile = suavizar_1d(profile, kernel_size=max(15, int(W * 0.025)))
        profile = normalizar_01(profile)

        left_mask = xs_abs < (x_prior - min_width * 0.20)
        right_mask = xs_abs > (x_prior + min_width * 0.20)

        xs_left = xs_abs[left_mask]
        prof_left = profile[left_mask]

        xs_right = xs_abs[right_mask]
        prof_right = profile[right_mask]

        left_peaks_x, left_peaks_s = seleccionar_picos_1d(
            prof_left,
            xs_left,
            n_picos=n_picos,
            min_sep=min_sep
        )

        right_peaks_x, right_peaks_s = seleccionar_picos_1d(
            prof_right,
            xs_right,
            n_picos=n_picos,
            min_sep=min_sep
        )

        mejor = None
        mejor_score = -np.inf

        for xl, sl in zip(left_peaks_x, left_peaks_s):
            for xr, sr in zip(right_peaks_x, right_peaks_s):

                if xr <= xl:
                    continue

                width = xr - xl

                if width < min_width or width > max_width:
                    continue

                centro = (xl + xr) / 2

                penal_width = ((width - ancho_esperado) / (ancho_esperado + 1e-6)) ** 2
                penal_prior = ((centro - x_prior) / W) ** 2

                score = sl + sr - 0.75 * penal_width - 1.50 * penal_prior

                if score > mejor_score:
                    mejor_score = score
                    mejor = (xl, xr, centro, width, sl, sr)

        if mejor is None:
            xl = x_prior - ancho_esperado / 2
            xr = x_prior + ancho_esperado / 2
            centro = x_prior
            width = ancho_esperado
            sl = np.nan
            sr = np.nan
        else:
            xl, xr, centro, width, sl, sr = mejor

        puntos_izq.append([y_mid, xl])
        puntos_der.append([y_mid, xr])
        puntos_centro.append([y_mid, centro])

        debug_rows.append({
            "franja": i,
            "y_mid": y_mid,
            "x_left": xl,
            "x_right": xr,
            "x_center": centro,
            "width": width,
            "x_prior": x_prior,
            "score_left": sl,
            "score_right": sr,
            "score_pair": mejor_score
        })

        # Actualización suave del prior para permitir escoliosis sin saltos bruscos.
        x_prior = 0.65 * x_prior + 0.35 * centro

    puntos_centro = np.array(puntos_centro, dtype=np.float32)
    puntos_izq = np.array(puntos_izq, dtype=np.float32)
    puntos_der = np.array(puntos_der, dtype=np.float32)

    if len(puntos_centro) < 4:
        raise ValueError("No se pudieron estimar suficientes puntos para el eje curvo.")

    y_raw = puntos_centro[:, 0]
    x_raw = puntos_centro[:, 1]

    deg = min(poly_degree, len(y_raw) - 1)
    coef = np.polyfit(y_raw, x_raw, deg=deg)
    polinomio = np.poly1d(coef)

    y_smooth = np.linspace(0, H - 1, 300)
    x_smooth = np.clip(polinomio(y_smooth), 0, W - 1)

    puntos_smooth = np.column_stack([y_smooth, x_smooth])

    info = {
        "puntos_centro_raw": puntos_centro,
        "puntos_izq": puntos_izq,
        "puntos_der": puntos_der,
        "puntos_smooth": puntos_smooth,
        "polinomio": polinomio,
        "roi": [x_roi0, y_roi0, x_roi1, y_roi1],
        "score_img": score_img,
        "debug": pd.DataFrame(debug_rows)
    }

    return info

In [ ]:
# ============================================================
# 12.5) Generar prompts automáticos T1-L5
# versión corregida con calibración vertical no lineal
# ============================================================

def generar_prompts_auto_por_bordes(
    img_rgb,
    template_bbox,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    min_w_frac=0.10,
    min_h_frac=0.035,
    max_w_frac=0.34,
    max_h_frac=0.13,
    debug=False
):
    """
    Genera cajas automáticas para las 17 vértebras.

    Correcciones incluidas:
    - El eje horizontal se estima desde bordes laterales.
    - Las claves son 'T1', 'T2', ..., 'L5'.
    - Se aplica una compresión vertical de la plantilla.
    - Se aplica una corrección vertical no lineal para subir más la zona media.
    - Se aplica un pequeño desplazamiento horizontal hacia la izquierda.
    """

    H, W = img_rgb.shape[:2]

    info_eje = estimar_eje_curvo_por_bordes(
        img_rgb,
        template_bbox=template_bbox
    )

    curva = info_eje["polinomio"]

    prompts_auto = {}
    filas_debug = []

    template_ordenado = template_bbox.sort_values("id_real").reset_index(drop=True)

    y_anchor = float(template_ordenado.iloc[0]["cy_rel"] * H)
    y_min_template = float(template_ordenado["cy_rel"].min() * H)
    y_max_template = float(template_ordenado["cy_rel"].max() * H)

    for _, row in template_ordenado.iterrows():

        vertebra = row["vertebra"]

        if vertebra not in CLASES_OBJETIVO:
            continue

        id_real = int(row["id_real"])

        # Posición vertical original de la plantilla.
        cy_original = float(row["cy_rel"] * H)

        # Posición relativa entre T1 y L5.
        t = (cy_original - y_min_template) / (y_max_template - y_min_template + 1e-6)
        t = float(np.clip(t, 0, 1))

        # Corrección no lineal:
        # levanta más la zona media y casi no modifica los extremos.
        correccion_media = mid_lift * np.sin(np.pi * t)

        # Corrección vertical final.
        cy = y_anchor + escala_pos_y * (cy_original - y_anchor) + offset_y - correccion_media
        cy = float(np.clip(cy, 0, H - 1))

        # Centro horizontal desde el eje curvo, evaluado en la altura corregida.
        cx_eje = float(curva(cy))

        # Centro horizontal promedio desde plantilla.
        cx_template = float(row["cx_rel"] * W)

        # Mezcla eje automático + plantilla, con corrección horizontal.
        cx = peso_eje * cx_eje + (1 - peso_eje) * cx_template + offset_x
        cx = float(np.clip(cx, 0, W - 1))

        # Tamaño de caja.
        bw = float(row["w_rel"] * W * escala_w)
        bh = float(row["h_rel"] * H * escala_h)

        bw = float(np.clip(bw, W * min_w_frac, W * max_w_frac))
        bh = float(np.clip(bh, H * min_h_frac, H * max_h_frac))

        x0 = int(round(cx - bw / 2))
        x1 = int(round(cx + bw / 2))
        y0 = int(round(cy - bh / 2))
        y1 = int(round(cy + bh / 2))

        x0 = max(0, x0)
        y0 = max(0, y0)
        x1 = min(W - 1, x1)
        y1 = min(H - 1, y1)

        prompts_auto[vertebra] = {
            "vertebra": vertebra,
            "id_real": id_real,
            "bbox_xyxy": [x0, y0, x1, y1],
            "prompt_origen": "bbox_auto_bordes_eje_curvo_y_calibrado"
        }

        filas_debug.append({
            "vertebra": vertebra,
            "id_real": id_real,
            "cy_original": cy_original,
            "cy_final": cy,
            "t_vertical": t,
            "correccion_media": correccion_media,
            "cx_eje": cx_eje,
            "cx_template": cx_template,
            "cx_final": cx,
            "escala_pos_y": escala_pos_y,
            "offset_y": offset_y,
            "offset_x": offset_x,
            "mid_lift": mid_lift,
            "bbox_xyxy": [x0, y0, x1, y1]
        })

    df_debug = pd.DataFrame(filas_debug)

    if debug:
        return prompts_auto, info_eje, df_debug

    return prompts_auto, info_eje

In [ ]:
# ============================================================
# 12.6) Visualización de eje, bordes y cajas automáticas
# ============================================================

def visualizar_cajas_auto_bordes(
    img_rgb,
    prompts_auto,
    info_eje=None,
    mascara_gt=None,
    titulo="Cajas automáticas por bordes laterales"
):
    plt.figure(figsize=(8, 10))
    plt.imshow(img_rgb)

    ax = plt.gca()

    if mascara_gt is not None:
        gt_overlay = np.zeros_like(img_rgb)
        gt_overlay[..., 0] = (mascara_gt > 0).astype(np.uint8) * 255
        plt.imshow(gt_overlay, alpha=0.20)

    if info_eje is not None:
        x0, y0, x1, y1 = info_eje["roi"]

        rect_roi = plt.Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            fill=False,
            edgecolor="white",
            linewidth=1.2,
            linestyle="--"
        )
        ax.add_patch(rect_roi)

        puntos_izq = info_eje["puntos_izq"]
        puntos_der = info_eje["puntos_der"]
        puntos_centro = info_eje["puntos_centro_raw"]
        puntos_smooth = info_eje["puntos_smooth"]

        plt.scatter(
            puntos_izq[:, 1],
            puntos_izq[:, 0],
            s=10,
            c="orange",
            label="Borde izquierdo"
        )

        plt.scatter(
            puntos_der[:, 1],
            puntos_der[:, 0],
            s=10,
            c="yellow",
            label="Borde derecho"
        )

        plt.scatter(
            puntos_centro[:, 1],
            puntos_centro[:, 0],
            s=12,
            c="cyan",
            label="Centro por bordes"
        )

        plt.plot(
            puntos_smooth[:, 1],
            puntos_smooth[:, 0],
            c="cyan",
            linewidth=2,
            label="Eje curvo suavizado"
        )

    for vertebra, info in prompts_auto.items():

        x0, y0, x1, y1 = info["bbox_xyxy"]

        rect = plt.Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            fill=False,
            edgecolor="lime",
            linewidth=1.2
        )
        ax.add_patch(rect)

        ax.text(
            x0,
            max(0, y0 - 3),
            vertebra,
            fontsize=8,
            color="white",
            bbox=dict(facecolor="black", alpha=0.45, pad=1)
        )

    plt.title(titulo)
    plt.axis("off")
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
# ============================================================
# 12.7) Cobertura de cajas automáticas
# ============================================================

def evaluar_cobertura_cajas_por_clase(prompts_auto, mascara_gt_multiclase):
    """
    Evalúa qué tanto de cada vértebra real queda cubierta por su caja automática.

    La máscara se usa solo para evaluación.
    """

    filas = []

    for vertebra in CLASES_OBJETIVO:

        if vertebra not in prompts_auto:
            continue

        info = prompts_auto[vertebra]
        x0, y0, x1, y1 = info["bbox_xyxy"]

        gt_i = construir_mask_binaria_vertebra(
            mascara_gt_multiclase,
            vertebra
        ).astype(bool)

        total_gt = gt_i.sum()

        cover = np.zeros_like(gt_i, dtype=bool)
        cover[y0:y1 + 1, x0:x1 + 1] = True

        if total_gt == 0:
            cobertura = np.nan
        else:
            cobertura = np.logical_and(gt_i, cover).sum() / total_gt

        filas.append({
            "vertebra": vertebra,
            "id_real": VERTEBRA_TO_ID[vertebra],
            "bbox_auto": [x0, y0, x1, y1],
            "gt_area": int(total_gt),
            "bbox_area": int((x1 - x0 + 1) * (y1 - y0 + 1)),
            "bbox_recall": cobertura
        })

    return pd.DataFrame(filas).sort_values("id_real").reset_index(drop=True)


def diagnosticar_desplazamiento_cajas(prompts_auto, mascara_gt_multiclase):
    """
    Diagnóstico: compara centro de caja vs centro de la máscara real.
    Solo evaluación, no generación de cajas.
    """

    filas = []

    for vertebra in CLASES_OBJETIVO:

        if vertebra not in prompts_auto:
            continue

        x0, y0, x1, y1 = prompts_auto[vertebra]["bbox_xyxy"]

        cx_box = (x0 + x1) / 2
        cy_box = (y0 + y1) / 2

        gt_i = construir_mask_binaria_vertebra(
            mascara_gt_multiclase,
            vertebra
        ).astype(bool)

        if gt_i.sum() == 0:
            continue

        ys, xs = np.where(gt_i)

        cx_gt = xs.mean()
        cy_gt = ys.mean()

        filas.append({
            "vertebra": vertebra,
            "cx_box": cx_box,
            "cy_box": cy_box,
            "cx_gt": cx_gt,
            "cy_gt": cy_gt,
            "dx_box_gt": cx_box - cx_gt,
            "dy_box_gt": cy_box - cy_gt
        })

    return pd.DataFrame(filas)

In [ ]:
# ============================================================
# 12.8) Prueba visual y cobertura en un paciente
# con calibración vertical no lineal
# ============================================================

split_auto = "val"
patient_id_auto = "N_12"

path_img_auto, path_mask_auto = resolver_paths_muestra(split_auto, patient_id_auto)

imagen_auto = cargar_imagen(path_img_auto)
mascara_auto = cargar_mascara(path_mask_auto)

prompts_auto_sample, info_eje_auto, df_debug_prompts_auto = generar_prompts_auto_por_bordes(
    img_rgb=imagen_auto,
    template_bbox=template_bbox_auto,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    debug=True
)

print("Claves generadas:")
print(list(prompts_auto_sample.keys()))

faltantes = [v for v in CLASES_OBJETIVO if v not in prompts_auto_sample]
print("Faltantes:", faltantes)

display(df_debug_prompts_auto)

visualizar_cajas_auto_bordes(
    img_rgb=imagen_auto,
    prompts_auto=prompts_auto_sample,
    info_eje=info_eje_auto,
    mascara_gt=mascara_auto,
    titulo=f"{patient_id_auto} - cajas automáticas corregidas con mid_lift"
)

df_cobertura_auto = evaluar_cobertura_cajas_por_clase(
    prompts_auto_sample,
    mascara_auto
)

display(df_cobertura_auto)

print("Cobertura promedio de cajas:")
print(df_cobertura_auto["bbox_recall"].mean())

df_diag_desplazamiento = diagnosticar_desplazamiento_cajas(
    prompts_auto_sample,
    mascara_auto
)

display(df_diag_desplazamiento)

print("Desplazamiento horizontal promedio:")
print(df_diag_desplazamiento["dx_box_gt"].mean())

print("Desplazamiento vertical promedio:")
print(df_diag_desplazamiento["dy_box_gt"].mean())

In [ ]:
# ============================================================
# 12.8B) Comparación rápida de parámetros mid_lift
# ============================================================

resultados_mid_lift = []

for mid_lift_val in [0, 15, 22, 28]:

    prompts_tmp, info_eje_tmp, df_debug_tmp = generar_prompts_auto_por_bordes(
        img_rgb=imagen_auto,
        template_bbox=template_bbox_auto,
        escala_w=1.90,
        escala_h=1.35,
        peso_eje=0.90,
        escala_pos_y=0.91,
        offset_y=-16,
        offset_x=-18,
        mid_lift=mid_lift_val,
        debug=True
    )

    df_cobertura_tmp = evaluar_cobertura_cajas_por_clase(
        prompts_tmp,
        mascara_auto
    )

    df_diag_tmp = diagnosticar_desplazamiento_cajas(
        prompts_tmp,
        mascara_auto
    )

    resultados_mid_lift.append({
        "mid_lift": mid_lift_val,
        "bbox_recall_promedio": df_cobertura_tmp["bbox_recall"].mean(),
        "bbox_recall_min": df_cobertura_tmp["bbox_recall"].min(),
        "dx_promedio": df_diag_tmp["dx_box_gt"].mean(),
        "dy_promedio": df_diag_tmp["dy_box_gt"].mean(),
        "dy_abs_promedio": df_diag_tmp["dy_box_gt"].abs().mean()
    })

df_comparacion_mid_lift = pd.DataFrame(resultados_mid_lift)

display(df_comparacion_mid_lift)

In [ ]:
# ============================================================
# 12.8C) Visualización del mejor parámetro seleccionado
# ============================================================

mejor_mid_lift = 22

prompts_auto_sample, info_eje_auto, df_debug_prompts_auto = generar_prompts_auto_por_bordes(
    img_rgb=imagen_auto,
    template_bbox=template_bbox_auto,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=mejor_mid_lift,
    debug=True
)

visualizar_cajas_auto_bordes(
    img_rgb=imagen_auto,
    prompts_auto=prompts_auto_sample,
    info_eje=info_eje_auto,
    mascara_gt=mascara_auto,
    titulo=f"{patient_id_auto} - mejor mid_lift={mejor_mid_lift}"
)

df_cobertura_auto = evaluar_cobertura_cajas_por_clase(
    prompts_auto_sample,
    mascara_auto
)

df_diag_desplazamiento = diagnosticar_desplazamiento_cajas(
    prompts_auto_sample,
    mascara_auto
)

display(df_cobertura_auto)
display(df_diag_desplazamiento)

print("Cobertura promedio de cajas:")
print(df_cobertura_auto["bbox_recall"].mean())

print("Desplazamiento horizontal promedio:")
print(df_diag_desplazamiento["dx_box_gt"].mean())

print("Desplazamiento vertical promedio:")
print(df_diag_desplazamiento["dy_box_gt"].mean())

print("Desplazamiento vertical absoluto promedio:")
print(df_diag_desplazamiento["dy_box_gt"].abs().mean())

In [ ]:
# ============================================================
# 12.9) Visualizar predicción semántica del experimento 12
# ============================================================

def visualizar_prediccion_semantica_auto(
    img_rgb,
    mask_gt,
    mask_pred,
    prompts_auto=None,
    info_curva=None,
    titulo="Predicción semántica con cajas automáticas"
):
    plt.figure(figsize=(8, 10))
    plt.imshow(img_rgb)

    gt_overlay = np.zeros_like(img_rgb)
    pred_overlay = np.zeros_like(img_rgb)

    # GT en rojo
    gt_overlay[..., 0] = (mask_gt > 0).astype(np.uint8) * 255

    # Predicción en verde
    pred_overlay[..., 1] = (mask_pred > 0).astype(np.uint8) * 255

    plt.imshow(gt_overlay, alpha=0.25)
    plt.imshow(pred_overlay, alpha=0.25)

    ax = plt.gca()

    if info_curva is not None:
        puntos_smooth = info_curva["puntos_smooth"]
        plt.plot(
            puntos_smooth[:, 1],
            puntos_smooth[:, 0],
            c="cyan",
            linewidth=2
        )

    if prompts_auto is not None:
        for _, info in prompts_auto.items():
            x0, y0, x1, y1 = info["bbox_xyxy"]
            vertebra = info["vertebra"]

            rect = plt.Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                fill=False,
                edgecolor="yellow",
                linewidth=1.0
            )
            ax.add_patch(rect)

            ax.text(
                x0,
                max(0, y0 - 3),
                vertebra,
                fontsize=7,
                color="white",
                bbox=dict(facecolor="black", alpha=0.45, pad=1)
            )

    plt.title(titulo)
    plt.axis("off")
    plt.show()


visualizar_prediccion_semantica_auto(
    img_rgb=imagen_auto,
    mask_gt=mask_sem_gt_auto,
    mask_pred=mask_sem_pred_auto,
    prompts_auto=prompts_auto_sample,
    info_curva=info_curva_auto,
    titulo=f"{patient_id_auto} | Dice={dice_auto:.3f} | IoU={iou_auto:.3f}"
)

In [ ]:
# ============================================================
# 12.10) Evaluación del experimento 12 en split completo
# ============================================================

def evaluar_experimento12_muestra(
    split,
    patient_id,
    predictor,
    template_bbox,
    escala_w=1.45,
    escala_h=1.30,
    frac_x=0.04,
    frac_y=0.06
):
    """
    Evalúa una imagen usando cajas automáticas basadas en eje curvo.
    """

    path_img, path_mask = resolver_paths_muestra(split, patient_id)

    imagen = cargar_imagen(path_img)
    mascara = cargar_mascara(path_mask)

    prompts_auto, info_curva = generar_prompts_auto_desde_curva(
        img_rgb=imagen,
        template_bbox=template_bbox,
        escala_w=escala_w,
        escala_h=escala_h
    )

    df_cobertura = evaluar_cobertura_cajas_por_clase(
        prompts_auto,
        mascara
    )

    resultado = reconstruir_mascara_semantica_medsam_con_score(
        imagen=imagen,
        mascara_gt_multiclase=mascara,
        prompts_sample=prompts_auto,
        predictor=predictor,
        frac_x=frac_x,
        frac_y=frac_y,
        return_quality=True
    )

    if len(resultado) == 5:
        mask_sem_pred, mask_sem_gt, detalles, score_map, quality = resultado
    else:
        mask_sem_pred, mask_sem_gt, detalles, score_map = resultado
        quality = None

    clases_ids_eval = list(range(1, N_CLASES + 1))

    dice_global = dice_multiclase_promedio(
        mask_sem_pred,
        mask_sem_gt,
        clases_ids=clases_ids_eval
    )

    iou_global = iou_multiclase_promedio(
        mask_sem_pred,
        mask_sem_gt,
        clases_ids=clases_ids_eval
    )

    resumen = {
        "split": split,
        "patient_id": patient_id,
        "dice_macro": dice_global,
        "iou_macro": iou_global,
        "bbox_recall_promedio": df_cobertura["bbox_recall"].mean(),
        "bbox_recall_min": df_cobertura["bbox_recall"].min(),
        "n_vertebras_bbox_recall_mayor_08": int((df_cobertura["bbox_recall"] >= 0.80).sum()),
        "prompt_origen": "bbox_auto_curva"
    }

    if quality is not None and isinstance(quality, dict):
        for k, v in quality.items():
            resumen[f"quality_{k}"] = v

    df_cobertura["split"] = split
    df_cobertura["patient_id"] = patient_id

    return {
        "resumen": resumen,
        "df_cobertura": df_cobertura,
        "detalles": detalles,
        "mask_pred": mask_sem_pred,
        "mask_gt": mask_sem_gt,
        "imagen": imagen,
        "prompts_auto": prompts_auto,
        "info_curva": info_curva
    }


def evaluar_experimento12_muestra(
    split,
    patient_id,
    predictor,
    template_bbox,
    escala_w=1.90,
    escala_h=1.35,
    peso_eje=0.90,
    escala_pos_y=0.91,
    offset_y=-16,
    offset_x=-18,
    mid_lift=22,
    frac_x=0.04,
    frac_y=0.06
):
    """
    Evalúa el experimento 12 en un split completo o parcial.
    Recomendación: usar val durante desarrollo.
    """

    patient_ids = sorted(list(PROMPTS_DICC[split].keys()))

    if max_items is not None:
        patient_ids = patient_ids[:max_items]

    resumenes = []
    coberturas = []
    detalles_todos = []

    errores = []

    for patient_id in tqdm(patient_ids, desc=f"Experimento 12 - {split}"):

        try:
            out = evaluar_experimento12_muestra(
                split=split,
                patient_id=patient_id,
                predictor=predictor,
                template_bbox=template_bbox_auto,
                escala_w=escala_w,
                escala_h=escala_h,
                frac_x=frac_x,
                frac_y=frac_y
            )

            resumenes.append(out["resumen"])
            coberturas.append(out["df_cobertura"])

            df_det = pd.DataFrame(out["detalles"])
            df_det["split"] = split
            df_det["patient_id"] = patient_id
            detalles_todos.append(df_det)

        except Exception as e:
            errores.append({
                "split": split,
                "patient_id": patient_id,
                "error": str(e)
            })

            print(f"Error en {patient_id}: {e}")

    df_resumen = pd.DataFrame(resumenes)

    if len(coberturas) > 0:
        df_coberturas = pd.concat(coberturas, ignore_index=True)
    else:
        df_coberturas = pd.DataFrame()

    if len(detalles_todos) > 0:
        df_detalles = pd.concat(detalles_todos, ignore_index=True)
    else:
        df_detalles = pd.DataFrame()

    df_errores = pd.DataFrame(errores)

    return df_resumen, df_coberturas, df_detalles, df_errores

In [ ]:
df_exp12_resumen_val_5, df_exp12_cobertura_val_5, df_exp12_detalles_val_5, df_exp12_errores_val_5 = evaluar_experimento12_split(
    split="val",
    max_items=5,
    escala_w=1.45,
    escala_h=1.30,
    frac_x=0.04,
    frac_y=0.06
)

display(df_exp12_resumen_val_5)
display(df_exp12_cobertura_val_5)
display(df_exp12_errores_val_5)

print("Resumen parcial experimento 12:")
print("Dice macro promedio:", df_exp12_resumen_val_5["dice_macro"].mean())
print("IoU macro promedio :", df_exp12_resumen_val_5["iou_macro"].mean())
print("BBox recall promedio:", df_exp12_resumen_val_5["bbox_recall_promedio"].mean())

In [ ]:
df_exp12_resumen_val, df_exp12_cobertura_val, df_exp12_detalles_val, df_exp12_errores_val = evaluar_experimento12_split(
    split="val",
    max_items=None,
    escala_w=1.45,
    escala_h=1.30,
    frac_x=0.04,
    frac_y=0.06
)

display(df_exp12_resumen_val)
display(df_exp12_cobertura_val)
display(df_exp12_errores_val)

print("Resumen experimento 12 en validación")
print(f"Dice macro promedio      : {df_exp12_resumen_val['dice_macro'].mean():.4f}")
print(f"IoU macro promedio       : {df_exp12_resumen_val['iou_macro'].mean():.4f}")
print(f"BBox recall promedio     : {df_exp12_resumen_val['bbox_recall_promedio'].mean():.4f}")
print(f"BBox recall mínimo prom. : {df_exp12_resumen_val['bbox_recall_min'].mean():.4f}")

## 12. Evaluacion comun por split completo

Esta es la evaluacion reportable para comparar escenarios. Calcula metricas sobre todos los pacientes del split seleccionado, normalmente `val` durante desarrollo.

La evaluacion produce tablas por paciente, por vertebra, por grupo y por estrato clinico. Esas tablas son las que deben usarse para decidir si un cambio mejora realmente el pipeline.

El test no se usa aqui durante el desarrollo. Test queda reservado para la ultima seccion.

<!-- codex-explicacion -->
La evaluacion comun evita comparar casos aislados. Un modelo solo cuenta si mantiene rendimiento sobre el split completo.


In [ ]:
# ==========================================
# 58) FUNCIONES DE EVALUACION POR SPLIT COMPLETO
# ==========================================

import pandas as pd
from tqdm.auto import tqdm

RESULTS_ROOT = OUTPUT_ROOT / "medsam_evaluacion"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

def evaluar_paciente_semantico(patient_id, split_eval="val", frac_x=0.04, frac_y=0.06, guardar_mascaras=False):
    path_imagen, path_mascara = resolver_paths_muestra(split_eval, patient_id)

    imagen_eval = cargar_imagen(path_imagen)
    mascara_eval = cargar_mascara(path_mascara)
    prompts_eval = PROMPTS_FILTRADOS[split_eval][patient_id]

    (
        mask_sem_pred,
        mask_sem_gt,
        detalles,
        score_map,
        quality
    ) = reconstruir_mascara_semantica_medsam_con_score(
        imagen=imagen_eval,
        mascara_gt_multiclase=mascara_eval,
        prompts_sample=prompts_eval,
        predictor=predictor,
        frac_x=frac_x,
        frac_y=frac_y,
        return_quality=True
    )

    # Promedio macro por vertebra en el paciente: todas las vertebras evaluables pesan igual.
    dice_global = dice_multiclase_promedio(
        mask_sem_pred,
        mask_sem_gt,
        clases_ids=list(range(1, N_CLASES + 1))
    )

    iou_global = iou_multiclase_promedio(
        mask_sem_pred,
        mask_sem_gt,
        clases_ids=list(range(1, N_CLASES + 1))
    )

    grupo = "normal" if str(patient_id).startswith("N_") else "scoliosis" if str(patient_id).startswith("S_") else "otro"

    if 'df_splits' in globals() and 'estrato_split' in df_splits.columns:
        meta_split = df_splits.loc[df_splits['patient_id'] == patient_id]
        estrato_split = meta_split['estrato_split'].iloc[0] if len(meta_split) > 0 else grupo
        cobb_deg = meta_split['cobb_deg'].iloc[0] if len(meta_split) > 0 and 'cobb_deg' in meta_split.columns else np.nan
    else:
        estrato_split = grupo
        cobb_deg = np.nan

    vertebras_con_prompt = [v for v in CLASES_OBJETIVO if v in prompts_eval]
    vertebras_sin_prompt = [v for v in CLASES_OBJETIVO if v not in prompts_eval]

    resultado = {
        "split": split_eval,
        "patient_id": patient_id,
        "grupo": grupo,
        "estrato_split": estrato_split,
        "cobb_deg": cobb_deg,
        "detalles": detalles,
        "n_vertebras_eval": len(detalles),
        "n_vertebras_con_prompt": len(vertebras_con_prompt),
        "n_vertebras_sin_prompt": len(vertebras_sin_prompt),
        "vertebras_sin_prompt": ",".join(vertebras_sin_prompt),
        "dice_global": float(dice_global),
        "iou_global": float(iou_global),
        **quality
    }

    # Para evaluar todo el split no guardamos matrices grandes por defecto.
    # Si se quiere visualizar un caso puntual, activar guardar_mascaras=True en ese paciente.
    if guardar_mascaras:
        resultado.update({
            "imagen": imagen_eval,
            "mascara_gt": mascara_eval,
            "mask_sem_pred": mask_sem_pred,
            "mask_sem_gt": mask_sem_gt,
            "score_map": score_map
        })

    return resultado


def evaluar_split_completo(split_eval="val", frac_x=0.04, frac_y=0.06):
    patient_ids = sorted(PROMPTS_FILTRADOS[split_eval].keys())
    resultados = []
    errores = []

    for patient_id in tqdm(patient_ids, desc=f"Evaluando {split_eval} completo"):
        try:
            resultado = evaluar_paciente_semantico(
                patient_id=patient_id,
                split_eval=split_eval,
                frac_x=frac_x,
                frac_y=frac_y,
                guardar_mascaras=False
            )
            resultados.append(resultado)
        except Exception as e:
            errores.append({"split": split_eval, "patient_id": patient_id, "error": str(e)})

    df_resumen = pd.DataFrame([
        {
            "split": r["split"],
            "patient_id": r["patient_id"],
            "grupo": r["grupo"],
            "estrato_split": r["estrato_split"],
            "cobb_deg": r["cobb_deg"],
            "n_vertebras_eval": r["n_vertebras_eval"],
            "n_vertebras_con_prompt": r["n_vertebras_con_prompt"],
            "n_vertebras_sin_prompt": r["n_vertebras_sin_prompt"],
            "vertebras_sin_prompt": r["vertebras_sin_prompt"],
            "dice_global": r["dice_global"],
            "iou_global": r["iou_global"],
            "overlap_pixels": r["overlap_pixels"],
            "pred_pixels": r["pred_pixels"],
            "overlap_fraction_pred": r["overlap_fraction_pred"],
            "n_predicciones_vacias": r["n_predicciones_vacias"]
        }
        for r in resultados
    ]).sort_values(["grupo", "patient_id"]).reset_index(drop=True)

    filas_detalle = []
    for r in resultados:
        for d in r["detalles"]:
            filas_detalle.append({
                "split": r["split"],
                "patient_id": r["patient_id"],
                "grupo": r["grupo"],
                "estrato_split": r["estrato_split"],
                "cobb_deg": r["cobb_deg"],
                "vertebra": d["vertebra"],
                "id_real": d["id_real"],
                "score_medsam": d.get("score_medsam", np.nan),
                "pix_gt": d.get("pix_gt", np.nan),
                "pix_pred": d.get("pix_pred", np.nan),
                "pred_vacia": d.get("pred_vacia", False),
                "dice": d.get("dice", np.nan),
                "iou": d.get("iou", np.nan)
            })

    df_detalle = pd.DataFrame(filas_detalle)
    if len(df_detalle) > 0:
        df_detalle = df_detalle.sort_values(["grupo", "patient_id", "id_real"]).reset_index(drop=True)

    df_errores = pd.DataFrame(errores)
    return resultados, df_resumen, df_detalle, df_errores


### 12.1 Ejecutar evaluacion completa

Usa `SPLIT_EVALUACION = 'val'` mientras sigas comparando escenarios. Cambiarlo a `test` antes de congelar el modelo rompe la independencia del test.
        


In [ ]:
# ==========================================
# 59) EVALUAR TODO EL SPLIT SELECCIONADO
# ==========================================

SPLIT_EVALUACION = 'val'  # cambiar a 'test' para el reporte final

resultados_eval, df_resumen_eval, df_detalle_eval, df_errores_eval = evaluar_split_completo(
    split_eval=SPLIT_EVALUACION,
    frac_x=0.04,
    frac_y=0.06
)

print('Split evaluado:', SPLIT_EVALUACION)
print('Pacientes evaluados:', len(df_resumen_eval))
print('Vertebras evaluadas:', len(df_detalle_eval))
print('Errores:', len(df_errores_eval))

display(df_resumen_eval)
if len(df_errores_eval) > 0:
    display(df_errores_eval)


In [ ]:
# ==========================================
# 60) RESUMEN GLOBAL, INTERVALOS Y GUARDADO
# ==========================================

def bootstrap_ci(values, n_boot=2000, ci=95, seed=SEED):
    values = pd.Series(values).dropna().to_numpy(dtype=float)
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot_means = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_means.append(sample.mean())
    alpha = (100 - ci) / 2
    return (
        float(np.percentile(boot_means, alpha)),
        float(np.percentile(boot_means, 100 - alpha))
    )


def guardar_tablas_evaluacion(nombre_eval, df_resumen, df_detalle, df_errores, resumenes):
    out_dir = RESULTS_ROOT / nombre_eval
    out_dir.mkdir(parents=True, exist_ok=True)

    df_resumen.to_csv(out_dir / "resumen_paciente.csv", index=False)
    df_detalle.to_csv(out_dir / "detalle_vertebra.csv", index=False)
    df_errores.to_csv(out_dir / "errores.csv", index=False)

    for nombre, df_tabla in resumenes.items():
        df_tabla.to_csv(out_dir / f"resumen_{nombre}.csv", index=False)

    print("Tablas guardadas en:", out_dir)


def resumir_metricas_split(df_resumen, df_detalle, df_errores, nombre_split, nombre_eval=None, guardar=True):
    dice_ci_low, dice_ci_high = bootstrap_ci(df_resumen["dice_global"])
    iou_ci_low, iou_ci_high = bootstrap_ci(df_resumen["iou_global"])

    resumen_global = pd.DataFrame([{
        "split": nombre_split,
        "n_pacientes": int(len(df_resumen)),
        "n_vertebras_eval": int(len(df_detalle)),
        "dice_macro_paciente_mean": df_resumen["dice_global"].mean(),
        "dice_macro_paciente_std": df_resumen["dice_global"].std(),
        "dice_macro_paciente_median": df_resumen["dice_global"].median(),
        "dice_ci95_low": dice_ci_low,
        "dice_ci95_high": dice_ci_high,
        "iou_macro_paciente_mean": df_resumen["iou_global"].mean(),
        "iou_macro_paciente_std": df_resumen["iou_global"].std(),
        "iou_macro_paciente_median": df_resumen["iou_global"].median(),
        "iou_ci95_low": iou_ci_low,
        "iou_ci95_high": iou_ci_high,
        "n_vertebras_promedio_por_paciente": df_resumen["n_vertebras_eval"].mean(),
        "overlap_fraction_pred_mean": df_resumen["overlap_fraction_pred"].mean(),
        "predicciones_vacias_total": int(df_resumen["n_predicciones_vacias"].sum())
    }])

    resumen_grupo = (
        df_resumen
        .groupby("grupo", as_index=False)
        .agg(
            n_pacientes=("patient_id", "nunique"),
            n_vertebras_promedio=("n_vertebras_eval", "mean"),
            dice_mean=("dice_global", "mean"),
            dice_std=("dice_global", "std"),
            dice_median=("dice_global", "median"),
            iou_mean=("iou_global", "mean"),
            iou_std=("iou_global", "std"),
            iou_median=("iou_global", "median"),
            overlap_fraction_pred_mean=("overlap_fraction_pred", "mean")
        )
    )

    resumen_estrato = (
        df_resumen
        .groupby("estrato_split", as_index=False)
        .agg(
            n_pacientes=("patient_id", "nunique"),
            dice_mean=("dice_global", "mean"),
            dice_std=("dice_global", "std"),
            iou_mean=("iou_global", "mean"),
            iou_std=("iou_global", "std"),
            cobb_mean=("cobb_deg", "mean")
        )
    )

    resumen_vertebra = (
        df_detalle
        .groupby(["vertebra", "id_real"], as_index=False)
        .agg(
            n=("dice", "count"),
            dice_mean=("dice", "mean"),
            dice_std=("dice", "std"),
            dice_median=("dice", "median"),
            iou_mean=("iou", "mean"),
            iou_std=("iou", "std"),
            iou_median=("iou", "median"),
            score_mean=("score_medsam", "mean"),
            pred_vacia_total=("pred_vacia", "sum"),
            pix_gt_mean=("pix_gt", "mean"),
            pix_pred_mean=("pix_pred", "mean")
        )
        .sort_values("id_real")
    )

    resumen_grupo_vertebra = (
        df_detalle
        .groupby(["grupo", "vertebra", "id_real"], as_index=False)
        .agg(
            n=("dice", "count"),
            dice_mean=("dice", "mean"),
            dice_median=("dice", "median"),
            iou_mean=("iou", "mean"),
            iou_median=("iou", "median"),
            score_mean=("score_medsam", "mean")
        )
        .sort_values(["grupo", "id_real"])
    )

    resumen_cobertura = pd.DataFrame([{
        "split": nombre_split,
        "n_pacientes": int(len(df_resumen)),
        "n_pacientes_con_todas_t1_l5": int((df_resumen["n_vertebras_sin_prompt"] == 0).sum()),
        "n_pacientes_incompletos": int((df_resumen["n_vertebras_sin_prompt"] > 0).sum()),
        "n_vertebras_sin_prompt_total": int(df_resumen["n_vertebras_sin_prompt"].sum())
    }])

    print("===== RESUMEN GLOBAL =====")
    display(resumen_global)
    print("===== COBERTURA DE VERTEBRAS =====")
    display(resumen_cobertura)
    print("===== RESUMEN POR GRUPO =====")
    display(resumen_grupo)
    print("===== RESUMEN POR ESTRATO CLINICO =====")
    display(resumen_estrato)
    print("===== RESUMEN POR VERTEBRA =====")
    display(resumen_vertebra)
    print("===== RESUMEN POR GRUPO Y VERTEBRA =====")
    display(resumen_grupo_vertebra)

    resumenes = {
        "global": resumen_global,
        "cobertura": resumen_cobertura,
        "grupo": resumen_grupo,
        "estrato": resumen_estrato,
        "vertebra": resumen_vertebra,
        "grupo_vertebra": resumen_grupo_vertebra
    }

    if guardar:
        if nombre_eval is None:
            nombre_eval = f"medsam_{nombre_split}"
        guardar_tablas_evaluacion(nombre_eval, df_resumen, df_detalle, df_errores, resumenes)

    return resumenes


nombre_eval = f"modelo_actual_{SPLIT_EVALUACION}_prompt_gt_fracx004_fracy006"
resumen_eval = resumir_metricas_split(
    df_resumen_eval,
    df_detalle_eval,
    df_errores_eval,
    SPLIT_EVALUACION,
    nombre_eval=nombre_eval,
    guardar=True
)


### Visualizacion de casos puntuales

Las visualizaciones se mantienen como inspeccion cualitativa. No se usan para seleccionar manualmente que pacientes entran en el promedio.


In [ ]:
# ==========================================
# 62) VISUALIZAR UN CASO DEL SPLIT EVALUADO
# ==========================================

def visualizar_resultado_semantico(resultado):
    fig, ax = plt.subplots(1, 3, figsize=(18, 8))

    ax[0].imshow(resultado['imagen'])
    ax[0].set_title(f"Imagen - {resultado['patient_id']} ({resultado['grupo']})")
    ax[0].axis('off')

    ax[1].imshow(resultado['mask_sem_gt'], cmap='nipy_spectral', vmin=0, vmax=N_CLASES)
    ax[1].set_title('GT semantica T1-L5')
    ax[1].axis('off')

    ax[2].imshow(resultado['mask_sem_pred'], cmap='nipy_spectral', vmin=0, vmax=N_CLASES)
    ax[2].set_title(f"Pred semantica | Dice={resultado['dice_global']:.3f} | IoU={resultado['iou_global']:.3f}")
    ax[2].axis('off')

    plt.tight_layout()
    plt.show()

# Ejemplo cualitativo: primer paciente del split evaluado.
# No afecta las tablas ni los promedios anteriores.
if len(df_resumen_eval) > 0:
    paciente_demo = df_resumen_eval.iloc[0]['patient_id']
    resultado_demo = evaluar_paciente_semantico(
        patient_id=paciente_demo,
        split_eval=SPLIT_EVALUACION,
        frac_x=0.04,
        frac_y=0.06,
        guardar_mascaras=True
    )
    visualizar_resultado_semantico(resultado_demo)


In [ ]:
# ==========================================
# 67) CONFIGURACION DE REEQUILIBRIO
# ==========================================

# Vertebras que queremos priorizar más
VERTEBRAS_DIFICILES = ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8"]

# Pesos de muestreo
PESO_NORMAL = 1.0
PESO_ESCOLIOSIS = 2.0

PESO_VERTEBRA_NORMAL = 1.0
PESO_VERTEBRA_DIFICIL = 1.8

print("VERTEBRAS_DIFICILES:", VERTEBRAS_DIFICILES)
print("PESO_NORMAL:", PESO_NORMAL)
print("PESO_ESCOLIOSIS:", PESO_ESCOLIOSIS)
print("PESO_VERTEBRA_NORMAL:", PESO_VERTEBRA_NORMAL)
print("PESO_VERTEBRA_DIFICIL:", PESO_VERTEBRA_DIFICIL)


## 13. Escenario 2: decoder_plus_weighted_sampler

**Tipo:** cambio 1 + cambio 2, escenario acumulativo.

**Hipotesis:** si `decoder_basico` aprende algo util, agregar reequilibrio puede ayudar a que clases o condiciones menos representadas tengan mas peso durante el entrenamiento.

**Que cambia respecto al escenario 1:** se usa un dataset/sampler ponderado para modificar la frecuencia efectiva de entrenamiento.

**Advertencia metodologica:** tal como esta escrito el notebook, este bloque parte del modelo ya ajustado en `decoder_basico`. Por tanto, no mide el efecto aislado del sampler; mide el efecto acumulado `decoder_basico + weighted_sampler`.

Si se quiere medir solo el cambio 2 de forma independiente, se debe recargar el checkpoint inicial de MedSAM antes de este bloque.

<!-- codex-explicacion -->
Este escenario intenta compensar desbalance entre vertebras/clases. Es una prueba razonable cuando algunas etiquetas aparecen menos que otras.


In [ ]:
# ==========================================
# 68) DATASET REEQUILIBRADO
# ==========================================

from torch.utils.data import Dataset

class MedSAMVertebraDatasetWeighted(Dataset):
    def __init__(self, split, prompts_filtrados, split_info, frac_x=0.04, frac_y=0.06):
        self.split = split
        self.prompts_filtrados = prompts_filtrados
        self.split_info = split_info
        self.frac_x = frac_x
        self.frac_y = frac_y

        self.samples = []
        self.sample_weights = []

        for patient_id, prompts_sample in self.prompts_filtrados[self.split].items():
            grupo = "normal" if str(patient_id).startswith("N_") else "scoliosis" if str(patient_id).startswith("S_") else "otro"

            for vertebra_objetivo in CLASES_OBJETIVO:
                if vertebra_objetivo not in prompts_sample:
                    continue

                peso_grupo = PESO_ESCOLIOSIS if grupo == "scoliosis" else PESO_NORMAL
                peso_vertebra = PESO_VERTEBRA_DIFICIL if vertebra_objetivo in VERTEBRAS_DIFICILES else PESO_VERTEBRA_NORMAL
                peso_total = peso_grupo * peso_vertebra

                self.samples.append({
                    "patient_id": patient_id,
                    "grupo": grupo,
                    "vertebra": vertebra_objetivo,
                    "bbox": prompts_sample[vertebra_objetivo]["bbox_xyxy"],
                    "weight": peso_total
                })

                self.sample_weights.append(float(peso_total))

        print(f"[{self.split}] muestras:", len(self.samples))

        # Diagnóstico rápido
        import pandas as pd
        df_tmp = pd.DataFrame(self.samples)
        resumen = df_tmp.groupby(["grupo", "vertebra"]).size().reset_index(name="n")
        print(f"[{self.split}] resumen por grupo/vertebra:")
        display(resumen.head(20))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        patient_id = item["patient_id"]
        vertebra_objetivo = item["vertebra"]
        bbox_original = item["bbox"]
        grupo = item["grupo"]
        weight = item["weight"]

        path_imagen, path_mascara = resolver_paths_muestra(self.split, patient_id)

        imagen = cargar_imagen(path_imagen)
        mascara = cargar_mascara(path_mascara)

        bbox_expandida = expand_bbox_xyxy_pequena(
            bbox_original,
            imagen.shape,
            frac_x=self.frac_x,
            frac_y=self.frac_y
        )

        mask_gt_bin = construir_mask_binaria_vertebra(mascara, vertebra_objetivo).astype(np.float32)

        image_uint8 = imagen.astype(np.uint8)
        box = torch.tensor(bbox_expandida, dtype=torch.float32)
        gt_mask = torch.tensor(mask_gt_bin[None, :, :], dtype=torch.float32)
        sample_weight = torch.tensor(weight, dtype=torch.float32)

        return {
            "patient_id": patient_id,
            "grupo": grupo,
            "vertebra": vertebra_objetivo,
            "image": image_uint8,
            "box": box,
            "gt_mask": gt_mask,
            "sample_weight": sample_weight
        }


In [ ]:
# ==========================================
# 69) DATALOADERS REEQUILIBRADOS
# ==========================================

from torch.utils.data import DataLoader, WeightedRandomSampler

batch_size = 2
workers = 0

train_dataset_w = MedSAMVertebraDatasetWeighted(
    split="train",
    prompts_filtrados=PROMPTS_FILTRADOS,
    split_info=SPLIT_INFO,
    frac_x=0.04,
    frac_y=0.06
)

val_dataset_w = MedSAMVertebraDatasetWeighted(
    split="val",
    prompts_filtrados=PROMPTS_FILTRADOS,
    split_info=SPLIT_INFO,
    frac_x=0.04,
    frac_y=0.06
)

def collate_fn_medsam(batch):
    return batch

train_sampler_w = WeightedRandomSampler(
    weights=train_dataset_w.sample_weights,
    num_samples=len(train_dataset_w.sample_weights),
    replacement=True
)

train_loader_w = DataLoader(
    train_dataset_w,
    batch_size=batch_size,
    sampler=train_sampler_w,
    num_workers=workers,
    collate_fn=collate_fn_medsam
)

val_loader_w = DataLoader(
    val_dataset_w,
    batch_size=batch_size,
    shuffle=False,
    num_workers=workers,
    collate_fn=collate_fn_medsam
)

print("Train weighted batches:", len(train_loader_w))
print("Val weighted batches  :", len(val_loader_w))


In [ ]:
# ==========================================
# 70) CONFIGURAR MONITOREO DOBLE
# ==========================================

# Escoge uno normal de val
pacientes_val_normales = [p for p in PROMPTS_FILTRADOS["val"].keys() if str(p).startswith("N_")]
pacientes_val_scol = [p for p in PROMPTS_FILTRADOS["val"].keys() if str(p).startswith("S_")]

PACIENTE_MONITOREO_NORMAL = pacientes_val_normales[0]
PACIENTE_MONITOREO_SCOL = "S_141" if "S_141" in pacientes_val_scol else pacientes_val_scol[0]

VERTEBRA_MONITOREO_NORMAL = "L3"
VERTEBRA_MONITOREO_SCOL = "T4" if "T4" in PROMPTS_FILTRADOS["val"][PACIENTE_MONITOREO_SCOL] else list(PROMPTS_FILTRADOS["val"][PACIENTE_MONITOREO_SCOL].keys())[0]

print("PACIENTE_MONITOREO_NORMAL:", PACIENTE_MONITOREO_NORMAL)
print("VERTEBRA_MONITOREO_NORMAL:", VERTEBRA_MONITOREO_NORMAL)
print("PACIENTE_MONITOREO_SCOL  :", PACIENTE_MONITOREO_SCOL)
print("VERTEBRA_MONITOREO_SCOL  :", VERTEBRA_MONITOREO_SCOL)


In [ ]:
# ==========================================
# 71) PREPARAR MUESTRAS DE MONITOREO
# ==========================================

def preparar_muestra_monitoreo(split_name, patient_id, vertebra_objetivo):
    path_img, path_mask = resolver_paths_muestra(split_name, patient_id)

    imagen_loc = cargar_imagen(path_img)
    mascara_loc = cargar_mascara(path_mask)

    prompt_info = PROMPTS_FILTRADOS[split_name][patient_id][vertebra_objetivo]
    bbox_original = prompt_info["bbox_xyxy"]
    bbox_expandida = expand_bbox_xyxy_pequena(
        bbox_original,
        imagen_loc.shape,
        frac_x=0.04,
        frac_y=0.06
    )

    mask_gt = construir_mask_binaria_vertebra(mascara_loc, vertebra_objetivo).astype(np.float32)
    mask_gt_t = torch.tensor(mask_gt[None, :, :], dtype=torch.float32).to(device)
    box_t = torch.tensor(bbox_expandida, dtype=torch.float32).to(device)

    return {
        "split": split_name,
        "patient_id": patient_id,
        "vertebra": vertebra_objetivo,
        "imagen": imagen_loc,
        "mascara": mascara_loc,
        "bbox_original": bbox_original,
        "bbox": bbox_expandida,
        "mask_gt": mask_gt,
        "mask_gt_t": mask_gt_t,
        "box_t": box_t
    }

muestra_mon_normal = preparar_muestra_monitoreo("val", PACIENTE_MONITOREO_NORMAL, VERTEBRA_MONITOREO_NORMAL)
muestra_mon_scol = preparar_muestra_monitoreo("val", PACIENTE_MONITOREO_SCOL, VERTEBRA_MONITOREO_SCOL)

print("Normal ->", muestra_mon_normal["patient_id"], muestra_mon_normal["vertebra"], muestra_mon_normal["bbox"])
print("Scol   ->", muestra_mon_scol["patient_id"], muestra_mon_scol["vertebra"], muestra_mon_scol["bbox"])


In [ ]:
# ==========================================
# 72) MONITOREO DOBLE
# ==========================================

def evaluar_una_muestra_monitoreo(muestra, titulo=""):
    medsam_model.eval()

    with torch.no_grad():
        pred_logits, iou_pred = forward_medsam_with_box(
            muestra["imagen"],
            muestra["box_t"]
        )

    dice_mon, iou_mon = visualizar_resultado_medsam(
        image_np=muestra["imagen"],
        bbox=muestra["bbox"],
        mask_gt=muestra["mask_gt_t"],
        pred_logits=pred_logits,
        titulo=f"{titulo} | {muestra['patient_id']} | {muestra['vertebra']}"
    )

    score_mon = None
    if torch.is_tensor(iou_pred):
        score_mon = float(iou_pred.reshape(-1)[0].detach().cpu().item())

    print(
        f"{muestra['patient_id']} | {muestra['vertebra']} -> "
        f"Dice={dice_mon:.4f} | IoU={iou_mon:.4f} | Score={score_mon}"
    )

    return dice_mon, iou_mon, score_mon

def evaluar_monitoreo_doble(titulo=""):
    dice_n, iou_n, score_n = evaluar_una_muestra_monitoreo(
        muestra_mon_normal,
        titulo=f"{titulo} | NORMAL"
    )

    dice_s, iou_s, score_s = evaluar_una_muestra_monitoreo(
        muestra_mon_scol,
        titulo=f"{titulo} | SCOLIOSIS"
    )

    return {
        "dice_normal": dice_n,
        "iou_normal": iou_n,
        "score_normal": score_n,
        "dice_scol": dice_s,
        "iou_scol": iou_s,
        "score_scol": score_s
    }


In [ ]:
# ==========================================
# 73) CHEQUEO INICIAL REEQUILIBRADO
# ==========================================

monitor_init = evaluar_monitoreo_doble(titulo="Antes de entrenar reequilibrado")
print(monitor_init)


In [ ]:
# ==========================================
# 74) FUNCION DE ENTRENAMIENTO CON PESOS
# ==========================================

from tqdm.auto import tqdm

def run_epoch_medsam_weighted(loader, train=True, epoch=None):
    if train:
        medsam_model.train()
        desc = f"TrainW Epoch {epoch}" if epoch is not None else "TrainW"
    else:
        medsam_model.eval()
        desc = f"ValW Epoch {epoch}" if epoch is not None else "ValW"

    total_loss = 0.0
    total_bce = 0.0
    total_dice = 0.0
    n_samples = 0

    pbar = tqdm(loader, desc=desc, leave=True)

    for batch in pbar:
        for sample in batch:
            image_np = sample["image"]
            box = sample["box"].to(device)
            gt_mask = sample["gt_mask"].to(device)
            sample_weight = sample["sample_weight"].to(device)

            if train:
                optimizer.zero_grad()

            with torch.set_grad_enabled(train):
                pred_logits, iou_pred = forward_medsam_with_box(image_np, box)
                gt_mask_batch = gt_mask.unsqueeze(0) if gt_mask.ndim == 3 else gt_mask

                loss_bce = bce_loss(pred_logits, gt_mask_batch)
                loss_dice = dice_loss_from_logits(pred_logits, gt_mask_batch)
                loss = (loss_bce + loss_dice) * sample_weight

                if train:
                    loss.backward()
                    optimizer.step()

            total_loss += loss.item()
            total_bce += loss_bce.item()
            total_dice += loss_dice.item()
            n_samples += 1

        pbar.set_postfix({
            "loss": f"{total_loss / max(n_samples, 1):.4f}",
            "bce": f"{total_bce / max(n_samples, 1):.4f}",
            "dice_loss": f"{total_dice / max(n_samples, 1):.4f}",
        })

    return {
        "loss": total_loss / max(n_samples, 1),
        "bce": total_bce / max(n_samples, 1),
        "dice_loss": total_dice / max(n_samples, 1)
    }


In [ ]:
# ==========================================
# 75) LOOP REEQUILIBRADO CON MONITOREO DOBLE
# ==========================================

num_epochs = 2
best_val_loss_w = float("inf")
best_state_dict_w = None
patience = 2
patience_counter = 0

history_w = []
monitor_history_w = []

for epoch in range(1, num_epochs + 1):
    train_metrics = run_epoch_medsam_weighted(train_loader_w, train=True, epoch=epoch)
    val_metrics = run_epoch_medsam_weighted(val_loader_w, train=False, epoch=epoch)

    scheduler.step(val_metrics["loss"])

    mon = evaluar_monitoreo_doble(titulo=f"Epoch {epoch}")

    history_w.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_bce": train_metrics["bce"],
        "train_dice_loss": train_metrics["dice_loss"],
        "val_loss": val_metrics["loss"],
        "val_bce": val_metrics["bce"],
        "val_dice_loss": val_metrics["dice_loss"],
        "lr": optimizer.param_groups[0]["lr"]
    })

    monitor_history_w.append({
        "epoch": epoch,
        "dice_normal": mon["dice_normal"],
        "iou_normal": mon["iou_normal"],
        "score_normal": mon["score_normal"],
        "dice_scol": mon["dice_scol"],
        "iou_scol": mon["iou_scol"],
        "score_scol": mon["score_scol"]
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"Dice_normal={mon['dice_normal']:.4f} | "
        f"Dice_scol={mon['dice_scol']:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.2e}"
    )

    if val_metrics["loss"] < best_val_loss_w:
        best_val_loss_w = val_metrics["loss"]
        best_state_dict_w = {k: v.cpu().clone() for k, v in medsam_model.state_dict().items()}
        patience_counter = 0
        print("  -> nuevo mejor modelo reequilibrado")
    else:
        patience_counter += 1
        print(f"  -> sin mejora ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print("Early stopping activado")
        break


In [ ]:
# ==========================================
# 76) RESTAURAR MEJOR MODELO REEQUILIBRADO
# ==========================================

from segment_anything import SamPredictor

if best_state_dict_w is not None:
    medsam_model.load_state_dict(best_state_dict_w)
    medsam_model = medsam_model.to(device)
    medsam_model.eval()
    predictor = SamPredictor(medsam_model)
    print("Mejor modelo reequilibrado restaurado.")
else:
    print("No se guardó best_state_dict_w.")


In [ ]:
# ==========================================
# 77) HISTORIAL REEQUILIBRADO
# ==========================================

import pandas as pd

df_history_w = pd.DataFrame(history_w)
df_monitor_w = pd.DataFrame(monitor_history_w)

display(df_history_w)
display(df_monitor_w)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(df_history_w["epoch"], df_history_w["train_loss"], marker="o", label="train_loss")
ax.plot(df_history_w["epoch"], df_history_w["val_loss"], marker="o", label="val_loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Entrenamiento reequilibrado")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(df_monitor_w["epoch"], df_monitor_w["dice_normal"], marker="o", label="dice_normal")
ax.plot(df_monitor_w["epoch"], df_monitor_w["dice_scol"], marker="o", label="dice_scol")
ax.set_xlabel("Epoch")
ax.set_ylabel("Dice")
ax.set_title("Monitoreo doble: normal vs scoliosis")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()


## 14. Escenario 3: decoder_plus_weighted_plus_soft

**Tipo:** escenario acumulativo.

**Hipotesis:** una ponderacion suave en la loss puede estabilizar el reequilibrio, reduciendo el riesgo de que el sampler sobrecorrija el desbalance.

**Que cambia respecto al escenario 2:** en vez de depender solo de la frecuencia de muestreo, se introduce una ponderacion mas controlada en la funcion de perdida.

**Lectura correcta:** este escenario debe compararse contra `decoder_basico` y contra `decoder_plus_weighted_sampler` en validacion. Si mejora, se interpreta como una receta acumulativa prometedora, no como el efecto aislado de la loss suave.

<!-- codex-explicacion -->
La perdida soft busca suavizar el aprendizaje y reducir castigos excesivos en bordes. Fue una forma de explorar estabilidad sin cambiar todo el modelo.


In [ ]:
# ==========================================
# 78) CONFIGURACION SUAVE DE REEQUILIBRIO
# ==========================================

VERTEBRAS_DIFICILES_SUAVE = ["T1", "T2", "T3", "T4", "T5", "T6"]

PESO_NORMAL_SUAVE = 1.0
PESO_ESCOLIOSIS_SUAVE = 1.3

PESO_VERTEBRA_NORMAL_SUAVE = 1.0
PESO_VERTEBRA_DIFICIL_SUAVE = 1.2

print("VERTEBRAS_DIFICILES_SUAVE:", VERTEBRAS_DIFICILES_SUAVE)
print("PESO_NORMAL_SUAVE:", PESO_NORMAL_SUAVE)
print("PESO_ESCOLIOSIS_SUAVE:", PESO_ESCOLIOSIS_SUAVE)
print("PESO_VERTEBRA_NORMAL_SUAVE:", PESO_VERTEBRA_NORMAL_SUAVE)
print("PESO_VERTEBRA_DIFICIL_SUAVE:", PESO_VERTEBRA_DIFICIL_SUAVE)


In [ ]:
# ==========================================
# 79) DATASET SUAVE (PESO EN LOSS, NO EN SAMPLER)
# ==========================================

class MedSAMVertebraDatasetSoft(Dataset):
    def __init__(self, split, prompts_filtrados, split_info, frac_x=0.04, frac_y=0.06):
        self.split = split
        self.prompts_filtrados = prompts_filtrados
        self.split_info = split_info
        self.frac_x = frac_x
        self.frac_y = frac_y

        self.samples = []

        for patient_id, prompts_sample in self.prompts_filtrados[self.split].items():
            grupo = "normal" if str(patient_id).startswith("N_") else "scoliosis" if str(patient_id).startswith("S_") else "otro"

            for vertebra_objetivo in CLASES_OBJETIVO:
                if vertebra_objetivo not in prompts_sample:
                    continue

                peso_grupo = PESO_ESCOLIOSIS_SUAVE if grupo == "scoliosis" else PESO_NORMAL_SUAVE
                peso_vertebra = (
                    PESO_VERTEBRA_DIFICIL_SUAVE
                    if vertebra_objetivo in VERTEBRAS_DIFICILES_SUAVE
                    else PESO_VERTEBRA_NORMAL_SUAVE
                )
                peso_total = peso_grupo * peso_vertebra

                self.samples.append({
                    "patient_id": patient_id,
                    "grupo": grupo,
                    "vertebra": vertebra_objetivo,
                    "bbox": prompts_sample[vertebra_objetivo]["bbox_xyxy"],
                    "weight": float(peso_total)
                })

        print(f"[{self.split}] muestras:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        patient_id = item["patient_id"]
        vertebra_objetivo = item["vertebra"]
        bbox_original = item["bbox"]
        grupo = item["grupo"]
        weight = item["weight"]

        path_imagen, path_mascara = resolver_paths_muestra(self.split, patient_id)

        imagen = cargar_imagen(path_imagen)
        mascara = cargar_mascara(path_mascara)

        bbox_expandida = expand_bbox_xyxy_pequena(
            bbox_original,
            imagen.shape,
            frac_x=self.frac_x,
            frac_y=self.frac_y
        )

        mask_gt_bin = construir_mask_binaria_vertebra(mascara, vertebra_objetivo).astype(np.float32)

        image_uint8 = imagen.astype(np.uint8)
        box = torch.tensor(bbox_expandida, dtype=torch.float32)
        gt_mask = torch.tensor(mask_gt_bin[None, :, :], dtype=torch.float32)
        sample_weight = torch.tensor(weight, dtype=torch.float32)

        return {
            "patient_id": patient_id,
            "grupo": grupo,
            "vertebra": vertebra_objetivo,
            "image": image_uint8,
            "box": box,
            "gt_mask": gt_mask,
            "sample_weight": sample_weight
        }


In [ ]:
# ==========================================
# 80) DATALOADERS SUAVES
# ==========================================

batch_size = 2
workers = 0

train_dataset_soft = MedSAMVertebraDatasetSoft(
    split="train",
    prompts_filtrados=PROMPTS_FILTRADOS,
    split_info=SPLIT_INFO,
    frac_x=0.04,
    frac_y=0.06
)

val_dataset_soft = MedSAMVertebraDatasetSoft(
    split="val",
    prompts_filtrados=PROMPTS_FILTRADOS,
    split_info=SPLIT_INFO,
    frac_x=0.04,
    frac_y=0.06
)

train_loader_soft = DataLoader(
    train_dataset_soft,
    batch_size=batch_size,
    shuffle=True,
    num_workers=workers,
    collate_fn=collate_fn_medsam
)

val_loader_soft = DataLoader(
    val_dataset_soft,
    batch_size=batch_size,
    shuffle=False,
    num_workers=workers,
    collate_fn=collate_fn_medsam
)

print("Train soft batches:", len(train_loader_soft))
print("Val soft batches  :", len(val_loader_soft))


In [ ]:
# ==========================================
# 81) MONITOREO DOBLE NUEVO (SIN S_141)
# ==========================================

pacientes_val_normales = [p for p in PROMPTS_FILTRADOS["val"].keys() if str(p).startswith("N_")]
pacientes_val_scol = [p for p in PROMPTS_FILTRADOS["val"].keys() if str(p).startswith("S_") and p != "S_141"]

PACIENTE_MONITOREO_NORMAL = pacientes_val_normales[0]

# elegir un caso de escoliosis con cobertura razonable
def contar_vertebras_objetivo(pid):
    return sum(v in PROMPTS_FILTRADOS["val"][pid] for v in CLASES_OBJETIVO)

pacientes_val_scol = sorted(pacientes_val_scol, key=lambda p: contar_vertebras_objetivo(p), reverse=True)
PACIENTE_MONITOREO_SCOL = pacientes_val_scol[0]

VERTEBRA_MONITOREO_NORMAL = "L3" if "L3" in PROMPTS_FILTRADOS["val"][PACIENTE_MONITOREO_NORMAL] else list(PROMPTS_FILTRADOS["val"][PACIENTE_MONITOREO_NORMAL].keys())[0]

# para escoliosis, elegir una torácica media/alta si existe
for cand in ["T4", "T5", "T6", "T7", "L3"]:
    if cand in PROMPTS_FILTRADOS["val"][PACIENTE_MONITOREO_SCOL]:
        VERTEBRA_MONITOREO_SCOL = cand
        break
else:
    VERTEBRA_MONITOREO_SCOL = list(PROMPTS_FILTRADOS["val"][PACIENTE_MONITOREO_SCOL].keys())[0]

print("PACIENTE_MONITOREO_NORMAL:", PACIENTE_MONITOREO_NORMAL)
print("VERTEBRA_MONITOREO_NORMAL:", VERTEBRA_MONITOREO_NORMAL)
print("PACIENTE_MONITOREO_SCOL  :", PACIENTE_MONITOREO_SCOL)
print("VERTEBRA_MONITOREO_SCOL  :", VERTEBRA_MONITOREO_SCOL)


In [ ]:
# ==========================================
# 82) PREPARAR NUEVAS MUESTRAS DE MONITOREO
# ==========================================

muestra_mon_normal = preparar_muestra_monitoreo("val", PACIENTE_MONITOREO_NORMAL, VERTEBRA_MONITOREO_NORMAL)
muestra_mon_scol = preparar_muestra_monitoreo("val", PACIENTE_MONITOREO_SCOL, VERTEBRA_MONITOREO_SCOL)

monitor_init_soft = evaluar_monitoreo_doble(titulo="Antes de entrenar suave")
print(monitor_init_soft)


In [ ]:
# ==========================================
# 83) FUNCION DE EPOCA SUAVE
# ==========================================

from tqdm.auto import tqdm

def run_epoch_medsam_soft(loader, train=True, epoch=None):
    if train:
        medsam_model.train()
        desc = f"TrainSoft Epoch {epoch}" if epoch is not None else "TrainSoft"
    else:
        medsam_model.eval()
        desc = f"ValSoft Epoch {epoch}" if epoch is not None else "ValSoft"

    total_loss = 0.0
    total_bce = 0.0
    total_dice = 0.0
    n_samples = 0

    pbar = tqdm(loader, desc=desc, leave=True)

    for batch in pbar:
        for sample in batch:
            image_np = sample["image"]
            box = sample["box"].to(device)
            gt_mask = sample["gt_mask"].to(device)
            sample_weight = sample["sample_weight"].to(device)

            if train:
                optimizer.zero_grad()

            with torch.set_grad_enabled(train):
                pred_logits, iou_pred = forward_medsam_with_box(image_np, box)
                gt_mask_batch = gt_mask.unsqueeze(0) if gt_mask.ndim == 3 else gt_mask

                loss_bce = bce_loss(pred_logits, gt_mask_batch)
                loss_dice = dice_loss_from_logits(pred_logits, gt_mask_batch)

                # peso suave solo sobre una mezcla estable
                loss = loss_bce + (loss_dice * sample_weight)

                if train:
                    loss.backward()
                    optimizer.step()

            total_loss += loss.item()
            total_bce += loss_bce.item()
            total_dice += loss_dice.item()
            n_samples += 1

        pbar.set_postfix({
            "loss": f"{total_loss / max(n_samples, 1):.4f}",
            "bce": f"{total_bce / max(n_samples, 1):.4f}",
            "dice_loss": f"{total_dice / max(n_samples, 1):.4f}",
        })

    return {
        "loss": total_loss / max(n_samples, 1),
        "bce": total_bce / max(n_samples, 1),
        "dice_loss": total_dice / max(n_samples, 1)
    }


In [ ]:
# ==========================================
# 84) LOOP SUAVE CON MONITOREO DOBLE
# ==========================================

num_epochs = 10
best_val_loss_soft = float("inf")
best_state_dict_soft = None
patience = 3
patience_counter = 0

history_soft = []
monitor_history_soft = []

for epoch in range(1, num_epochs + 1):
    train_metrics = run_epoch_medsam_soft(train_loader_soft, train=True, epoch=epoch)
    val_metrics = run_epoch_medsam_soft(val_loader_soft, train=False, epoch=epoch)

    scheduler.step(val_metrics["loss"])

    mon = evaluar_monitoreo_doble(titulo=f"Soft Epoch {epoch}")

    history_soft.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_bce": train_metrics["bce"],
        "train_dice_loss": train_metrics["dice_loss"],
        "val_loss": val_metrics["loss"],
        "val_bce": val_metrics["bce"],
        "val_dice_loss": val_metrics["dice_loss"],
        "lr": optimizer.param_groups[0]["lr"]
    })

    monitor_history_soft.append({
        "epoch": epoch,
        "dice_normal": mon["dice_normal"],
        "iou_normal": mon["iou_normal"],
        "score_normal": mon["score_normal"],
        "dice_scol": mon["dice_scol"],
        "iou_scol": mon["iou_scol"],
        "score_scol": mon["score_scol"]
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"Dice_normal={mon['dice_normal']:.4f} | "
        f"Dice_scol={mon['dice_scol']:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.2e}"
    )

    if val_metrics["loss"] < best_val_loss_soft:
        best_val_loss_soft = val_metrics["loss"]
        best_state_dict_soft = {k: v.cpu().clone() for k, v in medsam_model.state_dict().items()}
        patience_counter = 0
        print("  -> nuevo mejor modelo suave")
    else:
        patience_counter += 1
        print(f"  -> sin mejora ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print("Early stopping activado")
        break


In [ ]:
# ==========================================
# 85) RESTAURAR MEJOR MODELO SUAVE
# ==========================================

from segment_anything import SamPredictor

if best_state_dict_soft is not None:
    medsam_model.load_state_dict(best_state_dict_soft)
    medsam_model = medsam_model.to(device)
    medsam_model.eval()
    predictor = SamPredictor(medsam_model)
    print("Mejor modelo suave restaurado.")
else:
    print("No se guardó best_state_dict_soft.")


In [ ]:
# ==========================================
# 86) HISTORIAL SUAVE
# ==========================================

df_history_soft = pd.DataFrame(history_soft)
df_monitor_soft = pd.DataFrame(monitor_history_soft)

display(df_history_soft)
display(df_monitor_soft)


## 15. Escenario 4: decoder_plus_weighted_plus_soft_plus_encoder

**Tipo:** escenario acumulativo de mayor capacidad.

**Hipotesis:** si los cambios anteriores son prometedores, abrir una parte pequena del image encoder puede adaptar mejor las representaciones visuales al dominio radiografico.

**Que cambia:** se mantiene entrenable el `mask_decoder` y se abre solo el ultimo bloque del `image_encoder`, usando un learning rate menor para el encoder.

**Riesgo:** este escenario tiene mayor probabilidad de sobreajuste. Debe aceptarse solo si mejora validacion de forma consistente y no empeora mucho por estrato, por vertebra o en predicciones vacias.

<!-- codex-explicacion -->
Aqui se permite ajustar una parte adicional del encoder. Es mas potente, pero tambien aumenta riesgo de sobreajuste; por eso se decide con validacion.


In [ ]:
# ==========================================
# 87) INSPECCIONAR IMAGE ENCODER
# ==========================================

print("=== IMAGE ENCODER ===")
for name, module in medsam_model.image_encoder.named_children():
    print(name, "->", type(module))


In [ ]:
# ==========================================
# 88) ENTRENABILIDAD CONTROLADA
# ==========================================

# 1) congelar todo
for p in medsam_model.parameters():
    p.requires_grad = False

# 2) prompt encoder siempre congelado
for p in medsam_model.prompt_encoder.parameters():
    p.requires_grad = False

# 3) mask decoder entrenable
for p in medsam_model.mask_decoder.parameters():
    p.requires_grad = True

# 4) abrir solo el ultimo bloque del image encoder
#    normalmente en ViT de SAM esto suele vivir en image_encoder.blocks
if hasattr(medsam_model.image_encoder, "blocks"):
    print("Se encontró image_encoder.blocks")
    for p in medsam_model.image_encoder.blocks[-1].parameters():
        p.requires_grad = True
    ULTIMO_BLOQUE_ENCODER = "blocks[-1]"
else:
    raise AttributeError(
        "No encontré image_encoder.blocks. "
        "Revisa la salida del bloque 87 para ajustar el nombre del último bloque."
    )

# Diagnóstico
n_total = 0
n_trainable = 0
trainable_names = []

for name, p in medsam_model.named_parameters():
    n = p.numel()
    n_total += n
    if p.requires_grad:
        n_trainable += n
        trainable_names.append(name)

print("Ultimo bloque abierto:", ULTIMO_BLOQUE_ENCODER)
print("Parametros totales     :", n_total)
print("Parametros entrenables :", n_trainable)
print("Porcentaje entrenable  :", 100 * n_trainable / n_total)

print("\nPrimeros 40 parametros entrenables:")
for name in trainable_names[:40]:
    print(name)


In [ ]:
# ==========================================
# 89) OPTIMIZADOR CON DOS LEARNING RATES
# ==========================================

import torch.optim as optim

LR_ENCODER = 1e-5
LR_DECODER = 5e-5
WEIGHT_DECAY = 1e-4

params_encoder = []
params_decoder = []

for name, p in medsam_model.named_parameters():
    if not p.requires_grad:
        continue

    if name.startswith("image_encoder.blocks."):
        params_encoder.append(p)
    elif name.startswith("mask_decoder."):
        params_decoder.append(p)

print("Tensores encoder entrenables:", len(params_encoder))
print("Tensores decoder entrenables:", len(params_decoder))

optimizer = optim.AdamW(
    [
        {"params": params_encoder, "lr": LR_ENCODER},
        {"params": params_decoder, "lr": LR_DECODER},
    ],
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("LR encoder:", LR_ENCODER)
print("LR decoder:", LR_DECODER)


In [ ]:
# ==========================================
# 90) VERIFICAR PARAM GROUPS
# ==========================================

for i, g in enumerate(optimizer.param_groups):
    print(f"Grupo {i}: lr={g['lr']}, n_params={len(g['params'])}")


In [ ]:
# ==========================================
# 91) DEFINIR LOADERS DEL EXPERIMENTO
# ==========================================

# Recomendado: usar el setup suave
train_loader_exp = train_loader_soft
val_loader_exp = val_loader_soft

print("Usando loaders suaves para el experimento del encoder parcial.")
print("Train batches:", len(train_loader_exp))
print("Val batches  :", len(val_loader_exp))


In [ ]:
# ==========================================
# 92) FUNCION DE EPOCA - ENCODER PARCIAL
# ==========================================

from tqdm.auto import tqdm

def run_epoch_medsam_encoder_partial(loader, train=True, epoch=None):
    if train:
        medsam_model.train()
        desc = f"TrainEnc Epoch {epoch}" if epoch is not None else "TrainEnc"
    else:
        medsam_model.eval()
        desc = f"ValEnc Epoch {epoch}" if epoch is not None else "ValEnc"

    total_loss = 0.0
    total_bce = 0.0
    total_dice = 0.0
    n_samples = 0

    pbar = tqdm(loader, desc=desc, leave=True)

    for batch in pbar:
        for sample in batch:
            image_np = sample["image"]
            box = sample["box"].to(device)
            gt_mask = sample["gt_mask"].to(device)
            sample_weight = sample["sample_weight"].to(device) if "sample_weight" in sample else torch.tensor(1.0, device=device)

            if train:
                optimizer.zero_grad()

            with torch.set_grad_enabled(train):
                pred_logits, iou_pred = forward_medsam_with_box(image_np, box)
                gt_mask_batch = gt_mask.unsqueeze(0) if gt_mask.ndim == 3 else gt_mask

                loss_bce = bce_loss(pred_logits, gt_mask_batch)
                loss_dice = dice_loss_from_logits(pred_logits, gt_mask_batch)

                # versión suave: ponderar principalmente dice
                loss = loss_bce + (loss_dice * sample_weight)

                if train:
                    loss.backward()
                    optimizer.step()

            total_loss += loss.item()
            total_bce += loss_bce.item()
            total_dice += loss_dice.item()
            n_samples += 1

        pbar.set_postfix({
            "loss": f"{total_loss / max(n_samples, 1):.4f}",
            "bce": f"{total_bce / max(n_samples, 1):.4f}",
            "dice_loss": f"{total_dice / max(n_samples, 1):.4f}",
        })

    return {
        "loss": total_loss / max(n_samples, 1),
        "bce": total_bce / max(n_samples, 1),
        "dice_loss": total_dice / max(n_samples, 1)
    }


In [ ]:
# ==========================================
# 93) CHEQUEO INICIAL DEL EXPERIMENTO
# ==========================================

monitor_init_encoder = evaluar_monitoreo_doble(titulo="Antes de entrenar encoder parcial")
print(monitor_init_encoder)


In [ ]:
# ==========================================
# 94) LOOP DEL EXPERIMENTO: ENCODER PARCIAL
# ==========================================

num_epochs = 10
best_val_loss_enc = float("inf")
best_state_dict_enc = None
patience = 3
patience_counter = 0

history_enc = []
monitor_history_enc = []

for epoch in range(1, num_epochs + 1):
    train_metrics = run_epoch_medsam_encoder_partial(train_loader_exp, train=True, epoch=epoch)
    val_metrics = run_epoch_medsam_encoder_partial(val_loader_exp, train=False, epoch=epoch)

    scheduler.step(val_metrics["loss"])

    mon = evaluar_monitoreo_doble(titulo=f"EncoderPartial Epoch {epoch}")

    history_enc.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_bce": train_metrics["bce"],
        "train_dice_loss": train_metrics["dice_loss"],
        "val_loss": val_metrics["loss"],
        "val_bce": val_metrics["bce"],
        "val_dice_loss": val_metrics["dice_loss"],
        "lr_encoder": optimizer.param_groups[0]["lr"],
        "lr_decoder": optimizer.param_groups[1]["lr"],
    })

    monitor_history_enc.append({
        "epoch": epoch,
        "dice_normal": mon["dice_normal"],
        "iou_normal": mon["iou_normal"],
        "score_normal": mon["score_normal"],
        "dice_scol": mon["dice_scol"],
        "iou_scol": mon["iou_scol"],
        "score_scol": mon["score_scol"]
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"Dice_normal={mon['dice_normal']:.4f} | "
        f"Dice_scol={mon['dice_scol']:.4f} | "
        f"lr_enc={optimizer.param_groups[0]['lr']:.2e} | "
        f"lr_dec={optimizer.param_groups[1]['lr']:.2e}"
    )

    if val_metrics["loss"] < best_val_loss_enc:
        best_val_loss_enc = val_metrics["loss"]
        best_state_dict_enc = {k: v.cpu().clone() for k, v in medsam_model.state_dict().items()}
        patience_counter = 0
        print("  -> nuevo mejor modelo encoder parcial")
    else:
        patience_counter += 1
        print(f"  -> sin mejora ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print("Early stopping activado")
        break


In [ ]:
# ==========================================
# 95) RESTAURAR MEJOR MODELO ENCODER PARCIAL
# ==========================================

from segment_anything import SamPredictor

if best_state_dict_enc is not None:
    medsam_model.load_state_dict(best_state_dict_enc)
    medsam_model = medsam_model.to(device)
    medsam_model.eval()
    predictor = SamPredictor(medsam_model)
    print("Mejor modelo encoder parcial restaurado.")
else:
    print("No se guardó best_state_dict_enc.")


In [ ]:
# ==========================================
# 96) HISTORIAL DEL EXPERIMENTO ENCODER PARCIAL
# ==========================================

import pandas as pd

df_history_enc = pd.DataFrame(history_enc)
df_monitor_enc = pd.DataFrame(monitor_history_enc)

display(df_history_enc)
display(df_monitor_enc)


## 16. Comparacion de escenarios en validacion

Esta tabla es el punto de decision. Todos los escenarios disponibles se restauran desde su `state_dict` y se evaluan sobre el mismo split de validacion con el mismo protocolo.

Como algunos escenarios son acumulativos, los nombres deben leerse como recetas completas:

- `decoder_basico`: solo cambio 1.
- `decoder_plus_weighted_sampler`: cambio 1 + sampler ponderado.
- `decoder_plus_weighted_plus_soft`: cambio 1 + sampler + loss suave.
- `decoder_plus_weighted_plus_soft_plus_encoder`: receta acumulada con encoder parcial.

La seleccion del modelo final se hace aqui. Test queda intacto hasta que esta comparacion termine y se elija una sola configuracion.

<!-- codex-explicacion -->
La validacion se usa para escoger estrategia, no el test. Esta tabla es el filtro metodologico antes de congelar una receta.


In [ ]:
# ==========================================
# 97) COMPARAR VARIANTES EN VAL
# ==========================================

from segment_anything import SamPredictor

MEDSAM_VARIANTS = [
    {
        "nombre": "decoder_basico",
        "state_var": "best_state_dict",
        "split": "val",
        "frac_x": 0.04,
        "frac_y": 0.06,
        "prompt_origen": "bbox_gt"
    },
    {
        "nombre": "decoder_plus_weighted_sampler",
        "state_var": "best_state_dict_w",
        "split": "val",
        "frac_x": 0.04,
        "frac_y": 0.06,
        "prompt_origen": "bbox_gt"
    },
    {
        "nombre": "decoder_plus_weighted_plus_soft",
        "state_var": "best_state_dict_soft",
        "split": "val",
        "frac_x": 0.04,
        "frac_y": 0.06,
        "prompt_origen": "bbox_gt"
    },
    {
        "nombre": "decoder_plus_weighted_plus_soft_plus_encoder",
        "state_var": "best_state_dict_enc",
        "split": "val",
        "frac_x": 0.04,
        "frac_y": 0.06,
        "prompt_origen": "bbox_gt"
    }
]


def restaurar_variante_medsam(state_var):
    global predictor

    state = globals().get(state_var)
    if state is None:
        return False

    medsam_model.load_state_dict(state)
    medsam_model.to(device)
    medsam_model.eval()
    predictor = SamPredictor(medsam_model)
    return True


def evaluar_variantes_en_val(variantes):
    filas_comparacion = []
    resultados_por_variante = {}

    for cfg in variantes:
        nombre = cfg["nombre"]
        state_var = cfg["state_var"]

        if not restaurar_variante_medsam(state_var):
            print(f"Se omite {nombre}: no existe {state_var} en memoria.")
            continue

        split_eval = cfg.get("split", "val")
        frac_x = cfg.get("frac_x", 0.04)
        frac_y = cfg.get("frac_y", 0.06)

        print("=" * 80)
        print(f"Evaluando variante: {nombre} | split={split_eval} | frac_x={frac_x} | frac_y={frac_y}")
        print("=" * 80)

        resultados, df_resumen, df_detalle, df_errores = evaluar_split_completo(
            split_eval=split_eval,
            frac_x=frac_x,
            frac_y=frac_y
        )

        nombre_eval = f"{nombre}_{split_eval}_prompt_{cfg.get('prompt_origen', 'bbox_gt')}_fracx{int(frac_x*1000):03d}_fracy{int(frac_y*1000):03d}"
        resumenes = resumir_metricas_split(
            df_resumen,
            df_detalle,
            df_errores,
            split_eval,
            nombre_eval=nombre_eval,
            guardar=True
        )

        fila = resumenes["global"].iloc[0].to_dict()
        fila.update({
            "variante": nombre,
            "state_var": state_var,
            "prompt_origen": cfg.get("prompt_origen", "bbox_gt"),
            "frac_x": frac_x,
            "frac_y": frac_y,
            "errores": len(df_errores)
        })
        filas_comparacion.append(fila)

        resultados_por_variante[nombre] = {
            "resultados": resultados,
            "df_resumen": df_resumen,
            "df_detalle": df_detalle,
            "df_errores": df_errores,
            "resumenes": resumenes
        }

    df_comparacion_variantes = pd.DataFrame(filas_comparacion)
    if len(df_comparacion_variantes) > 0:
        columnas = [
            "variante", "split", "n_pacientes", "n_vertebras_eval",
            "dice_macro_paciente_mean", "dice_ci95_low", "dice_ci95_high",
            "iou_macro_paciente_mean", "iou_ci95_low", "iou_ci95_high",
            "overlap_fraction_pred_mean", "predicciones_vacias_total",
            "frac_x", "frac_y", "prompt_origen", "errores"
        ]
        columnas = [c for c in columnas if c in df_comparacion_variantes.columns]
        df_comparacion_variantes = df_comparacion_variantes[columnas]
        display(df_comparacion_variantes)
        df_comparacion_variantes.to_csv(RESULTS_ROOT / "comparacion_variantes_val.csv", index=False)

    return df_comparacion_variantes, resultados_por_variante


EVALUAR_VARIANTES_VAL = False

if EVALUAR_VARIANTES_VAL:
    df_comparacion_variantes, resultados_por_variante = evaluar_variantes_en_val(MEDSAM_VARIANTS)
else:
    print("Comparacion de variantes en val desactivada. Cambia EVALUAR_VARIANTES_VAL a True para ejecutarla.")


## 17. Data augmentation como escenario futuro

Esta seccion queda separada porque representa otra receta experimental. Si se activa, debe tratarse como un nuevo escenario completo: entrenar, evaluar en validacion y compararlo contra los escenarios anteriores.

No debe aplicarse augmentation a `val` ni a `test`. Validacion y test deben conservar la distribucion real de evaluacion.

<!-- codex-explicacion -->
La aumentacion queda separada porque cambia la distribucion de entrenamiento. Debe tratarse como experimento nuevo, no como ajuste menor.


In [ ]:
# ==========================================
# DATA AUGMENTATION OPCIONAL - NO EJECUTAR PARA TEST
# ==========================================

EJECUTAR_DATA_AUGMENTATION = False

if EJECUTAR_DATA_AUGMENTATION:
    import albumentations as A
    import cv2
    import numpy as np
    from PIL import Image
    import os

    # Variante inicial: transformaciones suaves que preservan la anatomia general.
    # Para una variante final con crops, conviene validar visualmente que no se corten vertebras objetivo.
    transform_aug = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.04, scale_limit=0.08, rotate_limit=10, p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.GaussianBlur(p=0.2),
    ], additional_targets={'mask': 'mask'})

    def augmentar_dataset_medsam(df_base, output_dir=OUTPUT_ROOT / "dataset_aug_medsam", n_aug=3):
        output_dir = Path(output_dir)
        (output_dir / "images").mkdir(parents=True, exist_ok=True)
        (output_dir / "masks").mkdir(parents=True, exist_ok=True)

        registros_aug = []

        for _, fila in tqdm(df_base.iterrows(), total=len(df_base), desc="Augment MedSAM train"):
            img = np.array(Image.open(fila['ruta_img']).convert('L'))
            mask = np.array(Image.open(fila['ruta_mask_id']))

            for i in range(n_aug):
                augmented = transform_aug(image=img, mask=mask)
                img_aug, mask_aug = augmented['image'], augmented['mask']

                pid_aug = f"{fila['patient_id']}_aug{i}"
                img_path = output_dir / "images" / f"{pid_aug}.png"
                mask_path = output_dir / "masks" / f"{pid_aug}.png"

                cv2.imwrite(str(img_path), img_aug)
                Image.fromarray(mask_aug.astype(mask.dtype)).save(mask_path)

                registros_aug.append({
                    "patient_id": pid_aug,
                    "tipo": fila.get("tipo", "aug"),
                    "ruta_img": str(img_path),
                    "ruta_mask_id": str(mask_path),
                    "tiene_mask_id": True,
                    "origen_aug": fila['patient_id']
                })

        return pd.DataFrame(registros_aug)

    df_aug_train = augmentar_dataset_medsam(df_tr, n_aug=3)
    display(df_aug_train.head())
    print("Imagenes aumentadas de train:", len(df_aug_train))
else:
    print("Data augmentation desactivado. Activar solo para entrenar una variante nueva y evaluarla primero en val.")


# 18. Evaluacion final en test - no tocar hasta congelar el modelo

Esta es la ultima seccion del notebook de forma intencional. El test no se usa para depurar, ajustar hiperparametros, escoger variantes ni decidir si una modificacion funciona.

Solo debe ejecutarse cuando ya se eligio una unica receta final usando validacion. En ese momento, el test entrega una estimacion honesta del desempeno esperado en datos no vistos.

<!-- codex-explicacion -->
Esta seccion protege el test. La idea correcta es mirar test una vez, cuando la receta ya fue elegida, para obtener una estimacion honesta.


In [ ]:
# ==========================================
# 98) TEST FINAL BLOQUEADO POR DEFECTO
# ==========================================

EJECUTAR_TEST_FINAL = False

# Elegir una sola variante final antes de activar EJECUTAR_TEST_FINAL.
# Opciones esperadas segun los experimentos anteriores:
#   "decoder_basico"          -> best_state_dict
#   "reequilibrado_weighted"  -> best_state_dict_w
#   "reequilibrado_suave"     -> best_state_dict_soft
#   "encoder_parcial"         -> best_state_dict_enc
VARIANTE_FINAL_TEST = "encoder_parcial"

STATE_VAR_FINAL = {
    "decoder_basico": "best_state_dict",
    "reequilibrado_weighted": "best_state_dict_w",
    "reequilibrado_suave": "best_state_dict_soft",
    "encoder_parcial": "best_state_dict_enc"
}

PROMPT_ORIGEN_TEST = "bbox_gt"  # caja ideal derivada del ground truth; no es pipeline automatico end-to-end
FRAC_X_TEST = 0.04
FRAC_Y_TEST = 0.06

from segment_anything import SamPredictor

def restaurar_variante_medsam_final(state_var):
    global predictor
    state = globals().get(state_var)
    if state is None:
        return False
    medsam_model.load_state_dict(state)
    medsam_model.to(device)
    medsam_model.eval()
    predictor = SamPredictor(medsam_model)
    return True

if EJECUTAR_TEST_FINAL:
    state_var = STATE_VAR_FINAL[VARIANTE_FINAL_TEST]
    if not restaurar_variante_medsam_final(state_var):
        raise RuntimeError(f"No existe {state_var}. Entrena/restaura la variante final antes de evaluar test.")

    resultados_test, df_resumen_test, df_detalle_test, df_errores_test = evaluar_split_completo(
        split_eval="test",
        frac_x=FRAC_X_TEST,
        frac_y=FRAC_Y_TEST
    )

    nombre_eval_test = (
        f"FINAL_TEST_{VARIANTE_FINAL_TEST}_prompt_{PROMPT_ORIGEN_TEST}_"
        f"fracx{int(FRAC_X_TEST*1000):03d}_fracy{int(FRAC_Y_TEST*1000):03d}"
    )

    resumen_test = resumir_metricas_split(
        df_resumen_test,
        df_detalle_test,
        df_errores_test,
        "test",
        nombre_eval=nombre_eval_test,
        guardar=True
    )

    print("RESULTADO FINAL TEST GUARDADO EN:", RESULTS_ROOT / nombre_eval_test)
else:
    print("TEST FINAL NO EJECUTADO. Cambia EJECUTAR_TEST_FINAL a True solo cuando el experimento este congelado.")


<!-- codex-cierre-etapa -->
## Cierre de etapa y siguiente paso

Esta etapa consolid? el salto hacia segmentaci?n multiclase y MedSAM. Aqu? se defini? el split estratificado, se export? el dataset en formato MedSAM y se probaron variantes de entrenamiento/evaluaci?n con prompts.

La conclusi?n pr?ctica fue que MedSAM puede segmentar mejor cuando recibe prompts ?tiles, pero depender de prompts ideales o demasiado manuales no resuelve el problema completo. Por eso la siguiente etapa fue buscar una forma autom?tica de generar cajas vertebrales que pudieran usarse como prompts.

**Siguiente notebook:** `04_cajas_nn_centernetlite.ipynb`.
